# 7.0 Doublet analysis

In [ ]:
import anndata as ad
import scanpy as sc
import scanpy.external as sce
import scirpy as ir
import pandas as pd
import numpy as np
import seaborn as sb
import decoupler as dc
import gseapy as gp
import os
import muon as mu
import scrublet as scr
from scipy.stats import binomtest
from scipy.stats import mannwhitneyu
from scipy.stats import spearmanr
from scipy.stats import wilcoxon
from scipy import sparse
from statsmodels.stats.multitest import multipletests
from matplotlib.patches import Patch

# Helps to define plotting area
import matplotlib.pyplot as plt
from matplotlib import rcParams 
from matplotlib.pyplot import rc_context
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib import colors
from matplotlib_venn import venn2
import matplotlib.patches as mpatches

# 
from sklearn.linear_model import LinearRegression

## For making text editable in .pdfs once exported use cairo backend instead of agg
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams["font.family"] = "Arial"
sb.set_theme(style="ticks", font="Arial")

# set seed
np.random.seed(44)

#display options
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999

In [ ]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("fontTools").setLevel(logging.WARNING)

# Helper functions from Mimi

In [ ]:
tcr_gene_prefixs = ['TRAV', 'TRAJ', 'TRAC', 'TRB', 'TRDV', 'TRDC', 'TRG']

def filter_tcr_genes(adata):
    non_tcr_genes = adata.var_names
    for prefix in tcr_gene_prefixs:
        non_tcr_genes = [el for el in non_tcr_genes if not el.startswith(prefix)]
    adata = adata[:, non_tcr_genes]
    return adata

def filter_high_var(adata, n):
    #sc.pp.highly_variable_genes(
    #adata,
    #flavor="seurat_v3",
    #n_top_genes=n,
    #layer="counts",
    #batch_key="pool",
    #subset=True,
    #span = 0.75) #default = 0.3. It's the fraction of cells used for estimation of hvg - if I have 150 cells in one sample but 150.000 cells in total then this sample becomes an issue. 
    
    sc.pp.highly_variable_genes(adata, n_top_genes=n)
    adata = adata[:, adata.var['highly_variable']].copy()
    return adata


def calculate_umap(adata, n_high_var=5000, remove_tcr_genes=True):
    """
    Calculates UMAP coordinates
    :param adata: AnnData object
    :param n_high_var: how many variable genes to use, None uses raw data
    :param remove_tcr_genes: whether to remove TCR genes
    :return: results stored in adata object
    """
    adata_tmp = adata.copy()

    if remove_tcr_genes:
        adata_tmp = filter_tcr_genes(adata_tmp)
    if n_high_var is not None:
        adata_tmp = filter_high_var(adata_tmp, n_high_var)

    sc.pp.neighbors(adata_tmp, n_pcs=20)            
    sc.tl.umap(adata_tmp)
    adata.obsm['X_umap'] = adata_tmp.obsm['X_umap']


def calculate_leiden(adata, resolution, n_high_var=5000, remove_tcr_genes=True):
    adata_tmp = adata.copy()

    if remove_tcr_genes:
        adata_tmp = filter_tcr_genes(adata_tmp)
    if n_high_var is not None:
        adata_tmp = filter_high_var(adata_tmp, n_high_var)

    sc.pp.neighbors(adata_tmp, n_pcs=20)
    sc.tl.leiden(adata_tmp, resolution=resolution)

    leiden_column_name = f"leiden_{resolution}"
    adata.obs[leiden_column_name] = adata_tmp.obs['leiden']

# Loading data

In [ ]:
sdata = mu.read(filename="../../04_data_objects/06_preprocessed_v2/2026_6_main_dataset_clustered_annotated.h5mu")


In [ ]:
sdata

# Fix catpat (assign IC)

In [ ]:
#Group annotation
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]
IC_list = ["Pt5", "Pt4"]

#Match the lists defined above to define "RRMS" and "Control"
sdata["gex"].obs["catpat"] = pd.Categorical(np.where(sdata["gex"].obs["donor"].isin(patient_list), "MS", np.where(sdata["gex"].obs["donor"].isin(control_list), "NIC", np.where(sdata["gex"].obs["donor"].isin(IC_list), "IC", "Error"))))


## Violin plot

In [ ]:
# Extract obs and compute medians
obs = sdata["gex"].obs
vals = sdata["gex"].obs["n_genes_by_counts"]
groups = sdata["gex"].obs["L1_ann"]

# Calculate median per group
medians = obs.groupby("L1_ann")["n_genes_by_counts"].median()

# Sort groups by median
sorted_groups = medians.sort_values().index.tolist()

# Plot with sorted order
with plt.rc_context({"figure.figsize": (4, 3)}):
    ax = sc.pl.violin(
        sdata["gex"], 
        keys="n_genes_by_counts",
        groupby="L1_ann",
        order=sorted_groups,     # 👈 pass sorted order
        multi_panel=False,
        rotation=45,
        show=False
    )

plt.xticks(rotation=45, ha='right')
plt.axhline(y=200, linewidth=1.5, color="black", linestyle="--")
plt.ylabel("Genes per cell")

# Rasterize violin bodies
for col in ax.collections:
    col.set_rasterized(True)

plt.tight_layout()
plt.savefig(
    "..\\..\\05_plots\\02_Overview_and_doublet_analysis\\13_genes_per_cell_L1.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()


In [ ]:
# Extract obs and compute medians
medians = sdata["gex"].obs.groupby("L1_ann")["total_counts"].median()

# Sort groups by median (ascending by default)
sorted_groups = medians.sort_values().index.tolist()

# Plot with sorted order
with plt.rc_context({"figure.figsize": (4, 3)}):
    ax = sc.pl.violin(
        sdata["gex"], 
        keys="total_counts",
        groupby="L1_ann",
        order=sorted_groups,     # 👈 pass sorted order
        multi_panel=False,
        rotation=45,
        show=False
    )

plt.xticks(rotation=45, ha='right')
plt.ylabel("Total read counts")

# Rasterize violin bodies
for col in ax.collections:
    col.set_rasterized(True)

plt.tight_layout()
plt.savefig(
    "..\\..\\05_plots\\02_Overview_and_doublet_analysis\\15_Total_reads_L1.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()



# Recluster CSF to annotate clusters with the highest possible resolution
Motivation: Annotations done with both PB and CSF cells together may underestimate the true number of multiplet. We therefore first have to recluster and re-annotate CSF cells

A strategy proposed by Kilian here, is to use the TCRseq data as a way to annotate all T-X multiplet among non-T cell populations

In [ ]:
CSF_data = sdata.copy()
mu.pp.filter_obs(CSF_data["gex"], "Compartment", lambda x: (x == "CSF")) 
CSF_data.update()

In [ ]:
# ---
# Remove genes that are present in only very few cells 
# ---
mu.pp.filter_var(CSF_data["gex"], 'n_cells_by_counts', lambda x: x > 10)
CSF_data.update()

# ---
# Allign the modalities emphasizing cells in gex.
#   Make certain that the cells in airr and prot modalities are the same cells as are present in gex.
# ---
#Make sure only the indexes recorded in GEX are used in the o*ther modalities too
index_list = CSF_data["gex"].obs.index.tolist()

## define a cell index
CSF_data["airr"].obs["cell_index"] = CSF_data["airr"].obs.index
CSF_data["prot"].obs["cell_index"] = CSF_data["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
CSF_data["airr"].obs["filter_index"] = pd.Categorical(np.where(CSF_data["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
CSF_data["prot"].obs["filter_index"] = pd.Categorical(np.where(CSF_data["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that CSF_data["airr"].obs and CSF_data["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(CSF_data["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(CSF_data["prot"], 'filter_index', lambda x: x == "GEX_present")
CSF_data.update()

# ---
# When the .var object has been filtered, then you need to renormalize
# ---
CSF_data["gex"].X = CSF_data["gex"].layers["counts"].copy()
CSF_data["gex"].uns.pop("log1p", None) #Removes log1P if it exists
sc.pp.normalize_total(CSF_data["gex"], target_sum=1e4, exclude_highly_expressed=False)   #1e4 is important. Do not change
sc.pp.log1p(CSF_data["gex"])

In [ ]:
calculate_leiden(CSF_data["gex"], resolution=3, n_high_var=2000, remove_tcr_genes=True)   
calculate_umap(CSF_data["gex"], n_high_var=2000, remove_tcr_genes=True)

In [ ]:
plt.rcParams["figure.figsize"] = (6, 6)
fig = sc.pl.umap(CSF_data["gex"], color=["leiden_3", "chain_pairing", "final_ann", "scrublet_score"], legend_fontweight='bold', cmap="RdBu_r", frameon= False , vmin=0, legend_fontoutline=2, legend_loc = "on data", size=10, return_fig = True, show=False, title="APC_singlets")

# # Rasterize all scatter collections
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/43_CSF_reclustered.pdf", 
    bbox_inches="tight",
    dpi=300
)
plt.show()

## 
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

In [ ]:
# marker_genes = {
#         "T cell": ["CD3E", "LCK", "TRAC", "CD4", "CD8A", "CD8B"],
#         "MAIT" : ["TRAV1-2", "SLC4A10"],
#         "MG/BAM":  ["TREM2", "APOE", "LYVE1", "SPP1"],
#         "Monocyte" : ["FCN1", "VCAN"], 
#         "DCs": ["CD14", "CD163", "CD1C", "FLT3", "CLEC9A", "AXL", "KLF4", "ATF3", "EMP1", "ALDOA", "FABP5"],
#         "B cell": ["MS4A1", "CD19", "IGHG1"],
#         "NK cell": ["NCAM1", "NCR1"], 
#         "pDC": ["IL3RA", "PTCRA", "IRF7"],
#         "APC-status": ["CST3", "CD74", "HLA-DRB1"]}


# dp = sc.pl.dotplot(
#     CSF_data["gex"],
#     marker_genes,
#     groupby="leiden_3",
#     standard_scale="var",
#     cmap="RdBu_r",
#     return_fig=True,
#     show=False,
# )

# dp.add_totals()
# dp.style(dot_edge_color="black", dot_edge_lw=0.5)

# # Build the figure
# dp.make_figure()
# dp.show()

# # Save
# dp.fig.savefig(
#     "../../05_plots/02_Overview_and_doublet_analysis/92_CSF_reclustered_dotplot.pdf",
#     bbox_inches="tight",
#     dpi=400,
# )

In [ ]:
annotation_dict = {
    '0': 'T cell',
    '1': 'T cell',
    '2': 'T cell',
    '3': 'T cell',
    '4': 'T cell',
    '5': 'T cell',
    '6': 'T cell',
    '7': 'T cell',
    '8': 'T cell',
    '9': 'T cell',
    '10': 'MG/BAM',
    '11': 'T cell',
    '12': 'T cell',
    '13': 'cDC',
    '14': 'T-MG/BAM',
    '15': 'T cell',
    '16': 'T cell',
    '17': 'T cell',
    '18': 'MG/BAM',
    '19': 'MG/BAM',
    '20': 'T cell',
    '21': 'Monocyte',
    '22': 'T cell',
    '23': 'T-cDC',
    '24': 'T cell',
    '25': 'NK cell',
    '26': 'T cell',
    '27': 'Ex-multiplet',
    '28': 'MG/BAM',
    '29': 'B cell',
    '30': 'T cell',
    '31': 'T cell',
    '32': 'T-Monocyte',
    '33': 'T-MG/BAM',
    '34': 'T cell',
    '35': 'MG/BAM',
    '36': 'pDC',
    '37': 'PBla',
    '38': 'T-B cell',
    '39': 'Apoptotic',
    '40': 'cDC',
    '41': 'T-pDC-MG/BAM',
}

# Add cell type column based on annotation
CSF_data["gex"].obs['Recluster_ann'] = [annotation_dict[sample] for sample in CSF_data["gex"].obs['leiden_3']]

In [ ]:
# plt.rcParams["figure.figsize"] = (6, 6)
# fig = sc.pl.umap(CSF_data["gex"], color=["Recluster_ann", "chain_pairing", "final_ann", "scrublet_score"], legend_fontweight='bold', cmap="RdBu_r", frameon= False , vmin=0, legend_fontoutline=2, legend_loc = "on data", size=10, return_fig = True, show=False, title="APC_singlets")

# # # Rasterize all scatter collections
# for ax in fig.axes:
#     for coll in ax.collections:
#         coll.set_rasterized(True)

# fig.savefig(
#     "../../05_plots/02_Overview_and_doublet_analysis/43_CSF_reclustered.pdf", 
#     bbox_inches="tight",
#     dpi=300
# )
# plt.show()

# ## 
# plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

## Error correct singlet APC clusters so thaT cell with a TCR chain is assigned a doublet identity

In [ ]:
CSF_data["gex"].obs["chain_pairing"] = (
    CSF_data["gex"].obs["chain_pairing"]
    .cat.add_categories(["No IR"])
)

CSF_data["gex"].obs["chain_pairing"] = CSF_data["gex"].obs["chain_pairing"].fillna("No IR")

In [ ]:
### If the cell has a T-cell receptor and is not a  T-cell, then define is as a multiplet
CSF_data["gex"].obs["Recluster_ann2"] = pd.Categorical(np.where(CSF_data["gex"].obs["Recluster_ann"].isin(["T cell", "Apoptotic", "Ex-multiplet"]), CSF_data["gex"].obs["Recluster_ann"],        #0 - if a T cell or apoptotic T cell then continue. No need to correct
                                                       np.where(CSF_data["gex"].obs["Recluster_ann"].str.contains("T-"), CSF_data["gex"].obs["Recluster_ann"],                    #1 - if T- then continue. No need to correct
                                                       np.where(CSF_data["gex"].obs["chain_pairing"] == "No IR", CSF_data["gex"].obs["Recluster_ann"],                            #2 - if a singlet cluster cell does not have a TCR, then continue. No need to correct
                                                       np.where(CSF_data["gex"].obs["chain_pairing"] != "No IR", "T-" + CSF_data["gex"].obs["Recluster_ann"].astype(str),         #3 - if a singlet cluster cell DOES have a TCR, then correct. 
                                                                "test")))))    


In [ ]:
pd.crosstab(
    CSF_data["gex"].obs["Recluster_ann"],
    CSF_data["gex"].obs["Recluster_ann2"])

## Annotate T-T doublets
### Annotation of T-T doublets this way is somewhat limited, because of the reduced sensitivity of T-T doublets compared to T-X doublets

In [ ]:
### Define T-cell doublets
# CSF_data["gex"].obs["Recluster_ann2"] = np.where(
#     (CSF_data["gex"].obs["Recluster_ann2"] == "T cell") & (CSF_data["gex"].obs["chain_pairing"] == "two full chains"),
#     "T-T cell",
#     CSF_data["gex"].obs["Recluster_ann2"]
# )

## Define multiplet in general
CSF_data["gex"].obs["cat_multiplet"] = pd.Categorical(np.where(CSF_data["gex"].obs["Recluster_ann2"].str.contains("T-"), "multiplet", "singlet"))

# ### update cat_multiplet
# CSF_data["gex"].obs["cat_multiplet"] = np.where(
#     (CSF_data["gex"].obs["Recluster_ann2"] == "T-T cell"),
#     "multiplet",
#     CSF_data["gex"].obs["cat_multiplet"]
# )

# Control chain_pairing colors

In [ ]:
palette = {
    'extra VDJ': '#4c72b0',
    'extra VJ': '#dd8452',
    'orphan VDJ': '#55a868',
    'orphan VJ': '#c44e52',
    'single pair': '#8172b3',
    'two full chains': '#937860',
    'No IR': '#D3D3D3'
}

adata = CSF_data["gex"]

adata.uns["chain_pairing_colors"] = [
    palette[cat] for cat in adata.obs["chain_pairing"].cat.categories
]

# Finalize overview figures

## Overview umaps

In [ ]:
mu.pp.filter_obs(CSF_data["gex"], "Recluster_ann2", lambda x: (x != "Apoptotic")) 
CSF_data["gex"].obs["Recluster_ann2"] = CSF_data["gex"].obs["Recluster_ann2"].cat.remove_unused_categories()

In [ ]:
plt.rcParams["figure.figsize"] = (6, 6)
fig = sc.pl.umap(CSF_data["gex"], color=["Recluster_ann2", "chain_pairing", "cat_multiplet"], legend_fontweight='bold', cmap="RdBu_r", frameon= False , vmin=0, legend_fontoutline=2, legend_loc = "on data", size=20, return_fig = True, show=False)

# # Rasterize all scatter collections
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/XX_Final_umaps_CSF_reclustered_reannotated.pdf", 
    bbox_inches="tight",
    dpi=300
)
plt.show()

## 
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

# Additional umaps

In [ ]:
CSF_data["gex"].obs["TRX_pairing"] = pd.Categorical(np.where(CSF_data["gex"].obs["chain_pairing"] == "No IR", "TCR (-)", "TCR (+)"), categories=["TCR (-)", "TCR (+)"])

# Define colors
trx_palette = {
    "TCR (-)": "#D3D3D3",  # light grey
    "TCR (+)": "#8172B3",  # sandy purple
}

# Save the colors in the .uns
CSF_data["gex"].uns["TRX_pairing_colors"] = [
    trx_palette["TCR (-)"],
    trx_palette["TCR (+)"]
]

In [ ]:
plt.rcParams["figure.figsize"] = (6, 6)
fig = sc.pl.umap(CSF_data["gex"], color=["TRX_pairing", "CD79A", "C1QA", "CD3E", "CD8A", "CD8B", "CD40LG", "CCL5", "FLT3", "PRF1", "VCAN", "GZMB"], legend_fontweight='bold', cmap="RdBu_r", frameon= False , vmin=0, legend_fontoutline=2, legend_loc = "on data", size=40, return_fig = True, show=False)

# # Rasterize all scatter collections
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/XX_Final_umaps_CSF_additional_umaps.pdf", 
    bbox_inches="tight",
    dpi=300
)
plt.show()

## 
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

## Dotplot 

In [ ]:
CSF_data["gex"].obs["Recluster_ann2"].value_counts()

In [ ]:
marker_genes = {
        "T cell": ["CD3E", "LCK", "TRAC", "CD4", "CD8A", "CD8B"],
        "cDC": ["CLEC10A", "CD1C", "FLT3", "CLEC9A"],
        "Monocyte" : ["FCN1", "VCAN"], 
        "MG/BAM":  ["C1QA",  "TREM2", "APOE",  "LYVE1","FOLR2",  "C3", "CX3CR1", "SPP1"],
        "NK cell": ["NCAM1", "NCR1", "CD160", "NCR3"], 
        "B cell": [  "MS4A1", "CD19", "CD79A", "CD27", "CD38", "PRDM1", "MZB1", "IGHG1", "EIF2A", "MKI67"],
        "pDC": ["IL3RA", "PTCRA", "IRF7"],
        "APC-status": ["CIITA", "HLA-DRA", "HLA-DRB1"]}

#     "Microglia": ["APOE", "TREM2", "SORL1","CX3CR1", "SLC2A5",  "SPP1", "P2RY12", "TMEM119" ], # "Microglia": ["APOE", "SORL1", "SLC2A5", "BHLHE41", "SPP1", "P2RY12"], 
#     "Complement\nfactors": ["C1QA", "C1QB", "C1QC",  "CFD", "C3","C2",],
#     "Complement\nregulators": [ "CD55", "CD46", "SERPING1", "CD59", "CFP",  "PROS1"],
#     "Complement\nreceptors": ["ITGB2","C3AR1", "C5AR1","VSIG4", "ITGAM", "ITGAX", "C5AR2", "CD93",  "CR1"],
#     "B cell": [ "CD79A", "CD19", "TNFRSF13B", "IGHG1", "IGHG2", "IGHG3"], #"IGKC", "IGHM", "IGHD", "IGHG1", "IGHG2", "IGHG3"], 
#     "PB/PC": ["MKI67", "MS4A1", "PRDM1", "XBP1", "CD38",  "SLAMF7", "MZB1", ],

# Define desired category order
desired_order = ['T cell', "Ex-multiplet", "cDC", "T-cDC", "Monocyte", "T-Monocyte", "MG/BAM", "T-MG/BAM", "NK cell", "T-NK cell", 
                 "B cell", "T-B cell", "PBla", "T-PBla", "pDC", "T-pDC", "T-pDC-MG/BAM"]

# Reorder categories
CSF_data["gex"].obs["Recluster_ann2"] = CSF_data["gex"].obs["Recluster_ann2"].cat.reorder_categories(
    desired_order, ordered=True
)


dp = sc.pl.dotplot(
    CSF_data["gex"],
    marker_genes,
    groupby="Recluster_ann2",
    standard_scale="var",
    cmap="RdBu_r",
    return_fig=True,
    show=False,
)

dp.add_totals()
dp.style(dot_edge_color="black", dot_edge_lw=0.5)

# Build the figure
dp.make_figure()
dp.show()

# Save
dp.fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/94_CSF_reclustered2_dotplot.pdf",
    bbox_inches="tight",
    dpi=400,
)

## Quantify TCR usage in each cluster

In [ ]:
obs_df = CSF_data["gex"].obs.copy()
obs_df = obs_df[["Recluster_ann2", "chain_pairing", "scrublet_score"]]

#Define multiplet categories
obs_df["cat_multiplet"] = np.where((obs_df["Recluster_ann2"]  == "T cell") | (obs_df["Recluster_ann2"].str.contains("-")), "multiplet or T cell", "singlet")

#filter
obs_df = obs_df[obs_df["Recluster_ann2"] != "Apoptotic"]
obs_df = obs_df[obs_df["cat_multiplet"] == "multiplet or T cell"]

#Remove unused categories
obs_df["Recluster_ann2"]= obs_df["Recluster_ann2"].cat.remove_unused_categories()

#Count df
count_df = obs_df.groupby(["Recluster_ann2", "chain_pairing"]).size().reset_index(name="count")

# convert to fraction within each Recluster_ann_2
count_df["fraction"] = count_df["count"] / count_df.groupby("Recluster_ann2")["count"].transform("sum")

# Convert to wide format before stacking
plot_df = count_df.pivot(
    index="Recluster_ann2",
    columns="chain_pairing",
    values="fraction"
).fillna(0)

## Control x and y-axis order
#### X ordered according to number of cells
x_order = obs_df["Recluster_ann2"].value_counts().reset_index()["Recluster_ann2"].tolist()
plot_df = plot_df.reindex(x_order).fillna(0)

chain_order = ["No IR", "orphan VJ", "orphan VDJ", "single pair", "extra VJ", "extra VDJ", "two full chains"]
plot_df = plot_df.reindex(columns=[c for c in chain_order if c in plot_df.columns])

## getting the palette
palette = {'extra VDJ': '#4c72b0',
 'extra VJ': '#dd8452',
 'orphan VDJ': '#55a868',
 'orphan VJ': '#c44e52',
 'single pair': '#8172b3',
 'two full chains': '#937860',
 'No IR': '#D3D3D3'}

## plot using matplotlib
fig, ax = plt.subplots(figsize=(8, 4))

bottom = None
for col in plot_df.columns:
    if bottom is None:
        ax.bar(plot_df.index, plot_df[col], label=col, color=palette.get(col, "#333333"))
        bottom = plot_df[col].copy()
    else:
        ax.bar(plot_df.index, plot_df[col], bottom=bottom, label=col, color=palette.get(col, "#333333"))
        bottom += plot_df[col]

ax.set_ylabel("Fraction of cells")
ax.set_xlabel("Recluster_ann_2")
ax.legend(title="chain_pairing", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.xticks(rotation=90)
sb.despine()
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_quantification_of_VDJ_chains.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
obs_df["Recluster_ann2"].value_counts().reset_index()

In [ ]:
### Raw numbers instead
plot_df = count_df.pivot(
    index="Recluster_ann2",
    columns="chain_pairing",
    values="count"
).fillna(0)

## Control x and y-axis order
#### X ordered according to number of cells
x_order = obs_df["Recluster_ann2"].value_counts().reset_index()["Recluster_ann2"].tolist()
plot_df = plot_df.reindex(x_order).fillna(0)

chain_order = ["No IR", "orphan VJ", "orphan VDJ", "single pair", "extra VJ", "extra VDJ", "two full chains"]
plot_df = plot_df.reindex(columns=[c for c in chain_order if c in plot_df.columns])

## getting the palette
palette = {'extra VDJ': '#4c72b0',
 'extra VJ': '#dd8452',
 'orphan VDJ': '#55a868',
 'orphan VJ': '#c44e52',
 'single pair': '#8172b3',
 'two full chains': '#937860',
 'No IR': '#D3D3D3'}

## plot using matplotlib
fig, ax = plt.subplots(figsize=(8, 4))

bottom = None
for col in plot_df.columns:
    if bottom is None:
        ax.bar(plot_df.index, plot_df[col], label=col, color=palette.get(col, "#333333"))
        bottom = plot_df[col].copy()
    else:
        ax.bar(plot_df.index, plot_df[col], bottom=bottom, label=col, color=palette.get(col, "#333333"))
        bottom += plot_df[col]

ax.set_ylabel("Fraction of cells")
ax.set_xlabel("Recluster_ann_2")
ax.legend(title="chain_pairing", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.xticks(rotation=90)
sb.despine()
plt.tight_layout()
plt.show()


## Scrublet score and number of genes

In [ ]:
# Extract obs and compute medians
obs = CSF_data["gex"].obs
vals = CSF_data["gex"].obs["n_genes_by_counts"]
groups = CSF_data["gex"].obs["Recluster_ann2"]

# Calculate median per group
medians = obs.groupby("Recluster_ann2")["total_counts"].median()

# Sort groups by median
sorted_groups = medians.sort_values().index.tolist()

# Plot with sorted order
with plt.rc_context({"figure.figsize": (6, 3)}):
    ax = sc.pl.violin(
        CSF_data["gex"], 
        keys="total_counts",
        stripplot=False,
        groupby="Recluster_ann2",
        order=sorted_groups,     # 👈 pass sorted order
        multi_panel=False,
        rotation=45,
        show=False
    )

plt.xticks(rotation=45, ha='right')
# plt.axhline(y=200, linewidth=1.5, color="black", linestyle="--")
plt.ylabel("Total counts")

# Rasterize violin bodies
# for col in ax.collections:
#     col.set_rasterized(True)

sb.despine()
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_total_reads_doublets.pdf", format="pdf", dpi=400)
plt.show()


In [ ]:
# Extract obs and compute medians
obs = CSF_data["gex"].obs
vals = CSF_data["gex"].obs["n_genes_by_counts"]
groups = CSF_data["gex"].obs["Recluster_ann2"]

# Calculate median per group
medians = obs.groupby("Recluster_ann2")["n_genes_by_counts"].median()

# Sort groups by median
sorted_groups = medians.sort_values().index.tolist()

# Plot with sorted order
with plt.rc_context({"figure.figsize": (6, 3)}):
    ax = sc.pl.violin(
        CSF_data["gex"], 
        keys="n_genes_by_counts",
        stripplot=False,
        groupby="Recluster_ann2",
        order=sorted_groups,     # 👈 pass sorted order
        multi_panel=False,
        rotation=45,
        show=False
    )

plt.xticks(rotation=45, ha='right')
# plt.axhline(y=200, linewidth=1.5, color="black", linestyle="--")
plt.ylabel("Number of genes")

# Rasterize violin bodies
# for col in ax.collections:
#     col.set_rasterized(True)

sb.despine()
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_Number_of_genes_doublets.pdf", format="pdf", dpi=400)
plt.show()


In [ ]:
# Extract obs and compute medians
obs = CSF_data["gex"].obs
vals = CSF_data["gex"].obs["scrublet_score"]
groups = CSF_data["gex"].obs["Recluster_ann2"]

# Calculate median per group
medians = obs.groupby("Recluster_ann2")["scrublet_score"].median()

# Sort groups by median
sorted_groups = medians.sort_values().index.tolist()

# Plot with sorted order
with plt.rc_context({"figure.figsize": (6, 3)}):
    ax = sc.pl.violin(
        CSF_data["gex"], 
        keys="scrublet_score",
        stripplot=False,
        groupby="Recluster_ann2",
        order=sorted_groups,     # 👈 pass sorted order
        multi_panel=False,
        rotation=45,
        show=False
    )

plt.xticks(rotation=45, ha='right')
# plt.axhline(y=200, linewidth=1.5, color="black", linestyle="--")
plt.ylabel("Doublet score (scrublet)")

# Rasterize violin bodies
for col in ax.collections:
    col.set_rasterized(True)

sb.despine()
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_doublet_score_doublets.pdf", format="pdf", dpi=400)
plt.show()


# Recluster CSF multiplet to generate high resolution annotations

In [ ]:
## Include ex_multiplet
# CSF_data["gex"].obs["cat_multi_ext"] = pd.Categorical(np.where(CSF_data["gex"].obs["Recluster_ann2"] == "Ex-multiplet", "multiplet", CSF_data["gex"].obs["cat_multiplet"])) 

multiplet_adata = CSF_data.copy()
mu.pp.filter_obs(multiplet_adata["gex"], "cat_multiplet", lambda x: (x == "multiplet")) 
multiplet_adata.update()

In [ ]:
# ---
# Remove genes that are present in only very few cells 
# ---
mu.pp.filter_var(multiplet_adata["gex"], 'n_cells_by_counts', lambda x: x > 2)
multiplet_adata.update()

# ---
# Allign the modalities emphasizing cells in gex.
#   Make certain that the cells in airr and prot modalities are the same cells as are present in gex.
# ---
#Make sure only the indexes recorded in GEX are used in the o*ther modalities too
index_list = multiplet_adata["gex"].obs.index.tolist()

## define a cell index
multiplet_adata["airr"].obs["cell_index"] = multiplet_adata["airr"].obs.index
multiplet_adata["prot"].obs["cell_index"] = multiplet_adata["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
multiplet_adata["airr"].obs["filter_index"] = pd.Categorical(np.where(multiplet_adata["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
multiplet_adata["prot"].obs["filter_index"] = pd.Categorical(np.where(multiplet_adata["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that multiplet_adata["airr"].obs and multiplet_adata["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(multiplet_adata["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(multiplet_adata["prot"], 'filter_index', lambda x: x == "GEX_present")
multiplet_adata.update()

# ---
# When the .var object has been filtered, then you need to renormalize
# ---
multiplet_adata["gex"].X = multiplet_adata["gex"].layers["counts"].copy()
multiplet_adata["gex"].uns.pop("log1p", None) #Removes log1P if it exists
sc.pp.normalize_total(multiplet_adata["gex"], target_sum=1e4, exclude_highly_expressed=False)   #1e4 is important. Do not change
sc.pp.log1p(multiplet_adata["gex"])

In [ ]:
calculate_leiden(multiplet_adata["gex"], resolution=1, n_high_var=2000, remove_tcr_genes=True)   
calculate_umap(multiplet_adata["gex"], n_high_var=2000, remove_tcr_genes=True)

In [ ]:
plt.rcParams["figure.figsize"] = (6, 6)
fig = sc.pl.umap(multiplet_adata["gex"], color=["leiden_1"], legend_fontweight='bold', cmap="RdBu_r", add_outline=True, frameon= False , 
                 legend_fontoutline=2, legend_loc = "on data", size=60, return_fig = True, show=False)

# # Rasterize all scatter collections
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/XX_Final_umaps_CSF_highrest_doublets_clustered.pdf", 
    bbox_inches="tight",
    dpi=300
)
plt.show()

## 
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

In [ ]:
marker_genes = {
         "T cell": ["CD3E", "CD4", "CD8A", "CD8B"],
        "Effector genes" : ["CCL5",  "NKG7", "GZMB", "FASLG", "GNLY", "PRF1", "GZMA", "GZMK", "GZMH", "LAG3", "PDCD1", "TNF", "IFNG", "XCL1"],
        "Migr. & Adhesion" : ["CXCR6", "CXCL16", "CCR7", "CCL19"],  
        "NK cell": ["NCAM1", "NCR1", "CD160", "NCR3"],
        "cDC": ["FLT3", "CLEC9A", "CLEC10A", "CD1C"],
        "Monocyte" : ["FCN1", "VCAN"], 
        "BAM":  ["C1QA",  "TREM2",  "APOE",  "MS4A7", "C3", "CX3CR1", "FOLR2", "LYVE1", "F13A1", "STAB1", "MRC1"],
        "MGs" : ["SPP1",  "SLC2A5", "P2RY12",  "TMEM119", "OLFML3", "SPARC", "SERPINE2"], 
        "B cell": [  "MS4A1", "CD19", "CD79A", "CD27", "CD38", "PRDM1", "MZB1", "IGHG1", "EIF2A", "SDC1", "MKI67"],
        "pDC": ["IL3RA", "PTCRA", "IRF7"],
        "APC-status": ["CIITA", "HLA-DRA", "HLA-DRB1"]}

#     "Microglia": ["APOE", "TREM2", "SORL1","CX3CR1", "SLC2A5",  "SPP1", "P2RY12", "TMEM119" ], # "Microglia": ["APOE", "SORL1", "SLC2A5", "BHLHE41", "SPP1", "P2RY12"], 
#     "Complement\nfactors": ["C1QA", "C1QB", "C1QC",  "CFD", "C3","C2",],
#     "Complement\nregulators": [ "CD55", "CD46", "SERPING1", "CD59", "CFP",  "PROS1"],
#     "Complement\nreceptors": ["ITGB2","C3AR1", "C5AR1","VSIG4", "ITGAM", "ITGAX", "C5AR2", "CD93",  "CR1"],
#     "B cell": [ "CD79A", "CD19", "TNFRSF13B", "IGHG1", "IGHG2", "IGHG3"], #"IGKC", "IGHM", "IGHD", "IGHG1", "IGHG2", "IGHG3"], 
#     "PB/PC": ["MKI67", "MS4A1", "PRDM1", "XBP1", "CD38",  "SLAMF7", "MZB1", ],

# Define desired category order
desired_order = ["10", "18", "15","16", "2", "4", "8", "5", "14", "9", "0", "1", "7", "3", "11", "19",  "6", "12","17", "13"]

# # # Reorder categories
multiplet_adata["gex"].obs["leiden_1"] = multiplet_adata["gex"].obs["leiden_1"].cat.reorder_categories(
    desired_order, ordered=True
)


dp = sc.pl.dotplot(
    multiplet_adata["gex"],
    marker_genes,
    # dendrogram=True,
    groupby="leiden_1",
    standard_scale="var",
    cmap="RdBu_r",
    return_fig=True,
    show=False,
)

dp.add_totals()
dp.style(dot_edge_color="black", dot_edge_lw=0.5)

# Build the figure
# dp.make_figure()
dp.show()

# Save
dp.fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/94_CSF_DOUBLET_HighRes_dotplot.pdf",
    bbox_inches="tight",
    dpi=400,
)

In [ ]:
marker_genes = {
         "T cell": ["CD4", "CD8A", "CD8B"],
        "Effector genes" : ["GZMB", "GZMK"],
        "Migr. & Adhesion" : ["CXCR6", "CXCL16", "CCR7", "CCL19", "XCL2"],  
        "NK cell": ["NCAM1"],
        "cDC": ["FLT3", "CD1C"],
        "Monocyte" : ["FCN1"], 
        "BAM":  ["C1QA", "TREM2", "FOLR2", "LYVE1"],
        "MGs" : ["SPP1",  "P2RY12"], 
        "B cell": ["MS4A1", "CD19", "CD79A", "CD38", "PRDM1", "MZB1", "MKI67"],
        "pDC": ["IL3RA"]}

#     "Microglia": ["APOE", "TREM2", "SORL1","CX3CR1", "SLC2A5",  "SPP1", "P2RY12", "TMEM119" ], # "Microglia": ["APOE", "SORL1", "SLC2A5", "BHLHE41", "SPP1", "P2RY12"], 
#     "Complement\nfactors": ["C1QA", "C1QB", "C1QC",  "CFD", "C3","C2",],
#     "Complement\nregulators": [ "CD55", "CD46", "SERPING1", "CD59", "CFP",  "PROS1"],
#     "Complement\nreceptors": ["ITGB2","C3AR1", "C5AR1","VSIG4", "ITGAM", "ITGAX", "C5AR2", "CD93",  "CR1"],
#     "B cell": [ "CD79A", "CD19", "TNFRSF13B", "IGHG1", "IGHG2", "IGHG3"], #"IGKC", "IGHM", "IGHD", "IGHG1", "IGHG2", "IGHG3"], 
#     "PB/PC": ["MKI67", "MS4A1", "PRDM1", "XBP1", "CD38",  "SLAMF7", "MZB1", ],

# Define desired category order
desired_order = ["10", "18", "15","16", "2", "4", "8", "5", "14", "9", "0", "1", "7", "3", "11", "19",  "6", "12","17", "13"]

# # # Reorder categories
multiplet_adata["gex"].obs["leiden_1"] = multiplet_adata["gex"].obs["leiden_1"].cat.reorder_categories(
    desired_order, ordered=True
)


dp = sc.pl.dotplot(
    multiplet_adata["gex"],
    marker_genes,
    # dendrogram=True,
    groupby="leiden_1",
    standard_scale="var",
    edgecolor="black",
    cmap="RdBu_r",
    figsize = (10,6),
    return_fig=True,
    show=False,
)

dp.add_totals()

# Build the figure
# dp.make_figure()
dp.show()

# Save
dp.fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/94_CSF_DOUBLET_HighRes_dotplot.pdf",
    bbox_inches="tight",
    dpi=400,
)

## Kilian wants to see IL-6 and IL-8

In [ ]:
marker_genes = {
         "T cell": ["CD4", "CD8A", "CD8B"],
        "Effector genes" : ["GZMB", "GZMK"],
        "Migr. & Adhesion" : ["CXCR6", "CXCL16", "CCR7", "CCL19", "XCL2"],  
        "NK cell": ["NCAM1"],
        "cDC": ["FLT3", "CD1C"],
        "Monocyte" : ["FCN1"], 
        "BAM":  ["C1QA", "TREM2", "FOLR2", "LYVE1"],
        "MGs" : ["SPP1",  "P2RY12"], 
        "B cell": ["MS4A1", "CD19", "CD79A", "CD38", "PRDM1", "MZB1", "MKI67"],
        "pDC": ["IL3RA"], 
        "Other" : ["IL1B", "IL6", "IL10", "IL12B", "IL15", "IL17A", "IL18",
    "IL21", "IL23A", "TNF", "IFNG", "TGFB1",
    "CXCL8", "CXCL9", "CXCL10", "CXCL11",
    "CCL2", "CCL3", "CCL4", "CCL5", "CCL20",
    "GZMA", "GZMB", "GZMK", "GNLY", "PRF1", "FASLG"]}

#     "Microglia": ["APOE", "TREM2", "SORL1","CX3CR1", "SLC2A5",  "SPP1", "P2RY12", "TMEM119" ], # "Microglia": ["APOE", "SORL1", "SLC2A5", "BHLHE41", "SPP1", "P2RY12"], 
#     "Complement\nfactors": ["C1QA", "C1QB", "C1QC",  "CFD", "C3","C2",],
#     "Complement\nregulators": [ "CD55", "CD46", "SERPING1", "CD59", "CFP",  "PROS1"],
#     "Complement\nreceptors": ["ITGB2","C3AR1", "C5AR1","VSIG4", "ITGAM", "ITGAX", "C5AR2", "CD93",  "CR1"],
#     "B cell": [ "CD79A", "CD19", "TNFRSF13B", "IGHG1", "IGHG2", "IGHG3"], #"IGKC", "IGHM", "IGHD", "IGHG1", "IGHG2", "IGHG3"], 
#     "PB/PC": ["MKI67", "MS4A1", "PRDM1", "XBP1", "CD38",  "SLAMF7", "MZB1", ],

# Define desired category order
desired_order = ["10", "18", "15","16", "2", "4", "8", "5", "14", "9", "0", "1", "7", "3", "11", "19",  "6", "12","17", "13"]

# # # Reorder categories
multiplet_adata["gex"].obs["leiden_1"] = multiplet_adata["gex"].obs["leiden_1"].cat.reorder_categories(
    desired_order, ordered=True
)


dp = sc.pl.dotplot(
    multiplet_adata["gex"],
    marker_genes,
    # dendrogram=True,
    groupby="leiden_1",
    standard_scale="var",
    edgecolor="black",
    cmap="RdBu_r",
    figsize = (14,6),
    return_fig=True,
    show=False,
)

dp.add_totals()

# Build the figure
# dp.make_figure()
dp.show()

# Save
dp.fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/94_CSF_DOUBLET_HighRes_dotplot.pdf",
    bbox_inches="tight",
    dpi=400,
)

In [ ]:
plt.rcParams["figure.figsize"] = (6, 6)
fig = sc.pl.umap(multiplet_adata["gex"], color=[
    "IL1B", "IL6", "IL10", "IL12B", "IL15", "IL17A", "IL18",
    "IL21", "IL23A", "TNF", "IFNG", "TGFB1",
    "CXCL8", "CXCL9", "CXCL10", "CXCL11",
    "CCL2", "CCL3", "CCL4", "CCL5", "CCL20",
    "GZMA", "GZMB", "GZMK", "GNLY", "PRF1", "FASLG"], legend_fontweight='bold', cmap="hot_r", add_outline=True, frameon= True , 
                 legend_fontoutline=2, legend_loc = "on data", size=60, return_fig = True, show=False)

# # Rasterize all scatter collections
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

# fig.savefig(
#     "../../05_plots/02_Overview_and_doublet_analysis/XX_Final_umaps_CSFbiased_score.pdf", 
#     bbox_inches="tight",
#     dpi=300
# )
plt.show()

## 
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

# Show the CSF-biased score

In [ ]:
# --- Export gene scores --- 
CSFbiased_genescore = pd.read_csv("../../06_csv_outputs/DEG/CD8_CSF_biased_filtered_DEGs.csv")

print(len(CSFbiased_genescore), "genes were CSF-biased")

# --- Calculate scores --- 
sc.tl.score_genes(multiplet_adata["gex"], 
                      CSFbiased_genescore["names"].drop_duplicates().tolist(),  
                      score_name="CSF-biased score")


In [ ]:
plt.rcParams["figure.figsize"] = (6, 6)
fig = sc.pl.umap(multiplet_adata["gex"], color=["CSF-biased score"], legend_fontweight='bold', cmap="RdBu_r", add_outline=True, frameon= False , 
                 legend_fontoutline=2, legend_loc = "on data", size=60, return_fig = True, show=False)

# # Rasterize all scatter collections
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/XX_Final_umaps_CSFbiased_score.pdf", 
    bbox_inches="tight",
    dpi=300
)
plt.show()

## 
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

In [ ]:
# Extract obs and compute medians
obs = multiplet_adata["gex"].obs
vals = multiplet_adata["gex"].obs["CSF-biased score"]
groups = multiplet_adata["gex"].obs["leiden_1"]

# Calculate median per group
medians = obs.groupby("leiden_1")["CSF-biased score"].median()

# Sort groups by median
sorted_groups = medians.sort_values().index.tolist()

# Plot with sorted order
with plt.rc_context({"figure.figsize": (6, 3)}):
    ax = sc.pl.violin(
        multiplet_adata["gex"], 
        keys="CSF-biased score",
        stripplot=False,
        groupby="leiden_1",
        order=sorted_groups,     # 👈 pass sorted order
        multi_panel=False,
        rotation=45,
        show=False
    )

plt.xticks(rotation=45, ha='right')
# plt.axhline(y=200, linewidth=1.5, color="black", linestyle="--")
plt.ylabel("CSF-biased score")

# Rasterize violin bodies
for col in ax.collections:
    col.set_rasterized(True)

sb.despine()
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_doublet_CSFbiased_score.pdf", format="pdf", dpi=400)
plt.show()


# Show antigen-specificities

In [ ]:
total_ann = pd.read_csv("../../06_csv_outputs/Clones/2026_7_total_VirusAnnotationsPerClone.csv")
mult_ann = pd.read_csv("../../03_annotation/11_multimer_experiment_ann/2_MultPostClones.csv")
# LCL_list = pd.read_csv("../../06_csv_outputs/Clones/2026_4_BLCL_reactive_d19.csv")
# LCL_list = LCL_list["Clone_handle"].drop_duplicates().tolist()

# Annotate .obs

In [ ]:
#Annotate "Virus_extended" to contain the full Virus annotations combining multimer and databases searches. 
clone_dict = total_ann.set_index("clone_handle")["Virus_final"].to_dict()
multiplet_adata["gex"].obs["Virus_extended"] = multiplet_adata["gex"].obs["clone_handle"].map(clone_dict)

#Annotate "Antigen_extended" to contain the full Antigen annotations like for Virus_extended
clone_dict = total_ann.set_index("clone_handle")["Antigen_final"].to_dict()
multiplet_adata["gex"].obs["Antigen_extended"] = multiplet_adata["gex"].obs["clone_handle"].map(clone_dict)

#Virus, i.e. multimer virus make a dictionary, then map it's values to .obs
clone_dict = mult_ann.set_index("clone_handle")["Virus"].to_dict()
multiplet_adata["gex"].obs["Virus"] = multiplet_adata["gex"].obs["clone_handle"].map(clone_dict)
#Add antigen too
clone_dict = mult_ann.set_index("clone_handle")["Antigen"].to_dict()
multiplet_adata["gex"].obs["Antigen"] = multiplet_adata["gex"].obs["clone_handle"].map(clone_dict)

## Should CMV-specific cells in CMV(-) donors be excluded? 
multiplet_adata["gex"].obs["Virus_extended"] = np.where((multiplet_adata["gex"].obs["donor"].isin(["Pt4", "Pt5", "Pt8", "Pt11", "Pt17", "Pt20", "Pt21"])) & (multiplet_adata["gex"].obs["Virus_extended"] == "CMV"), np.nan,
                                              multiplet_adata["gex"].obs["Virus_extended"])#### annotate LCL reactive

# ## annotate LCL-reactive
# multiplet_adata["gex"].obs["IR_VDJ_1_junction_aa"] = multiplet_adata["airr"].obs["IR_VDJ_1_junction_aa"].copy()
# multiplet_adata["gex"].obs["clone_handle2"] = multiplet_adata["gex"].obs["donor"].astype(str) + "_" + multiplet_adata["gex"].obs["IR_VDJ_1_junction_aa"].astype(str) 
# multiplet_adata["gex"].obs["LCL_reactive"] = np.where(multiplet_adata["gex"].obs["clone_handle2"].isin(LCL_list), "LCL-reactive", "Non-reactive")
# ## EXTEND VIRUS EXTENDED
# multiplet_adata["gex"].obs["Virus_extended"] = np.where((multiplet_adata["gex"].obs["Virus_extended"].isna()) & (multiplet_adata["gex"].obs["LCL_reactive"] == "LCL-reactive"), "LCL-reactive", multiplet_adata["gex"].obs["Virus_extended"])

In [ ]:
virus_palette = {
    "CMV" : "#4C72B0", 
    "EBV" : "#DD8452", 
    "FLU-A" : "#55A868", 
    "HHV-1" : "#9467BD", 
    "VZV" : "#17BECF", 
    "5-OP-RU" : "#000000", 
    "Unknown" : "#F5F5F5"
}

## Kilian wants the umap without 5-OP-RU
multiplet_adata["gex"].obs["Virus_extended_edit"] = pd.Categorical(np.where(multiplet_adata["gex"].obs["Virus_extended"] == "5-OP-RU", np.nan, multiplet_adata["gex"].obs["Virus_extended"]))

plt.rcParams["figure.figsize"] = (6, 6)
fig = sc.pl.umap(multiplet_adata["gex"], color=["Virus_extended_edit"], palette=virus_palette, legend_fontweight='bold', cmap="RdBu_r", frameon= False , 
                 legend_fontoutline=2, size=140, return_fig = True, show=False)



# # Rasterize all scatter col")
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/XX_Final_umaps_CSF_highrest_doublets_clustered.pdf", 
    bbox_inches="tight",
    dpi=300
)
plt.show()

## 
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

# Define distribution of virus-specific cells for clusters with more than 5 cells annotated

In [ ]:
obs_df = multiplet_adata["gex"].obs.copy()
temp_counts = obs_df[["leiden_1", "Virus_extended"]].value_counts().reset_index().groupby("leiden_1")["count"].sum().reset_index().sort_values("count", ascending=False)
leiden_list = temp_counts[temp_counts["count"] > 5]["leiden_1"].tolist()
leiden_list = leiden_list + ["12"]  #Kilian also wants to show cluster 122
obs_df = obs_df[obs_df["leiden_1"].isin(leiden_list)]
obs_df = obs_df[obs_df["Virus_extended"] != "5-OP-RU"] #Kilian wants to not show 5-OP-RU annotations

#Drop unusedcategories
for col in obs_df.select_dtypes(include="category").columns:
  obs_df[col] = obs_df[col].cat.remove_unused_categories()

#Count df
count_df = obs_df.groupby(["leiden_1", "Virus_extended"]).size().reset_index(name="count")

#pivot and transform to percentages
count_df = count_df.pivot(index="leiden_1", columns = "Virus_extended", values="count")
print(count_df)
count_df = count_df.div(count_df.sum(axis=1), axis=0)*100

# Plot
plot_df = count_df.reset_index()

# Convert to long format
plot_long = plot_df.melt(
    id_vars="leiden_1",
    var_name="Virus",
    value_name="Percentage"
)

# Order of viruses (optional)
virus_order = ["5-OP-RU", "CMV", "EBV", "FLU-A"]
plot_long["Virus"] = pd.Categorical(
    plot_long["Virus"],
    categories=virus_order,
    ordered=True
)

# Pivot back so we can stack
plot_wide = (
    plot_long
    .pivot(index="leiden_1", columns="Virus", values="Percentage")
    .fillna(0)
)

virus_palette = {
    "CMV" : "#4C72B0", 
    "EBV" : "#DD8452", 
    "FLU-A" : "#55A868", 
    "HHV-1" : "#9467BD", 
    "VZV" : "#17BECF", 
    "5-OP-RU" : "#000000", 
    "Unknown" : "#F5F5F5"
}

colors = [virus_palette[v] for v in plot_wide.columns]

fig, ax = plt.subplots(figsize=(6.3,2.5))

bottom = pd.Series(0, index=plot_wide.index)

for color, virus in zip(colors, plot_wide.columns):
    ax.bar(
        plot_wide.index.astype(str),
        plot_wide[virus],
        bottom=bottom,
        label=virus,
        color=color,
        edgecolor="black",
        linewidth=0.5,
    )
    bottom += plot_wide[virus]

ax.set_ylabel("Percentage (%)")
ax.set_xlabel("Leiden cluster")
ax.set_ylim(0, 100)
ax.legend(title="Virus", bbox_to_anchor=(1.02, 1), loc="upper left")
sb.despine()

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/95_antigen-specific preferencec.pdf", format="pdf", dpi=400)
plt.show()

# % of cells across each cluster

In [ ]:
obs_df = multiplet_adata["gex"].obs.copy()

#Rename nan to Unknown count all cells
obs_df["Virus_extended"] = obs_df["Virus_extended"].cat.add_categories(["Unknown"])
obs_df["Virus_extended"] = pd.Categorical(np.where(obs_df["Virus_extended"].isna(), "Unknown", obs_df["Virus_extended"]))

temp_counts = obs_df[["leiden_1", "Virus_extended"]].value_counts().reset_index().groupby("leiden_1")["count"].sum().reset_index().sort_values("count", ascending=False)
#Leiden_list is defined by the previous figure
obs_df = obs_df[obs_df["leiden_1"].isin(leiden_list)]
obs_df = obs_df[obs_df["Virus_extended"] != "5-OP-RU"] #Kilian wants to not show 5-OP-RU annotations

#Drop unusedcategories
for col in obs_df.select_dtypes(include="category").columns:
  obs_df[col] = obs_df[col].cat.remove_unused_categories()

#Count df
count_df = obs_df.groupby(["leiden_1", "Virus_extended"]).size().reset_index(name="count")

#pivot and transform to percentages
count_df = count_df.pivot(index="leiden_1", columns = "Virus_extended", values="count")
print(count_df)
count_df = count_df.div(count_df.sum(axis=1), axis=0)*100

# Plot
plot_df = count_df.reset_index()

#Select relevant columns
plot_df = plot_df[["leiden_1", "CMV", "EBV", "FLU-A"]]

# Convert to long format
plot_long = plot_df.melt(
    id_vars="leiden_1",
    var_name="Virus",
    value_name="Percentage"
)


plt.figure(figsize=(6.3,2.5))

# Barplot
sb.barplot(
    data=plot_long,
    x="leiden_1",
    y="Percentage",
    hue="Virus",
    alpha=1,
    edgecolor="black",
    linewidth=0.5
)


plt.xlabel("Leiden cluster")
plt.ylabel("Virus (%)")
plt.xticks(rotation=45)
plt.legend(title="Virus", bbox_to_anchor=(1.02, 1), loc="upper left")

sb.despine()
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/96_EBV-specific_cells_of_observed_doublets.pdf", format="pdf", dpi=400)
plt.show()

# Report - which donors are these coming from?
## Clone lists

In [ ]:
obs_df = multiplet_adata["gex"].obs.copy()

#Rename nan to Unknown count all cells
obs_df["Virus_extended"] = obs_df["Virus_extended"].cat.add_categories(["Unknown"])
obs_df["Virus_extended"] = pd.Categorical(np.where(obs_df["Virus_extended"].isna(), "Unknown", obs_df["Virus_extended"]))

temp_counts = obs_df[["leiden_1", "Virus_extended"]].value_counts().reset_index().groupby("leiden_1")["count"].sum().reset_index().sort_values("count", ascending=False)
#Leiden_list is defined by the previous figure
obs_df = obs_df[obs_df["leiden_1"].isin(leiden_list)]
obs_df = obs_df[obs_df["Virus_extended"] != "5-OP-RU"] #Kilian wants to not show 5-OP-RU annotations

## Save all clonotypes interacting with B cell (need it later)
T_B_clone_list_all = obs_df[obs_df["leiden_1"] == "6"][["Virus_extended", "clone_handle", "catpat", "donor"]].value_counts().reset_index()
T_B_clone_list_all = T_B_clone_list_all["clone_handle"].tolist()

## Save all clonotypes interacting with B cell (need it later)
TBAM_clone_list_all = obs_df[obs_df["leiden_1"].isin(["0", "1", "3", "7", "9", "11"])][["Virus_extended", "clone_handle", "catpat", "donor"]].value_counts().reset_index()
TBAM_clone_list_all = TBAM_clone_list_all["clone_handle"].tolist()

# Save TPB list
TPB_clone_list_all = obs_df[obs_df["leiden_1"] == "12"][["Virus_extended", "clone_handle", "catpat", "donor"]].value_counts().reset_index()
TPB_clone_list_all = TPB_clone_list_all["clone_handle"].tolist()

## Get only the annotated cells
obs_df = obs_df[obs_df["Virus_extended"]!= "Unknown"]

## Save all clonotypes interacting with B cell (need it later)
T_B_clone_list_agsp = obs_df[obs_df["leiden_1"] == "6"][["Virus_extended", "clone_handle", "catpat", "donor"]].value_counts().reset_index()
T_B_clone_list_agsp = T_B_clone_list_agsp["clone_handle"].tolist()

## Save all clonotypes interacting with B cell (need it later)
TBAM_clone_list_agsp = obs_df[obs_df["leiden_1"].isin(["0", "1", "3", "7", "9", "11"])][["Virus_extended", "clone_handle", "catpat", "donor"]].value_counts().reset_index()
TBAM_clone_list_agsp = TBAM_clone_list_agsp["clone_handle"].tolist()

## Save all clonotypes interacting with PBla cell (need it later)
TBla_clone_list_agsp = obs_df[obs_df["leiden_1"].isin(["12"])][["Virus_extended", "clone_handle", "catpat", "donor"]].value_counts().reset_index()
TBla_clone_list_agsp = TBla_clone_list_agsp["clone_handle"].tolist()

# Report
# obs_df[["Virus_extended", "catpat", "donor"]].value_counts()

In [ ]:
## Load cd8data
cd8data = mu.read(filename="../../04_data_objects/06_preprocessed_v2/2026_6_CD8_annotated.h5mu")
total_ann = pd.read_csv("../../06_csv_outputs/Clones/2026_7_total_VirusAnnotationsPerClone.csv")


#Annotate "Virus_extended" to contain the full Virus annotations combining multimer and databases searches. 
clone_dict = total_ann.set_index("clone_handle")["Virus_final"].to_dict()
cd8data["gex"].obs["Virus_extended"] = cd8data["gex"].obs["clone_handle"].map(clone_dict)

#Annotate "Antigen_extended" to contain the full Antigen annotations like for Virus_extended
clone_dict = total_ann.set_index("clone_handle")["Antigen_final"].to_dict()
cd8data["gex"].obs["Antigen_extended"] = cd8data["gex"].obs["clone_handle"].map(clone_dict)

## Should CMV-specific cells in CMV(-) donors be excluded? 
cd8data["gex"].obs["Virus_extended"] = np.where((cd8data["gex"].obs["donor"].isin(["Pt4", "Pt5", "Pt8", "Pt11", "Pt17", "Pt20", "Pt21"])) & (cd8data["gex"].obs["Virus_extended"] == "CMV"), np.nan,
                                              cd8data["gex"].obs["Virus_extended"])#### annotate LCL reactive


In [ ]:
pt12_df = cd8data["gex"].obs[["donor", "Virus_extended", "Compartment"]].value_counts().reset_index()
pt12_df[pt12_df["donor"] == "Pt12"]

In [ ]:
obs_df = cd8data["gex"].obs.copy()

## Filter to get to CSF clusters
obs_df = obs_df[obs_df["Compartment"]== "CSF"]
count_df = obs_df[["donor", "catpat", "L3_ann", "Virus_extended"]].value_counts().reset_index()

## Relevant patients are Pt17, Pt20, Pt14, Pt18
count_df = count_df[count_df["donor"].isin(["Pt14", "Pt17", "Pt18", "Pt20"])]
count_df[count_df["L3_ann"] == "TCSF (CXCR6hi XCL1hi)"]

In [ ]:
## Define B-cell singlet numbers
obs_df = CSF_data["gex"].obs.copy()
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()
count_df
count_df[count_df["Recluster_ann2"] == "B cell"]

# T-B clonotype analysis

## Donor count

In [ ]:
obs = CSF_data["gex"].obs
obs["Recluster_ann2"].drop_duplicates().tolist()

In [ ]:
obs = CSF_data["gex"].obs
print(obs[obs["Recluster_ann"].isin(["B cell", "T-B cell"])][["Recluster_ann2", "catpat", "donor"]].sort_values(["Recluster_ann2", "catpat"]).drop_duplicates())
## 4 MS and 5 NIC has B cells
## 4 MS and 5 NIC has T-B cells

B_donor_list = obs[obs["Recluster_ann"].isin(["B cell", "T-B cell"])][["Recluster_ann2", "catpat", "donor"]].value_counts().reset_index()["donor"].drop_duplicates().tolist()

exclude = {"Pt4", "Pt5"}
B_donor_list = [x for x in B_donor_list if x not in exclude]
B_donor_list

## Sister cells in PB and CSF

In [ ]:
## Annotate where all T-B clones are
cd8data["gex"].obs["TB_highlight"] = pd.Categorical(np.where((cd8data["gex"].obs["donor"].isin(B_donor_list)) & (cd8data["gex"].obs["clone_handle"].isin(T_B_clone_list_all)), "T-B clones", 
                                                    np.where(cd8data["gex"].obs["donor"].isin(B_donor_list), "Other cells", "Other cells")), 
                                                   ordered=True, 
                                                   categories = ["T-B clones", 
                                                                "Other cells"])

In [ ]:
obs = cd8data["gex"].obs

# Get UMAP coordinates
umap = cd8data["gex"].obsm["X_umap"]

# Define masks
mask_other = obs["TB_highlight"] == "Other cells"
mask_tb = obs["TB_highlight"] == "T-B clones"

palette_highlight = {
    "T-B clones" : "#000000", 
    "Other cells" : "#D3D3D3", 
}


fig, ax = plt.subplots(figsize=(6,6))

# Layer 1: Other donors (bottom)
ax.scatter(
    umap[mask_other, 0],
    umap[mask_other, 1],
    s=1,
    c=palette_highlight["Other cells"],
    linewidths=0,
    rasterized=True
)


# Layer 2: Pt12 T-B clones (top)
ax.scatter(
    umap[mask_tb, 0],
    umap[mask_tb, 1],
    s=20,
    c=palette_highlight["T-B clones"],
    linewidths=0,
    rasterized=True
)

ax.set_axis_off()

plt.tight_layout()

fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/XX_Highlight_CD20_TBclones_alldonors.pdf",
    bbox_inches="tight",
    dpi=300
)

plt.show()

In [ ]:
## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["donor"].isin(B_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
Donor_all = obs_df[["L3_ann"]].value_counts().reset_index()
Donor_all["source"] = "All CD8 T cell"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "EBV"]
obs_df = obs_df[obs_df["donor"].isin(B_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
EBV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
EBV_agspf["source"] = "EBV-specific clones"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "CMV"]
obs_df = obs_df[obs_df["donor"].isin(B_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
CMV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
CMV_agspf["source"] = "CMV-specific clones"

## Get the FLU-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "FLU-A"]
obs_df = obs_df[obs_df["donor"].isin(B_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
FLU_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
FLU_agspf["source"] = "FLU-A-specific clones"

# ## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(T_B_clone_list_all)]
obs_df = obs_df[obs_df["donor"].isin(B_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_all = obs_df[["L3_ann"]].value_counts().reset_index()
TB_all["source"] = "T-B clones"

## Get the clonotypes of interest (T_B_clone_list_agsp)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(T_B_clone_list_agsp)]
obs_df = obs_df[obs_df["donor"].isin(B_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
TB_agspf["source"] = "EBV-specific T-B clones"

## Concat, pivot and transform
count_df = pd.concat([Donor_all, CMV_agspf, FLU_agspf, EBV_agspf, TB_all, TB_agspf])
count_df = count_df.pivot(index = "source", columns = "L3_ann", values="count")
print("Number of cells for each category")
print(count_df.sum(axis=1))

#Transform into percentages
count_df = count_df.div(count_df.sum(axis=1), axis=0)*100


cluster_color_dict = {
    # --- Pastel colors / TN ---
    "0":  "#dbe9f6",  # very light blue
    "1":  "#c7e9c0",  # pastel green
    "2":  "#e2d9f3",  # light lavender
    "11": "#fdd0a2",  # pastel peach

    # --- TCMs ---
    "7":  "#74c476",  # green
    "13": "#31a354",  # dark green
    "21": "#3BFD16",  # neon

    # --- TEM ---
    "4":  "#fdae6b",  # light orange
    "8":  "#fd8d3c",  # orange

    # --- TEMRA ---
    "9":  "#fb6a4a",  # soft red
    "10": "#8B0000",  # dark red

    # --- TE ---
    "6":  "#C00000",  # red
    "12": "#843c39",  # brown red

    # --- Activated ---
    "17": "#f768a1",  # pink
    "19": "#c51b8a",  # 
    "20": "#FF1B76",  # 
    "22": "#d9d9d9",  # 

    # --- Gamma delta ---
    "15": "#737373",  # Dark grey

    # --- CSF blue / purple-blue gradient ---
    "3":  "#6baed6",  # light blue
    "5":  "#3182bd",  # medium blue
    "14": "#9e9ac8",  # light purple
    "16": "#756bb1",  # purple
    "18": "#1f4fbf",  # dark blue

}

# Annotate a dataframe that contains L3_ann too
L3ann_df = cd8data["gex"].obs[["leiden_1.8", "L3_ann"]].drop_duplicates().copy()
L3ann_df["cluster_colors"] = L3ann_df["leiden_1.8"].map(cluster_color_dict)

## Create L3_ann_dict
clusters = L3ann_df["L3_ann"]
colors = L3ann_df["cluster_colors"]
L3_colors = dict(zip(clusters, colors))

#plot distribution changes
count_df = count_df.reset_index()
count_df["source"] = pd.Categorical(count_df["source"], ordered=True, categories = ["All CD8 T cell", "FLU-A-specific clones", "CMV-specific clones", "EBV-specific clones", "T-B clones",  "EBV-specific T-B clones"])

# Sort and keep source as index
count_df = count_df.sort_values("source").set_index("source")

fig, ax = plt.subplots(figsize=(3,3))

count_df.plot(
    kind="bar",
    stacked=True,
    color=L3_colors,
    edgecolor="white",
    linewidth=0.2,
    width=0.80,
    ax=ax
)

ax.set_ylim(0,100)
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
# ax.set_xticklabels([])

## export the legend once, and remove it again
# plt.legend(title="L3_ann", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.get_legend().remove()

ax.grid(False)

plt.title("Of PB CD8 T cell")
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/97_Phenotype_of_all_clones_outside_T-B_alldonors.pdf")
plt.show()

In [ ]:
## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["donor"].isin(B_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
Donor_all = obs_df[["L3_ann"]].value_counts().reset_index()
Donor_all["source"] = "All CD8 T cell"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "EBV"]
obs_df = obs_df[obs_df["donor"].isin(B_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
EBV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
EBV_agspf["source"] = "EBV-specific clones"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "CMV"]
obs_df = obs_df[obs_df["donor"].isin(B_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
CMV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
CMV_agspf["source"] = "CMV-specific clones"

## Get the FLU-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "FLU-A"]
obs_df = obs_df[obs_df["donor"].isin(B_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
FLU_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
FLU_agspf["source"] = "FLU-A-specific clones"

# ## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(T_B_clone_list_all)]
obs_df = obs_df[obs_df["donor"].isin(B_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_all = obs_df[["L3_ann"]].value_counts().reset_index()
TB_all["source"] = "T-B clones"

## Get the clonotypes of interest (T_B_clone_list_agsp)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(T_B_clone_list_agsp)]
obs_df = obs_df[obs_df["donor"].isin(B_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
TB_agspf["source"] = "EBV-specific T-B clones"

## Concat, pivot and transform
count_df = pd.concat([Donor_all, CMV_agspf, FLU_agspf, EBV_agspf, TB_all, TB_agspf])
count_df = count_df.pivot(index = "source", columns = "L3_ann", values="count")
print("Number of cells for each category")
print(count_df.sum(axis=1))

#Transform into percentages
count_df = count_df.div(count_df.sum(axis=1), axis=0)*100


cluster_color_dict = {
    # --- Pastel colors / TN ---
    "0":  "#dbe9f6",  # very light blue
    "1":  "#c7e9c0",  # pastel green
    "2":  "#e2d9f3",  # light lavender
    "11": "#fdd0a2",  # pastel peach

    # --- TCMs ---
    "7":  "#74c476",  # green
    "13": "#31a354",  # dark green
    "21": "#3BFD16",  # neon

    # --- TEM ---
    "4":  "#fdae6b",  # light orange
    "8":  "#fd8d3c",  # orange

    # --- TEMRA ---
    "9":  "#fb6a4a",  # soft red
    "10": "#8B0000",  # dark red

    # --- TE ---
    "6":  "#C00000",  # red
    "12": "#843c39",  # brown red

    # --- Activated ---
    "17": "#f768a1",  # pink
    "19": "#c51b8a",  # 
    "20": "#FF1B76",  # 
    "22": "#d9d9d9",  # 

    # --- Gamma delta ---
    "15": "#737373",  # Dark grey

    # --- CSF blue / purple-blue gradient ---
    "3":  "#6baed6",  # light blue
    "5":  "#3182bd",  # medium blue
    "14": "#9e9ac8",  # light purple
    "16": "#756bb1",  # purple
    "18": "#1f4fbf",  # dark blue

}

# Annotate a dataframe that contains L3_ann too
L3ann_df = cd8data["gex"].obs[["leiden_1.8", "L3_ann"]].drop_duplicates().copy()
L3ann_df["cluster_colors"] = L3ann_df["leiden_1.8"].map(cluster_color_dict)

## Create L3_ann_dict
clusters = L3ann_df["L3_ann"]
colors = L3ann_df["cluster_colors"]
L3_colors = dict(zip(clusters, colors))

#plot distribution changes
count_df = count_df.reset_index()
count_df["source"] = pd.Categorical(count_df["source"], ordered=True, categories = ["All CD8 T cell", "FLU-A-specific clones", "CMV-specific clones", "EBV-specific clones", "T-B clones",  "EBV-specific T-B clones"])

# Sort and keep source as index
count_df = count_df.sort_values("source").set_index("source")

fig, ax = plt.subplots(figsize=(3,3))

count_df.plot(
    kind="bar",
    stacked=True,
    color=L3_colors,
    edgecolor="white",
    linewidth=0.2,
    width=0.80,
    ax=ax
)

ax.set_ylim(0,100)
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
# ax.set_xticklabels([])

## export the legend once, and remove it again
# plt.legend(title="L3_ann", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.get_legend().remove()

ax.grid(False)

plt.title("Of CSF CD8 T cell")
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/97_Phenotype_of_all_clones_outside_T-B_alldonors_CSF.pdf")
plt.show()

# GZMK-expressing in clonotypes interacing with B cell

In [ ]:
# Work with clones of interest (those with an annotated antigen-specificity)
mtemp = cd8data.copy()
print(len(mtemp["gex"]))
mu.pp.filter_obs(mtemp["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"])))
print(len(mtemp["gex"]))

# Remove unused categories
for col in mtemp["gex"].obs.select_dtypes(include="category").columns:
    mtemp["gex"].obs[col] = (
        mtemp["gex"].obs[col].cat.remove_unused_categories()
    )

## Transfer GZMK to .obs
# Get index of GZMK
gene_idx = mtemp["gex"].var_names.get_loc("GZMK")

# Extract expression (handle sparse matrices safely)
gzmk_expr = mtemp["gex"].X[:, gene_idx]

# Convert to dense if needed
if not isinstance(gzmk_expr, np.ndarray):
    gzmk_expr = gzmk_expr.toarray().flatten()
else:
    gzmk_expr = gzmk_expr.flatten()

# Add to obs
mtemp["gex"].obs["GZMK_norm"] = gzmk_expr

## Filter
obs_df = mtemp["gex"].obs[mtemp["gex"].obs["Compartment"] == "CSF"].copy()
obs_df = obs_df[obs_df["donor"].isin(Patients_with_B_cells)]

## Define categories
All_df = obs_df[["clone_handle", "GZMK_norm"]].copy()
All_df = All_df.groupby(["clone_handle"], observed=True)["GZMK_norm"].mean().reset_index()
All_df["source"] = "All CD8 T cell"

CMV_df = obs_df[obs_df["Virus_extended"] =="CMV"][["clone_handle", "GZMK_norm"]].copy()
CMV_df = CMV_df.groupby(["clone_handle"], observed=True)["GZMK_norm"].mean().reset_index()
CMV_df["source"] = "CMV-specific"

FLU_df = obs_df[obs_df["Virus_extended"] =="FLU-A"][["clone_handle", "GZMK_norm"]].copy()
FLU_df = FLU_df.groupby(["clone_handle"], observed=True)["GZMK_norm"].mean().reset_index()
FLU_df["source"] = "FLU-specific"

EBV_df = obs_df[obs_df["Virus_extended"] =="EBV"][["clone_handle", "GZMK_norm"]].copy()
EBV_df = EBV_df.groupby(["clone_handle"], observed=True)["GZMK_norm"].mean().reset_index()
EBV_df["source"] = "EBV-specific"

TX_df = obs_df[obs_df["clone_handle"].isin(TB_total_list)][["clone_handle", "GZMK_norm"]].copy()
TX_df = TX_df.groupby(["clone_handle"], observed=True)["GZMK_norm"].mean().reset_index()
TX_df["source"] = "T-B clones"

TX_ag_df = obs_df[(obs_df["clone_handle"].isin(TB_total_list)) & obs_df["Virus_extended"].notna() ][["clone_handle", "GZMK_norm"]].copy()
TX_ag_df = TX_ag_df.groupby(["clone_handle"], observed=True)["GZMK_norm"].mean().reset_index()
TX_ag_df["source"] = "EBV-specific T-B clones"

## T-BAM
TX_df2 = obs_df[obs_df["clone_handle"].isin(TBAM_clone_list_all)][["clone_handle", "GZMK_norm"]].copy()
TX_df2 = TX_df2.groupby(["clone_handle"], observed=True)["GZMK_norm"].mean().reset_index()
TX_df2["source"] = "T-BAM clones"

## Concat, pivot and transform
count_df = pd.concat([All_df, CMV_df, FLU_df, EBV_df, TX_df, TX_ag_df, TX_df2])
count_df.groupby("source", observed=True).size()
print("Number of clones for each category")

## plot_df
plot_df = pd.concat([All_df, CMV_df, FLU_df, EBV_df, TX_df, TX_ag_df, TX_df2]).reset_index()

#plotting order: 
order =  ["All CD8 T cell", "CMV-specific", "FLU-specific", "EBV-specific", "T-B clones", "EBV-specific T-B clones", "T-BAM clones"]

fig, ax = plt.subplots(figsize=(4,4))

# Boxplot
sb.boxplot(
    data=plot_df,
    x="source",
    y="GZMK_norm",
    width=0.5,
    color="white",
    linecolor="#4D4D4D", 
    linewidth=0.5,
    fliersize=0,
    zorder=2,
    order=order,
    ax=ax
)

# Set transparency of the boxes
for patch in ax.patches:
    patch.set_alpha(0.90)

# Stripplot
sb.stripplot(
    data=plot_df,
    x="source",
    y="GZMK_norm",
    jitter=0.25,
    size=4,
    alpha=0.7,
    color="black",
    zorder=1,
    order=order,
    ax=ax
)

plt.ylabel("Mean GZMK expr.\n")
plt.xticks(rotation=90)
sb.despine()
plt.grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/02_Overview_and_doublet_analysis/99_GZMK_category_CSF_boxplots.pdf", format="pdf", dpi=400)
plt.show()

# Virus-specific clonotypes with sister cells in PB & CSF - All donors w. MG/BAM

In [ ]:
## Which donors have B cell? 
obs_df = CSF_data["gex"].obs.copy()
obs_df = obs_df[obs_df["catpat"] != "IC"]
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()
any_b = count_df[count_df["Recluster_ann2"].isin(["MG/BAM", "T-MG/BAM"])]
plot_df = any_b.groupby(["donor", "catpat"], observed = True)["count"].sum().reset_index()

# Order viruses by counts
order = ["Pt1","Pt8", "Pt18", "Pt19", "Pt21",  "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]

# Plot
fig, ax = plt.subplots(figsize=(4,2))

sb.barplot(data=plot_df, x="donor", y="count", ax=ax, order=order)

# Add count labels on top of bars
for bar in ax.patches:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=11
    )

plt.xticks(rotation=0)
plt.ylabel("No of B cell\nincluding multiplet")
plt.xlabel("")

# Move legend to the right
# ax.legend(title="Compartment", bbox_to_anchor=(1.05, 1), loc="upper left")

sb.despine()
ax.grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/02_Overview_and_doublet_analysis/98_NumberOfBcells_including_multiplet.pdf")
plt.show()

In [ ]:
## Annotate where all T-B clones are in patient 20
Patients_with_X_cells = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21", "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
cd8data["gex"].obs["TX_highlight"] = pd.Categorical(np.where((cd8data["gex"].obs["donor"].isin(Patients_with_X_cells)) & (cd8data["gex"].obs["clone_handle"].isin(TBAM_clone_list_all)), "T-BAM clones", 
                                                    np.where(cd8data["gex"].obs["donor"].isin(Patients_with_X_cells), "Other cells", "Other cells")), 
                                                   ordered=True, 
                                                   categories = ["T-BAM clones", 
                                                                "Other cells"])

In [ ]:
obs = cd8data["gex"].obs

# Get UMAP coordinates
umap = cd8data["gex"].obsm["X_umap"]

# Define masks
mask_other = obs["TX_highlight"] == "Other cells"
mask_tb = obs["TX_highlight"] == "T-BAM clones"

palette_highlight = {
    "T-BAM clones" : "#000000", 
    "Other cells" : "#D3D3D3", 
}


fig, ax = plt.subplots(figsize=(6,6))

# Layer 1: Other donors (bottom)
ax.scatter(
    umap[mask_other, 0],
    umap[mask_other, 1],
    s=1,
    c=palette_highlight["Other cells"],
    linewidths=0,
    rasterized=True
)


# Layer 2: Pt12 T-B clones (top)
ax.scatter(
    umap[mask_tb, 0],
    umap[mask_tb, 1],
    s=20,
    c=palette_highlight["T-BAM clones"],
    linewidths=0,
    rasterized=True
)

ax.set_axis_off()

plt.tight_layout()

fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/XX_Highlight_TBAM_alldonors.pdf",
    bbox_inches="tight",
    dpi=300
)

plt.show()

In [ ]:
## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["donor"].isin(Patients_with_X_cells)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
Donor_all = obs_df[["L3_ann"]].value_counts().reset_index()
Donor_all["source"] = "All CD8 T cell"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "EBV"]
obs_df = obs_df[obs_df["donor"].isin(Patients_with_X_cells)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
EBV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
EBV_agspf["source"] = "EBV-specific clones"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "CMV"]
obs_df = obs_df[obs_df["donor"].isin(Patients_with_X_cells)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
CMV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
CMV_agspf["source"] = "CMV-specific clones"

## Get the FLU-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "FLU-A"]
obs_df = obs_df[obs_df["donor"].isin(Patients_with_X_cells)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
FLU_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
FLU_agspf["source"] = "FLU-A-specific clones"

# ## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(TBAM_clone_list_all)]
obs_df = obs_df[obs_df["donor"].isin(Patients_with_X_cells)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_all = obs_df[["L3_ann"]].value_counts().reset_index()
TB_all["source"] = "T-MG/BAM clones"

## Get the clonotypes of interest (TBAM_clone_list_agsp)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(TBAM_clone_list_agsp)]
obs_df = obs_df[obs_df["donor"].isin(Patients_with_X_cells)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
TB_agspf["source"] = "Ag-specific T-MG/BAM clones"

## Concat, pivot and transform
count_df = pd.concat([Donor_all, CMV_agspf, FLU_agspf, EBV_agspf, TB_all, TB_agspf])
count_df = count_df.pivot(index = "source", columns = "L3_ann", values="count")
print("Number of cells for each category")
print(count_df.sum(axis=1))

#Transform into percentages
count_df = count_df.div(count_df.sum(axis=1), axis=0)*100


cluster_color_dict = {
    # --- Pastel colors / TN ---
    "0":  "#dbe9f6",  # very light blue
    "1":  "#c7e9c0",  # pastel green
    "2":  "#e2d9f3",  # light lavender
    "11": "#fdd0a2",  # pastel peach

    # --- TCMs ---
    "7":  "#74c476",  # green
    "13": "#31a354",  # dark green
    "21": "#3BFD16",  # neon

    # --- TEM ---
    "4":  "#fdae6b",  # light orange
    "8":  "#fd8d3c",  # orange

    # --- TEMRA ---
    "9":  "#fb6a4a",  # soft red
    "10": "#8B0000",  # dark red

    # --- TE ---
    "6":  "#C00000",  # red
    "12": "#843c39",  # brown red

    # --- Activated ---
    "17": "#f768a1",  # pink
    "19": "#c51b8a",  # 
    "20": "#FF1B76",  # 
    "22": "#d9d9d9",  # 

    # --- Gamma delta ---
    "15": "#737373",  # Dark grey

    # --- CSF blue / purple-blue gradient ---
    "3":  "#6baed6",  # light blue
    "5":  "#3182bd",  # medium blue
    "14": "#9e9ac8",  # light purple
    "16": "#756bb1",  # purple
    "18": "#1f4fbf",  # dark blue

}

# Annotate a dataframe that contains L3_ann too
L3ann_df = cd8data["gex"].obs[["leiden_1.8", "L3_ann"]].drop_duplicates().copy()
L3ann_df["cluster_colors"] = L3ann_df["leiden_1.8"].map(cluster_color_dict)

## Create L3_ann_dict
clusters = L3ann_df["L3_ann"]
colors = L3ann_df["cluster_colors"]
L3_colors = dict(zip(clusters, colors))

#plot distribution changes
count_df = count_df.reset_index()
count_df["source"] = pd.Categorical(count_df["source"], ordered=True, categories = ["All CD8 T cell",  "CMV-specific clones", "FLU-A-specific clones", "T-MG/BAM clones", "EBV-specific clones", "Ag-specific T-MG/BAM clones"])

# Sort and keep source as index
count_df = count_df.sort_values("source").set_index("source")

fig, ax = plt.subplots(figsize=(3,3))

count_df.plot(
    kind="bar",
    stacked=True,
    color=L3_colors,
    edgecolor="white",
    linewidth=0.2,
    width=0.80,
    ax=ax
)

ax.set_ylim(0,100)
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
# ax.set_xticklabels([])

## export the legend once, and remove it again
# plt.legend(title="L3_ann", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.get_legend().remove()

ax.grid(False)

plt.title("Of PB CD8 T cell")
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/97_Phenotype_of_all_clones_outside_T-B_alldonors.pdf")
plt.show()

In [ ]:
## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["donor"].isin(Patients_with_X_cells)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
Donor_all = obs_df[["L3_ann"]].value_counts().reset_index()
Donor_all["source"] = "All CD8 T cell"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "EBV"]
obs_df = obs_df[obs_df["donor"].isin(Patients_with_X_cells)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
EBV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
EBV_agspf["source"] = "EBV-specific clones"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "CMV"]
obs_df = obs_df[obs_df["donor"].isin(Patients_with_X_cells)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
CMV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
CMV_agspf["source"] = "CMV-specific clones"

## Get the FLU-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "FLU-A"]
obs_df = obs_df[obs_df["donor"].isin(Patients_with_X_cells)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
FLU_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
FLU_agspf["source"] = "FLU-A-specific clones"

# ## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(TBAM_clone_list_all)]
obs_df = obs_df[obs_df["donor"].isin(Patients_with_X_cells)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_all = obs_df[["L3_ann"]].value_counts().reset_index()
TB_all["source"] = "T-MG/BAM clones"

## Get the clonotypes of interest (T_B_clone_list_agsp)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(T_B_clone_list_agsp)]
obs_df = obs_df[obs_df["donor"].isin(Patients_with_X_cells)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
TB_agspf["source"] = "Ag-specific T-MG/BAM clones"

## Concat, pivot and transform
count_df = pd.concat([Donor_all, CMV_agspf, FLU_agspf, EBV_agspf, TB_all, TB_agspf])
count_df = count_df.pivot(index = "source", columns = "L3_ann", values="count")
print("Number of cells for each category")
print(count_df.sum(axis=1))

#Transform into percentages
count_df = count_df.div(count_df.sum(axis=1), axis=0)*100


cluster_color_dict = {
    # --- Pastel colors / TN ---
    "0":  "#dbe9f6",  # very light blue
    "1":  "#c7e9c0",  # pastel green
    "2":  "#e2d9f3",  # light lavender
    "11": "#fdd0a2",  # pastel peach

    # --- TCMs ---
    "7":  "#74c476",  # green
    "13": "#31a354",  # dark green
    "21": "#3BFD16",  # neon

    # --- TEM ---
    "4":  "#fdae6b",  # light orange
    "8":  "#fd8d3c",  # orange

    # --- TEMRA ---
    "9":  "#fb6a4a",  # soft red
    "10": "#8B0000",  # dark red

    # --- TE ---
    "6":  "#C00000",  # red
    "12": "#843c39",  # brown red

    # --- Activated ---
    "17": "#f768a1",  # pink
    "19": "#c51b8a",  # 
    "20": "#FF1B76",  # 
    "22": "#d9d9d9",  # 

    # --- Gamma delta ---
    "15": "#737373",  # Dark grey

    # --- CSF blue / purple-blue gradient ---
    "3":  "#6baed6",  # light blue
    "5":  "#3182bd",  # medium blue
    "14": "#9e9ac8",  # light purple
    "16": "#756bb1",  # purple
    "18": "#1f4fbf",  # dark blue

}

# Annotate a dataframe that contains L3_ann too
L3ann_df = cd8data["gex"].obs[["leiden_1.8", "L3_ann"]].drop_duplicates().copy()
L3ann_df["cluster_colors"] = L3ann_df["leiden_1.8"].map(cluster_color_dict)

## Create L3_ann_dict
clusters = L3ann_df["L3_ann"]
colors = L3ann_df["cluster_colors"]
L3_colors = dict(zip(clusters, colors))

#plot distribution changes
count_df = count_df.reset_index()
count_df["source"] = pd.Categorical(count_df["source"], ordered=True, categories = ["All CD8 T cell",  "CMV-specific clones", "FLU-A-specific clones", "T-MG/BAM clones", "EBV-specific clones", "Ag-specific T-MG/BAM clones"])

# Sort and keep source as index
count_df = count_df.sort_values("source").set_index("source")

fig, ax = plt.subplots(figsize=(3,3))

count_df.plot(
    kind="bar",
    stacked=True,
    color=L3_colors,
    edgecolor="white",
    linewidth=0.2,
    width=0.80,
    ax=ax
)

ax.set_ylim(0,100)
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
# ax.set_xticklabels([])

## export the legend once, and remove it again
# plt.legend(title="L3_ann", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.get_legend().remove()

ax.grid(False)

plt.title("Of CSF CD8 T cell")
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/97_Phenotype_of_all_clones_outside_T-B_alldonors_CSF.pdf")
plt.show()

# Pbla - clonotype analysis

## Donor count

In [ ]:
obs = CSF_data["gex"].obs
print(obs[obs["Recluster_ann"].isin(["PBla", "T-PBla"])][["Recluster_ann2", "catpat", "donor"]].sort_values(["Recluster_ann2", "catpat"]).drop_duplicates())
## 3 MS and 1 NIC has T-PBla
## 4 MS and 4 NIC has PBla

Pbla_donor_list = obs[obs["Recluster_ann"].isin(["PBla", "T-PBla"])][["Recluster_ann2", "catpat", "donor"]].value_counts().reset_index()["donor"].drop_duplicates().tolist()
Pbla_donor_list

## Sister cells in PB and CSF

In [ ]:
## Annotate where all T-PBla clones are in patient 20
cd8data["gex"].obs["TPBla_highlight"] = pd.Categorical(np.where((cd8data["gex"].obs["donor"].isin(Pbla_donor_list)) & (cd8data["gex"].obs["clone_handle"].isin(TPB_clone_list_all)), "T-PBla clones", 
                                                    np.where(cd8data["gex"].obs["donor"].isin(Pbla_donor_list), "Other cells", "Other cells")), 
                                                   ordered=True, 
                                                   categories = ["T-PBla clones", 
                                                                "Other cells"])

In [ ]:
obs = cd8data["gex"].obs

# Get UMAP coordinates
umap = cd8data["gex"].obsm["X_umap"]

# Define masks
mask_other = obs["TPBla_highlight"] == "Other cells"
mask_tb = obs["TPBla_highlight"] == "T-PBla clones"

palette_highlight = {
    "T-PBla clones" : "#000000", 
    "Other cells" : "#D3D3D3", 
}


fig, ax = plt.subplots(figsize=(6,6))

# Layer 1: Other donors (bottom)
ax.scatter(
    umap[mask_other, 0],
    umap[mask_other, 1],
    s=1,
    c=palette_highlight["Other cells"],
    linewidths=0,
    rasterized=True
)


# Layer 2: Pt12 T-PBla clones (top)
ax.scatter(
    umap[mask_tb, 0],
    umap[mask_tb, 1],
    s=20,
    c=palette_highlight["T-PBla clones"],
    linewidths=0,
    rasterized=True
)

ax.set_axis_off()

plt.tight_layout()

fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/XX_Highlight_CD20_TPbla_clones_alldonors.pdf",
    bbox_inches="tight",
    dpi=300
)

plt.show()

In [ ]:
## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["donor"].isin(Pbla_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
Donor_all = obs_df[["L3_ann"]].value_counts().reset_index()
Donor_all["source"] = "All CD8 T cell"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "EBV"]
obs_df = obs_df[obs_df["donor"].isin(Pbla_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
EBV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
EBV_agspf["source"] = "EBV-specific clones"

## Get the CMV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "CMV"]
obs_df = obs_df[obs_df["donor"].isin(Pbla_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
CMV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
CMV_agspf["source"] = "CMV-specific clones"

## Get the FLU-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "FLU-A"]
obs_df = obs_df[obs_df["donor"].isin(Pbla_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
FLU_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
FLU_agspf["source"] = "FLU-A-specific clones"

# ## Get the clonotypes of interest 
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(TPB_clone_list_all)]
obs_df = obs_df[obs_df["donor"].isin(Pbla_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_all = obs_df[["L3_ann"]].value_counts().reset_index()
TB_all["source"] = "T-PBla clones"

## Get the clonotypes of interest (T_B_clone_list_agsp)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(TBla_clone_list_agsp)]
obs_df = obs_df[obs_df["donor"].isin(Pbla_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
TB_agspf["source"] = "EBV-specific T-PBla clones"

## Concat, pivot and transform
count_df = pd.concat([Donor_all, CMV_agspf, FLU_agspf, EBV_agspf, TB_all, TB_agspf])
count_df = count_df.pivot(index = "source", columns = "L3_ann", values="count")
print("Number of cells for each category")
print(count_df.sum(axis=1))

#Transform into percentages
count_df = count_df.div(count_df.sum(axis=1), axis=0)*100


cluster_color_dict = {
    # --- Pastel colors / TN ---
    "0":  "#dbe9f6",  # very light blue
    "1":  "#c7e9c0",  # pastel green
    "2":  "#e2d9f3",  # light lavender
    "11": "#fdd0a2",  # pastel peach

    # --- TCMs ---
    "7":  "#74c476",  # green
    "13": "#31a354",  # dark green
    "21": "#3BFD16",  # neon

    # --- TEM ---
    "4":  "#fdae6b",  # light orange
    "8":  "#fd8d3c",  # orange

    # --- TEMRA ---
    "9":  "#fb6a4a",  # soft red
    "10": "#8B0000",  # dark red

    # --- TE ---
    "6":  "#C00000",  # red
    "12": "#843c39",  # brown red

    # --- Activated ---
    "17": "#f768a1",  # pink
    "19": "#c51b8a",  # 
    "20": "#FF1B76",  # 
    "22": "#d9d9d9",  # 

    # --- Gamma delta ---
    "15": "#737373",  # Dark grey

    # --- CSF blue / purple-blue gradient ---
    "3":  "#6baed6",  # light blue
    "5":  "#3182bd",  # medium blue
    "14": "#9e9ac8",  # light purple
    "16": "#756bb1",  # purple
    "18": "#1f4fbf",  # dark blue

}

# Annotate a dataframe that contains L3_ann too
L3ann_df = cd8data["gex"].obs[["leiden_1.8", "L3_ann"]].drop_duplicates().copy()
L3ann_df["cluster_colors"] = L3ann_df["leiden_1.8"].map(cluster_color_dict)

## Create L3_ann_dict
clusters = L3ann_df["L3_ann"]
colors = L3ann_df["cluster_colors"]
L3_colors = dict(zip(clusters, colors))

#plot distribution changes
count_df = count_df.reset_index()
count_df["source"] = pd.Categorical(count_df["source"], ordered=True, categories = ["All CD8 T cell", "FLU-A-specific clones", "CMV-specific clones", "EBV-specific clones", "T-PBla clones",  "EBV-specific T-PBla clones"])

# Sort and keep source as index
count_df = count_df.sort_values("source").set_index("source")

fig, ax = plt.subplots(figsize=(3,3))

count_df.plot(
    kind="bar",
    stacked=True,
    color=L3_colors,
    edgecolor="white",
    linewidth=0.2,
    width=0.80,
    ax=ax
)

ax.set_ylim(0,100)
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
# ax.set_xticklabels([])

## export the legend once, and remove it again
# plt.legend(title="L3_ann", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.get_legend().remove()

ax.grid(False)

plt.title("Of PB CD8 T cell")
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/97_Phenotype_of_all_clones_outside_T-PBla.pdf")
plt.show()

In [ ]:
## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["donor"].isin(Pbla_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
Donor_all = obs_df[["L3_ann"]].value_counts().reset_index()
Donor_all["source"] = "All CD8 T cell"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "EBV"]
obs_df = obs_df[obs_df["donor"].isin(Pbla_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
EBV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
EBV_agspf["source"] = "EBV-specific clones"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "CMV"]
obs_df = obs_df[obs_df["donor"].isin(Pbla_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
CMV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
CMV_agspf["source"] = "CMV-specific clones"

## Get the FLU-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "FLU-A"]
obs_df = obs_df[obs_df["donor"].isin(Pbla_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
FLU_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
FLU_agspf["source"] = "FLU-A-specific clones"

# ## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(TPB_clone_list_all)]
obs_df = obs_df[obs_df["donor"].isin(Pbla_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_all = obs_df[["L3_ann"]].value_counts().reset_index()
TB_all["source"] = "T-PBla clones"

## Get the clonotypes of interest (T_B_clone_list_agsp)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(TBla_clone_list_agsp)]
obs_df = obs_df[obs_df["donor"].isin(Pbla_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
TB_agspf["source"] = "EBV-specific T-PBla clones"

## Concat, pivot and transform
count_df = pd.concat([Donor_all, CMV_agspf, FLU_agspf, EBV_agspf, TB_all, TB_agspf])
count_df = count_df.pivot(index = "source", columns = "L3_ann", values="count")
print("Number of cells for each category")
print(count_df.sum(axis=1))

#Transform into percentages
count_df = count_df.div(count_df.sum(axis=1), axis=0)*100


cluster_color_dict = {
    # --- Pastel colors / TN ---
    "0":  "#dbe9f6",  # very light blue
    "1":  "#c7e9c0",  # pastel green
    "2":  "#e2d9f3",  # light lavender
    "11": "#fdd0a2",  # pastel peach

    # --- TCMs ---
    "7":  "#74c476",  # green
    "13": "#31a354",  # dark green
    "21": "#3BFD16",  # neon

    # --- TEM ---
    "4":  "#fdae6b",  # light orange
    "8":  "#fd8d3c",  # orange

    # --- TEMRA ---
    "9":  "#fb6a4a",  # soft red
    "10": "#8B0000",  # dark red

    # --- TE ---
    "6":  "#C00000",  # red
    "12": "#843c39",  # brown red

    # --- Activated ---
    "17": "#f768a1",  # pink
    "19": "#c51b8a",  # 
    "20": "#FF1B76",  # 
    "22": "#d9d9d9",  # 

    # --- Gamma delta ---
    "15": "#737373",  # Dark grey

    # --- CSF blue / purple-blue gradient ---
    "3":  "#6baed6",  # light blue
    "5":  "#3182bd",  # medium blue
    "14": "#9e9ac8",  # light purple
    "16": "#756bb1",  # purple
    "18": "#1f4fbf",  # dark blue

}

# Annotate a dataframe that contains L3_ann too
L3ann_df = cd8data["gex"].obs[["leiden_1.8", "L3_ann"]].drop_duplicates().copy()
L3ann_df["cluster_colors"] = L3ann_df["leiden_1.8"].map(cluster_color_dict)

## Create L3_ann_dict
clusters = L3ann_df["L3_ann"]
colors = L3ann_df["cluster_colors"]
L3_colors = dict(zip(clusters, colors))

#plot distribution changes
count_df = count_df.reset_index()
count_df["source"] = pd.Categorical(count_df["source"], ordered=True, categories = ["All CD8 T cell", "FLU-A-specific clones", "CMV-specific clones", "EBV-specific clones", "T-PBla clones",  "EBV-specific T-PBla clones"])

# Sort and keep source as index
count_df = count_df.sort_values("source").set_index("source")

fig, ax = plt.subplots(figsize=(3,3))

count_df.plot(
    kind="bar",
    stacked=True,
    color=L3_colors,
    edgecolor="white",
    linewidth=0.2,
    width=0.80,
    ax=ax
)

ax.set_ylim(0,100)
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
# ax.set_xticklabels([])

## export the legend once, and remove it again
# plt.legend(title="L3_ann", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.get_legend().remove()

ax.grid(False)

plt.title("Of CSF CD8 T cell")
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/97_Phenotype_of_all_clones_outside_T-PBla_CSF.pdf")
plt.show()

# MG/BAM - clonotype analysis

## Donor count

In [ ]:
obs = CSF_data["gex"].obs
obs["Recluster_ann"].drop_duplicates().tolist()

In [ ]:
obs = CSF_data["gex"].obs
print(obs[obs["Recluster_ann"].isin(["MG/BAM", "T-MG/BAM"])][["Recluster_ann2", "catpat", "donor"]].sort_values(["Recluster_ann2", "catpat"]).drop_duplicates())
## 5 MS and 5 NIC has T-MG/BAM
## 5 MS and 5 NIC has MG/BAM

BAM_donor_list = obs[obs["Recluster_ann"].isin(["MG/BAM", "T-MG/BAM"])][["Recluster_ann2", "catpat", "donor"]].value_counts().reset_index()["donor"].drop_duplicates().tolist()
BAM_donor_list

exclude = {"Pt4", "Pt5"}
BAM_donor_list = [x for x in BAM_donor_list if x not in exclude]
BAM_donor_list

## Sister cells in PB and CSF

In [ ]:
## Annotate where all T-MG/BAM clones are in patient 20
cd8data["gex"].obs["TPBla_highlight"] = pd.Categorical(np.where((cd8data["gex"].obs["donor"].isin(BAM_donor_list)) & (cd8data["gex"].obs["clone_handle"].isin(TBAM_clone_list_all)), "T-MG/BAM clones", 
                                                    np.where(cd8data["gex"].obs["donor"].isin(BAM_donor_list), "Other cells", "Other cells")), 
                                                   ordered=True, 
                                                   categories = ["T-MG/BAM clones", 
                                                                "Other cells"])

In [ ]:
obs = cd8data["gex"].obs

# Get UMAP coordinates
umap = cd8data["gex"].obsm["X_umap"]

# Define masks
mask_other = obs["TPBla_highlight"] == "Other cells"
mask_tb = obs["TPBla_highlight"] == "T-MG/BAM clones"

palette_highlight = {
    "T-MG/BAM clones" : "#000000", 
    "Other cells" : "#D3D3D3", 
}


fig, ax = plt.subplots(figsize=(6,6))

# Layer 1: Other donors (bottom)
ax.scatter(
    umap[mask_other, 0],
    umap[mask_other, 1],
    s=1,
    c=palette_highlight["Other cells"],
    linewidths=0,
    rasterized=True
)


# Layer 2: Pt12 T-MG/BAM clones (top)
ax.scatter(
    umap[mask_tb, 0],
    umap[mask_tb, 1],
    s=20,
    c=palette_highlight["T-MG/BAM clones"],
    linewidths=0,
    rasterized=True
)

ax.set_axis_off()

plt.tight_layout()

fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/XX_Highlight_CD20_TMGBAM_clones_alldonors.pdf",
    bbox_inches="tight",
    dpi=300
)

plt.show()

In [ ]:
## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["donor"].isin(BAM_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
Donor_all = obs_df[["L3_ann"]].value_counts().reset_index()
Donor_all["source"] = "All CD8 T cell"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "EBV"]
obs_df = obs_df[obs_df["donor"].isin(BAM_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
EBV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
EBV_agspf["source"] = "EBV-specific clones"

## Get the CMV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "CMV"]
obs_df = obs_df[obs_df["donor"].isin(BAM_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
CMV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
CMV_agspf["source"] = "CMV-specific clones"

## Get the FLU-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "FLU-A"]
obs_df = obs_df[obs_df["donor"].isin(BAM_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
FLU_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
FLU_agspf["source"] = "FLU-A-specific clones"

# ## Get the clonotypes of interest 
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(TBAM_clone_list_all)]
obs_df = obs_df[obs_df["donor"].isin(BAM_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_all = obs_df[["L3_ann"]].value_counts().reset_index()
TB_all["source"] = "T-MG/BAM clones"

## Get the clonotypes of interest (T_B_clone_list_agsp)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(TBAM_clone_list_agsp)]
obs_df = obs_df[obs_df["Virus_extended"] == "CMV"]
obs_df = obs_df[obs_df["donor"].isin(BAM_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_agspf1 = obs_df[["L3_ann"]].value_counts().reset_index()
TB_agspf1["source"] = "CMV-specific T-MG/BAM clones"

## Get the clonotypes of interest (T_B_clone_list_agsp)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(TBAM_clone_list_agsp)]
obs_df = obs_df[obs_df["Virus_extended"] == "EBV"]
obs_df = obs_df[obs_df["donor"].isin(BAM_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_agspf2 = obs_df[["L3_ann"]].value_counts().reset_index()
TB_agspf2["source"] = "EBV-specific T-MG/BAM clones"

## Get the clonotypes of interest (T_B_clone_list_agsp)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(TBAM_clone_list_agsp)]
obs_df = obs_df[obs_df["Virus_extended"] == "FLU-A"]
obs_df = obs_df[obs_df["donor"].isin(BAM_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_agspf3 = obs_df[["L3_ann"]].value_counts().reset_index()
TB_agspf3["source"] = "FLU-specific T-MG/BAM clones"

## Concat, pivot and transform
count_df = pd.concat([Donor_all, CMV_agspf, FLU_agspf, EBV_agspf, TB_all, TB_agspf1,TB_agspf2, TB_agspf3])
count_df = count_df.pivot(index = "source", columns = "L3_ann", values="count")
print("Number of cells for each category")
print(count_df.sum(axis=1))

#Transform into percentages
count_df = count_df.div(count_df.sum(axis=1), axis=0)*100


cluster_color_dict = {
    # --- Pastel colors / TN ---
    "0":  "#dbe9f6",  # very light blue
    "1":  "#c7e9c0",  # pastel green
    "2":  "#e2d9f3",  # light lavender
    "11": "#fdd0a2",  # pastel peach

    # --- TCMs ---
    "7":  "#74c476",  # green
    "13": "#31a354",  # dark green
    "21": "#3BFD16",  # neon

    # --- TEM ---
    "4":  "#fdae6b",  # light orange
    "8":  "#fd8d3c",  # orange

    # --- TEMRA ---
    "9":  "#fb6a4a",  # soft red
    "10": "#8B0000",  # dark red

    # --- TE ---
    "6":  "#C00000",  # red
    "12": "#843c39",  # brown red

    # --- Activated ---
    "17": "#f768a1",  # pink
    "19": "#c51b8a",  # 
    "20": "#FF1B76",  # 
    "22": "#d9d9d9",  # 

    # --- Gamma delta ---
    "15": "#737373",  # Dark grey

    # --- CSF blue / purple-blue gradient ---
    "3":  "#6baed6",  # light blue
    "5":  "#3182bd",  # medium blue
    "14": "#9e9ac8",  # light purple
    "16": "#756bb1",  # purple
    "18": "#1f4fbf",  # dark blue

}

# Annotate a dataframe that contains L3_ann too
L3ann_df = cd8data["gex"].obs[["leiden_1.8", "L3_ann"]].drop_duplicates().copy()
L3ann_df["cluster_colors"] = L3ann_df["leiden_1.8"].map(cluster_color_dict)

## Create L3_ann_dict
clusters = L3ann_df["L3_ann"]
colors = L3ann_df["cluster_colors"]
L3_colors = dict(zip(clusters, colors))

#plot distribution changes
count_df = count_df.reset_index()
count_df["source"] = pd.Categorical(count_df["source"], ordered=True, categories = ["All CD8 T cell", "FLU-A-specific clones", 
                                                                                    "CMV-specific clones", "EBV-specific clones", 
                                                                                    "T-MG/BAM clones", 
                                                                                    "FLU-specific T-MG/BAM clones",
                                                                                   "CMV-specific T-MG/BAM clones",
                                                                                   "EBV-specific T-MG/BAM clones"])

# Sort and keep source as index
count_df = count_df.sort_values("source").set_index("source")

fig, ax = plt.subplots(figsize=(3,3))

count_df.plot(
    kind="bar",
    stacked=True,
    color=L3_colors,
    edgecolor="white",
    linewidth=0.2,
    width=0.80,
    ax=ax
)

ax.set_ylim(0,100)
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
# ax.set_xticklabels([])

## export the legend once, and remove it again
# plt.legend(title="L3_ann", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.get_legend().remove()

ax.grid(False)

plt.title("Of PB CD8 T cell")
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/97_Phenotype_of_all_clones_outside_T-BAM_PB.pdf")
plt.show()

In [ ]:
## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["donor"].isin(BAM_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
Donor_all = obs_df[["L3_ann"]].value_counts().reset_index()
Donor_all["source"] = "All CD8 T cell"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "EBV"]
obs_df = obs_df[obs_df["donor"].isin(BAM_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
EBV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
EBV_agspf["source"] = "EBV-specific clones"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "CMV"]
obs_df = obs_df[obs_df["donor"].isin(BAM_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
CMV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
CMV_agspf["source"] = "CMV-specific clones"

## Get the FLU-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "FLU-A"]
obs_df = obs_df[obs_df["donor"].isin(BAM_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
FLU_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
FLU_agspf["source"] = "FLU-A-specific clones"

# ## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(TBAM_clone_list_all)]
obs_df = obs_df[obs_df["donor"].isin(BAM_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_all = obs_df[["L3_ann"]].value_counts().reset_index()
TB_all["source"] = "T-MG/BAM clones"

## Get the clonotypes of interest (T_B_clone_list_agsp)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(TBAM_clone_list_agsp)]
obs_df = obs_df[obs_df["Virus_extended"] == "CMV"]
obs_df = obs_df[obs_df["donor"].isin(BAM_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_agspf1 = obs_df[["L3_ann"]].value_counts().reset_index()
TB_agspf1["source"] = "CMV-specific T-MG/BAM clones"

## Get the clonotypes of interest (T_B_clone_list_agsp)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(TBAM_clone_list_agsp)]
obs_df = obs_df[obs_df["Virus_extended"] == "EBV"]
obs_df = obs_df[obs_df["donor"].isin(BAM_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_agspf2 = obs_df[["L3_ann"]].value_counts().reset_index()
TB_agspf2["source"] = "EBV-specific T-MG/BAM clones"

## Get the clonotypes of interest (T_B_clone_list_agsp)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(TBAM_clone_list_agsp)]
obs_df = obs_df[obs_df["Virus_extended"] == "FLU-A"]
obs_df = obs_df[obs_df["donor"].isin(BAM_donor_list)]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_agspf3 = obs_df[["L3_ann"]].value_counts().reset_index()
TB_agspf3["source"] = "FLU-specific T-MG/BAM clones"

## Concat, pivot and transform
count_df = pd.concat([Donor_all, CMV_agspf, FLU_agspf, EBV_agspf, TB_all, TB_agspf1,TB_agspf2, TB_agspf3])
count_df = count_df.pivot(index = "source", columns = "L3_ann", values="count")
print("Number of cells for each category")
print(count_df.sum(axis=1))

#Transform into percentages
count_df = count_df.div(count_df.sum(axis=1), axis=0)*100


cluster_color_dict = {
    # --- Pastel colors / TN ---
    "0":  "#dbe9f6",  # very light blue
    "1":  "#c7e9c0",  # pastel green
    "2":  "#e2d9f3",  # light lavender
    "11": "#fdd0a2",  # pastel peach

    # --- TCMs ---
    "7":  "#74c476",  # green
    "13": "#31a354",  # dark green
    "21": "#3BFD16",  # neon

    # --- TEM ---
    "4":  "#fdae6b",  # light orange
    "8":  "#fd8d3c",  # orange

    # --- TEMRA ---
    "9":  "#fb6a4a",  # soft red
    "10": "#8B0000",  # dark red

    # --- TE ---
    "6":  "#C00000",  # red
    "12": "#843c39",  # brown red

    # --- Activated ---
    "17": "#f768a1",  # pink
    "19": "#c51b8a",  # 
    "20": "#FF1B76",  # 
    "22": "#d9d9d9",  # 

    # --- Gamma delta ---
    "15": "#737373",  # Dark grey

    # --- CSF blue / purple-blue gradient ---
    "3":  "#6baed6",  # light blue
    "5":  "#3182bd",  # medium blue
    "14": "#9e9ac8",  # light purple
    "16": "#756bb1",  # purple
    "18": "#1f4fbf",  # dark blue

}

# Annotate a dataframe that contains L3_ann too
L3ann_df = cd8data["gex"].obs[["leiden_1.8", "L3_ann"]].drop_duplicates().copy()
L3ann_df["cluster_colors"] = L3ann_df["leiden_1.8"].map(cluster_color_dict)

## Create L3_ann_dict
clusters = L3ann_df["L3_ann"]
colors = L3ann_df["cluster_colors"]
L3_colors = dict(zip(clusters, colors))

#plot distribution changes
count_df = count_df.reset_index()
count_df["source"] = pd.Categorical(count_df["source"], ordered=True, categories = ["All CD8 T cell", "FLU-A-specific clones", 
                                                                                    "CMV-specific clones", "EBV-specific clones", 
                                                                                    "T-MG/BAM clones", 
                                                                                    "FLU-specific T-MG/BAM clones",
                                                                                   "CMV-specific T-MG/BAM clones",
                                                                                   "EBV-specific T-MG/BAM clones"])
# Sort and keep source as index
count_df = count_df.sort_values("source").set_index("source")

fig, ax = plt.subplots(figsize=(3,3))

count_df.plot(
    kind="bar",
    stacked=True,
    color=L3_colors,
    edgecolor="white",
    linewidth=0.2,
    width=0.80,
    ax=ax
)

ax.set_ylim(0,100)
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
# ax.set_xticklabels([])

## export the legend once, and remove it again
# plt.legend(title="L3_ann", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.get_legend().remove()

ax.grid(False)

plt.title("Of CSF CD8 T cell")
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/97_Phenotype_of_all_clones_outside_T-MGBAM_CSF.pdf")
plt.show()

In [ ]:
count_df

# Do we have APC-APC doublets?

In [ ]:
CSF_data["gex"].obs["Recluster_ann2"].value_counts()

In [ ]:
CSF_data["gex"].obs["Recluster_ann2"].drop_duplicates()

In [ ]:
APC_data = CSF_data.copy()
mu.pp.filter_obs(APC_data["gex"], "Recluster_ann2", lambda x: (x.isin(["MG/BAM", "cDC", "Monocyte", "pDC", "PBla", "B cell"])))

#Drop unusedcategories
for col in APC_data["gex"].obs.select_dtypes(include="category").columns:
  APC_data["gex"].obs[col] = APC_data["gex"].obs[col].cat.remove_unused_categories()

### Recluster APCs at higher resolution

In [ ]:
# ---
# Remove genes that are present in only very few cells 
# ---
mu.pp.filter_var(APC_data["gex"], 'n_cells_by_counts', lambda x: x > 10)
APC_data.update()

# ---
# Allign the modalities emphasizing cells in gex.
#   Make certain that the cells in airr and prot modalities are the same cells as are present in gex.
# ---
#Make sure only the indexes recorded in GEX are used in the o*ther modalities too
index_list = APC_data["gex"].obs.index.tolist()

## define a cell index
APC_data["airr"].obs["cell_index"] = APC_data["airr"].obs.index
APC_data["prot"].obs["cell_index"] = APC_data["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
APC_data["airr"].obs["filter_index"] = pd.Categorical(np.where(APC_data["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
APC_data["prot"].obs["filter_index"] = pd.Categorical(np.where(APC_data["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that APC_data["airr"].obs and APC_data["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(APC_data["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(APC_data["prot"], 'filter_index', lambda x: x == "GEX_present")
APC_data.update()

# ---
# When the .var object has been filtered, then you need to renormalize
# ---
APC_data["gex"].X = APC_data["gex"].layers["counts"].copy()
APC_data["gex"].uns.pop("log1p", None) #Removes log1P if it exists
sc.pp.normalize_total(APC_data["gex"], target_sum=1e4, exclude_highly_expressed=False)   #1e4 is important. Do not change
sc.pp.log1p(APC_data["gex"])

In [ ]:
calculate_leiden(APC_data["gex"], resolution=2, n_high_var=3000, remove_tcr_genes=True)   
calculate_umap(APC_data["gex"], n_high_var=3000, remove_tcr_genes=True)

In [ ]:
plt.rcParams["figure.figsize"] = (6, 6)
fig = sc.pl.umap(APC_data["gex"], color=["leiden_2", "cat_multiplet", "final_ann"], legend_fontweight='bold', cmap="RdBu_r", frameon= False , vmin=0, 
                 legend_fontoutline=2, legend_loc = "on data", size=20, return_fig = True, show=False)

# # Rasterize all scatter collections
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/43_APC_reclustered.pdf", 
    bbox_inches="tight",
    dpi=300
)
plt.show()

## 
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

In [ ]:
marker_genes = {
         # "T cell": ["CD3E", "CD4", "CD8A", "CD8B"],
        # "Effector genes" : ["CCL5",  "NKG7", "GZMB", "FASLG", "GNLY", "PRF1", "GZMA", "GZMK", "GZMH", "LAG3", "PDCD1", "TNF", "IFNG", "XCL1"],
        # "Migr. & Adhesion" : ["CXCR6", "CXCL16", "CCR7", "CCL19"],  
        # "NK cell": ["NCAM1", "NCR1", "CD160", "NCR3"],
        "DC": ["FLT3"],
         "cDC1" : ["CLEC9A", "C1orf54", "XCR1"],
         "cDC2" : ["CLEC10A", "CD1C", "FCER1A"],
        "Monocyte" : ["CTSS", "FCN1", "VCAN", "LYZ"],
        "BAM":  ["CD14", "C1QA",  "TREM2",  "APOE",  "MS4A7", "C3", "CX3CR1", "FOLR2", "LYVE1", "F13A1", "STAB1", "MRC1"],
        "MGs" : ["SPP1", "P2RY12",  "TMEM119", "OLFML3", "SPARC"], 
        "B cell": [  "MS4A1", "CD19", "CD79A", "CD27", "CD38", "PRDM1", "MZB1", "IGHG1", "EIF2A", "SDC1", "MKI67"],
        "pDC": ["IL3RA", "PTCRA", "IRF7"],
        "APC-status": ["CIITA", "HLA-DRA", "HLA-DRB1"]}

#     "Microglia": ["APOE", "TREM2", "SORL1","CX3CR1", "SLC2A5",  "SPP1", "P2RY12", "TMEM119" ], # "Microglia": ["APOE", "SORL1", "SLC2A5", "BHLHE41", "SPP1", "P2RY12"], 
#     "Complement\nfactors": ["C1QA", "C1QB", "C1QC",  "CFD", "C3","C2",],
#     "Complement\nregulators": [ "CD55", "CD46", "SERPING1", "CD59", "CFP",  "PROS1"],
#     "Complement\nreceptors": ["ITGB2","C3AR1", "C5AR1","VSIG4", "ITGAM", "ITGAX", "C5AR2", "CD93",  "CR1"],
#     "B cell": [ "CD79A", "CD19", "TNFRSF13B", "IGHG1", "IGHG2", "IGHG3"], #"IGKC", "IGHM", "IGHD", "IGHG1", "IGHG2", "IGHG3"], 
#     "PB/PC": ["MKI67", "MS4A1", "PRDM1", "XBP1", "CD38",  "SLAMF7", "MZB1", ],

# # # Define desired category order
desired_order = ["29", "28", "14", "16", "24", "0", 
                 "21", "22", 
                 "2", "12", "17", "30",
                 "1", "3", "4", "5", "9", "10", "8", "6", "11", "13","19", "20", "25", "27", "31",  "18", 
                "7", "26", "23", "15"]

# # # # # # Reorder categories
APC_data["gex"].obs["leiden_2"] = APC_data["gex"].obs["leiden_2"].cat.reorder_categories(
    desired_order, ordered=True
)

dp = sc.pl.dotplot(
    APC_data["gex"],
    marker_genes,
    # dendrogram=True,
    groupby="leiden_2",
    standard_scale="var",
    cmap="RdBu_r",
    return_fig=True,
    show=False,
)

dp.add_totals()
dp.style(dot_edge_color="black", dot_edge_lw=0.5)

# Build the figure
# dp.make_figure()
dp.show()

# Save
dp.fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/94_CSF_DOUBLET_HighRes_dotplot.pdf",
    bbox_inches="tight",
    dpi=400,
)

In [ ]:
# Extract obs and compute medians
obs = APC_data["gex"].obs
vals = APC_data["gex"].obs["scrublet_score"]
groups = APC_data["gex"].obs["Recluster_ann2"]

# Calculate median per group
medians = obs.groupby("leiden_2")["scrublet_score"].median()

# Sort groups by median
sorted_groups = medians.sort_values().index.tolist()

# Plot with sorted order
with plt.rc_context({"figure.figsize": (6, 3)}):
    ax = sc.pl.violin(
        APC_data["gex"], 
        keys="scrublet_score",
        stripplot=False,
        groupby="leiden_2",
        order=sorted_groups,     # 👈 pass sorted order
        multi_panel=False,
        rotation=45,
        show=False
    )

plt.xticks(rotation=45, ha='right')
# plt.axhline(y=200, linewidth=1.5, color="black", linestyle="--")
plt.ylabel("Doublet score (scrublet)")
plt.ylim(-0.05, 0.85)

# Rasterize violin bodies
for col in ax.collections:
    col.set_rasterized(True)

sb.despine()
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_doublet_score_doublets.pdf", format="pdf", dpi=400)
plt.show()


In [ ]:
# Extract obs and compute medians
obs = APC_data["gex"].obs
vals = APC_data["gex"].obs["n_genes_by_counts"]
groups = APC_data["gex"].obs["n_genes_by_counts"]

# Calculate median per group
medians = obs.groupby("leiden_2")["n_genes_by_counts"].median()

# Sort groups by median
sorted_groups = medians.sort_values().index.tolist()

# Plot with sorted order
with plt.rc_context({"figure.figsize": (6, 3)}):
    ax = sc.pl.violin(
        APC_data["gex"], 
        keys="n_genes_by_counts",
        stripplot=False,
        groupby="leiden_2",
        order=sorted_groups,     # 👈 pass sorted order
        multi_panel=False,
        rotation=45,
        show=False
    )

plt.xticks(rotation=45, ha='right')
# plt.axhline(y=200, linewidth=1.5, color="black", linestyle="--")
plt.ylabel("Number of genes")
# plt.ylim(-0.05, 0.85)

# Rasterize violin bodies
for col in ax.collections:
    col.set_rasterized(True)

sb.despine()
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_doublet_score_doublets.pdf", format="pdf", dpi=400)
plt.show()


In [ ]:
annotation_dict = {
    "29": "cDC1",
    "28": "cDC2",
    "14": "cDC2",
    "16": "cDC2",
    "24": "cDC2",
    "0": "cDC2",
    "21": "cDC2-BAM",
    "22": "Monocyte",
    "2": "Monocyte",
    "12": "Monocyte",
    "17": "Monocyte",
    "30": "Monocyte-BAM",
    "1": "BAM",
    "3": "BAM",
    "4": "BAM",
    "5": "BAM",
    "9": "BAM",
    "10": "BAM",
    "8": "BAM",
    "6": "BAM",
    "11": "BAM",
    "13": "BAM",
    "19": "BAM",
    "20": "BAM",
    "25": "BAM",
    "27": "BAM",
    "31": "BAM",
    "18": "MG",
    "7": "B cell",
    "26": "B cell",
    "23": "PBla",
    "15": "pDC",
}

# annotation_dict = {
#     '0': 'Monocyte',
#     '1': 'cDC2s',
#     '2': 'BAM',
#     '3': 'cDC2s',
#     '4': 'BAM',
#     '5': 'BAM',
#     '6': 'BAM',
#     '7': 'B cell',
#     '8': 'BAM',
#     '9': 'Monocyte',
#     '10': 'BAM',
#     '11': 'BAM',
#     '12': 'BAM',
#     '13': 'pDC',
#     '14': 'BAM',
#     '15': 'MGs',
#     '16': 'BAM',
#     '17': 'Monocyte',
#     '18': 'BAM',
#     '19': 'BAM',
#     '20': 'cDC2s',
#     '21': 'BAM',
#     '22': 'BAM',
#     '23': 'B cell',
#     '24': 'BAM',
#     '25': 'PBla',
#     '26': 'cDC2-BAM',
#     '27': 'BAM',
#     '28': 'BAM',
#     '29': 'cDC2s',
#     '30': 'cDC1s',
# }


# Add cell type column based on annotation
APC_data["gex"].obs['Recluster_ann3'] = [annotation_dict[sample] for sample in APC_data["gex"].obs['leiden_2']]

In [ ]:
plt.rcParams["figure.figsize"] = (6, 6)
fig = sc.pl.umap(APC_data["gex"], color=["leiden_2", "Recluster_ann3", "final_ann"], legend_fontweight='bold', cmap="RdBu_r", frameon= False , vmin=0, 
                 legend_fontoutline=2, legend_loc = "on data", size=30, return_fig = True, show=False)

# # Rasterize all scatter collections
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/43_APC_reclustered.pdf", 
    bbox_inches="tight",
    dpi=300
)
plt.show()

## 
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

In [ ]:
marker_genes = {
        "DC": ["FLT3"],
         "cDC1" : ["CLEC9A", "XCR1"],
         "cDC2" : ["CLEC10A", "CD1C"],
        "Monocyte" : ["FCN1", "VCAN"],
        "BAM":  ["CD14", "C1QA",  "TREM2",  "APOE", "CX3CR1"],
        "MGs" : ["SPP1", "P2RY12",  "TMEM119", "OLFML3"], 
        "B cell": [  "MS4A1", "CD19", "CD79A", "CD27", "CD38", "PRDM1", "MZB1", "IGHG1", "EIF2A", "SDC1", "MKI67"],
        "pDC": ["IL3RA", "PTCRA", "IRF7"]}

# # # # Define desired category order
desired_order = ["cDC1", "cDC2",  "Monocyte", "cDC2-BAM", "Monocyte-BAM", "BAM", "MG", "B cell", "PBla", "pDC"]
# # # # # # Reorder categories
APC_data["gex"].obs["Recluster_ann3"] = (
    APC_data["gex"].obs["Recluster_ann3"].astype("category")
)
APC_data["gex"].obs["Recluster_ann3"] = APC_data["gex"].obs["Recluster_ann3"].cat.reorder_categories(desired_order, ordered=True)

dp = sc.pl.dotplot(
    APC_data["gex"],
    marker_genes,
    # dendrogram=True,
    groupby="Recluster_ann3",
    standard_scale="var",
    cmap="RdBu_r",
    return_fig=True,
    show=False,
)

dp.add_totals()
dp.style(dot_edge_color="black", dot_edge_lw=0.5)

# Build the figure
# dp.make_figure()
dp.show()

# Save
dp.fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/94_CSF_DOUBLET_HighRes_dotplot_consolidated.pdf",
    bbox_inches="tight",
    dpi=400,
)

In [ ]:
# Extract obs and compute medians
obs = APC_data["gex"].obs
vals = APC_data["gex"].obs["scrublet_score"]
groups = APC_data["gex"].obs["Recluster_ann3"]

# Calculate median per group
medians = obs.groupby("Recluster_ann3")["scrublet_score"].median()

# Sort groups by median
sorted_groups = medians.sort_values().index.tolist()

# Plot with sorted order
with plt.rc_context({"figure.figsize": (3, 2)}):
    ax = sc.pl.violin(
        APC_data["gex"], 
        keys="scrublet_score",
        stripplot=False,
        groupby="Recluster_ann3",
        order=sorted_groups,     # 👈 pass sorted order
        multi_panel=False,
        rotation=45,
        show=False
    )

plt.xticks(rotation=45, ha='right')
# plt.axhline(y=200, linewidth=1.5, color="black", linestyle="--")
plt.ylabel("Doublet score (scrublet)")
plt.ylim(-0.05, 0.85)

# Rasterize violin bodies
for col in ax.collections:
    col.set_rasterized(True)

sb.despine()
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_doublet_score_doublets.pdf", format="pdf", dpi=400)
plt.show()


In [ ]:
# Extract obs and compute medians
obs = APC_data["gex"].obs
vals = APC_data["gex"].obs["n_genes_by_counts"]
groups = APC_data["gex"].obs["Recluster_ann3"]

# Calculate median per group
medians = obs.groupby("Recluster_ann3")["n_genes_by_counts"].median()

# Sort groups by median
sorted_groups = medians.sort_values().index.tolist()

# Plot with sorted order
with plt.rc_context({"figure.figsize": (3, 2)}):
    ax = sc.pl.violin(
        APC_data["gex"], 
        keys="n_genes_by_counts",
        stripplot=False,
        groupby="Recluster_ann3",
        order=sorted_groups,     # 👈 pass sorted order
        multi_panel=False,
        rotation=45,
        show=False
    )

plt.xticks(rotation=45, ha='right')
# plt.axhline(y=200, linewidth=1.5, color="black", linestyle="--")
plt.ylabel("No. of genes")
plt.ylim(-100, 9000)
plt.yticks([0, 2000, 4000, 6000, 8000])

# Rasterize violin bodies
for col in ax.collections:
    col.set_rasterized(True)

sb.despine()
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_count_genes_APC_doublets.pdf", format="pdf", dpi=400)
plt.show()


# Number of T-X multiplet

In [ ]:
obs_df = CSF_data["gex"].obs.copy()
obs_df["cat_count"] = np.where(obs_df["cat_multiplet"] == "multiplet", "T_cell_or_multiplet", 
                               np.where(obs_df["Recluster_ann2"].isin(["T cell"]), "T_cell_or_multiplet", "Irrellevant"))
obs_df = obs_df[obs_df["cat_count"] == "T_cell_or_multiplet"]
count_df1 = obs_df[["cat_multiplet"]].value_counts().reset_index()


In [ ]:
plt.figure(figsize=(2, 2))

sb.barplot(
    data=count_df1,
    x="cat_multiplet",
    y="count",
    order=["singlet", "multiplet"]
)

plt.ylabel("Number of cells")
plt.xlabel("")
plt.xticks(rotation=0)
plt.tight_layout()

sb.despine()
plt.grid(False)

plt.show()

In [ ]:
obs_df = APC_data["gex"].obs.copy()
obs_df["cat_multiplet"] = np.where(obs_df["Recluster_ann3"].isin(["cDC2-BAM", "Monocyte-BAM"]), "APC-APC", "Singlet")

count_df2 = obs_df[["cat_multiplet"]].value_counts().reset_index()
print(count_df)
plt.figure(figsize=(2, 2))

sb.barplot(
    data=count_df2,
    x="cat_multiplet",
    y="count",
    order=["Singlet", "APC-APC"]
)

plt.ylabel("Number of cells")
plt.xlabel("")
plt.xticks(rotation=0)
plt.tight_layout()

sb.despine()
plt.grid(False)

plt.show()

In [ ]:
count_df1["source"] = "All cells"
count_df2["source"] = "APCs"
plot_df = pd.concat([count_df1, count_df2])
plot_df["cat_multiplet"] = np.where(plot_df["cat_multiplet"] == "singlet", "T-cell singlet", plot_df["cat_multiplet"])
print(plot_df)

plt.figure(figsize=(2, 2))

sb.barplot(
    data=plot_df,
    x="cat_multiplet",
    y="count",
    order=["T-cell singlet", "multiplet",  "Singlet", "APC-APC"]
)

plt.ylabel("Cell count")
plt.yticks([0, 30000, 60000])
plt.xlabel("")
plt.xticks(rotation=90)
plt.tight_layout()

sb.despine()
plt.grid(False)
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/100_CellCounts_multiplet.pdf", format="pdf", dpi=400)

plt.show()

In [ ]:
12731/323

# Absolute cell counts

In [ ]:
### --------- B cell --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select B cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["PBla"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
plot_df["csf_vol"] = plot_df["donor"].map(csf_ml)

## normalize
plot_df["count_per_ml"] = plot_df["count"] / plot_df["csf_vol"]

## Remove na
plot_df = plot_df.dropna(subset="count_per_ml")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("B cell in CSF (per ml)")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "count_per_ml"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "count_per_ml"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["count_per_ml"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 2.3*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_PlasmacellCounts_pr_ml.pdf")
plt.show()

In [ ]:
### --------- B cell --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select B cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["B cell"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
plot_df["csf_vol"] = plot_df["donor"].map(csf_ml)

## normalize
plot_df["count_per_ml"] = plot_df["count"] / plot_df["csf_vol"]

## Remove na
plot_df = plot_df.dropna(subset="count_per_ml")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("B cell in CSF (per ml)")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "count_per_ml"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "count_per_ml"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["count_per_ml"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 2.3*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_CD20BcellCounts_pr_ml.pdf")
plt.show()

In [ ]:
### --------- B cell --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select B cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["B cell", "PBla"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
plot_df["csf_vol"] = plot_df["donor"].map(csf_ml)

## normalize
plot_df["count_per_ml"] = plot_df["count"] / plot_df["csf_vol"]

## Remove na
plot_df = plot_df.dropna(subset="count_per_ml")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("B cell in CSF (per ml)")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "count_per_ml"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "count_per_ml"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["count_per_ml"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 2.3*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_BcellCounts_pr_ml.pdf")
plt.show()

In [ ]:
### --------- T cell --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select B cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["T cell"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
plot_df["csf_vol"] = plot_df["donor"].map(csf_ml)

## normalize
plot_df["count_per_ml"] = plot_df["count"] / plot_df["csf_vol"]

## Remove na
plot_df = plot_df.dropna(subset="count_per_ml")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("T cell in CSF (per ml)")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "count_per_ml"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "count_per_ml"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["count_per_ml"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 2.3*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_TcellCounts_pr_ml.pdf")
plt.show()

In [ ]:
### --------- T cell --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select B cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["cDC"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
plot_df["csf_vol"] = plot_df["donor"].map(csf_ml)

## normalize
plot_df["count_per_ml"] = plot_df["count"] / plot_df["csf_vol"]

## Remove na
plot_df = plot_df.dropna(subset="count_per_ml")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("DCs in CSF (per ml)")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "count_per_ml"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "count_per_ml"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["count_per_ml"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 2.3*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_DC_Counts_pr_ml.pdf")
plt.show()

In [ ]:
### --------- T cell --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select B cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["pDC"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
plot_df["csf_vol"] = plot_df["donor"].map(csf_ml)

## normalize
plot_df["count_per_ml"] = plot_df["count"] / plot_df["csf_vol"]

## Remove na
plot_df = plot_df.dropna(subset="count_per_ml")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("pDC in CSF (per ml)")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "count_per_ml"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "count_per_ml"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["count_per_ml"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 2.3*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_pDC_Counts_pr_ml.pdf")
plt.show()

In [ ]:
### --------- Monocyte --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select B cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["cDC"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
plot_df["csf_vol"] = plot_df["donor"].map(csf_ml)

## normalize
plot_df["count_per_ml"] = plot_df["count"] / plot_df["csf_vol"]

## Remove na
plot_df = plot_df.dropna(subset="count_per_ml")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("Monocyte in CSF (per ml)")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "count_per_ml"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "count_per_ml"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["count_per_ml"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 2.3*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_Monocyte_Counts_pr_ml.pdf")
plt.show()

In [ ]:
 ### --------- Monocyte --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select B cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["MG/BAM"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
plot_df["csf_vol"] = plot_df["donor"].map(csf_ml)

## normalize
plot_df["count_per_ml"] = plot_df["count"] / plot_df["csf_vol"]

## Remove na
plot_df = plot_df.dropna(subset="count_per_ml")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("MG/BAM in CSF (per ml)")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "count_per_ml"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "count_per_ml"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["count_per_ml"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 2.3*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_MGBAM_Counts_pr_ml.pdf")
plt.show()

In [ ]:
 ### --------- Monocyte --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select B cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["NK cell"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
plot_df["csf_vol"] = plot_df["donor"].map(csf_ml)

## normalize
plot_df["count_per_ml"] = plot_df["count"] / plot_df["csf_vol"]

## Remove na
plot_df = plot_df.dropna(subset="count_per_ml")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("NK cell in CSF (per ml)")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "count_per_ml"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "count_per_ml"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["count_per_ml"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 2.3*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_NK_Counts_pr_ml.pdf")
plt.show()

In [ ]:
 ### --------- Monocyte --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

#Count all cell types
count_df = obs_df[["donor", "catpat", "final_ann"]].value_counts().reset_index()

## --- select B cell singlets
any_b = count_df[count_df["final_ann"].isin(["CD4 T cell"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
plot_df["csf_vol"] = plot_df["donor"].map(csf_ml)

## normalize
plot_df["count_per_ml"] = plot_df["count"] / plot_df["csf_vol"]

## Remove na
plot_df = plot_df.dropna(subset="count_per_ml")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("CD4 T cell in CSF (per ml)")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "count_per_ml"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "count_per_ml"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["count_per_ml"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 2.3*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_CD4T_Counts_pr_ml.pdf")
plt.show()

In [ ]:
 ### --------- Monocyte --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

#Count all cell types
count_df = obs_df[["donor", "catpat", "final_ann"]].value_counts().reset_index()

## --- select B cell singlets
any_b = count_df[count_df["final_ann"].isin(["CD8 T cell"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
plot_df["csf_vol"] = plot_df["donor"].map(csf_ml)

## normalize
plot_df["count_per_ml"] = plot_df["count"] / plot_df["csf_vol"]

## Remove na
plot_df = plot_df.dropna(subset="count_per_ml")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("CD8 T cell in CSF (per ml)")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "count_per_ml"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "count_per_ml"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["count_per_ml"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 2.3*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_CD8T_Counts_pr_ml.pdf")
plt.show()

## T-X multiplet

In [ ]:
 ### --------- T-B cell --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select B cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["T-B cell"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
plot_df["csf_vol"] = plot_df["donor"].map(csf_ml)

## normalize
plot_df["count_per_ml"] = plot_df["count"] / plot_df["csf_vol"]

## Remove na
plot_df = plot_df.dropna(subset="count_per_ml")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

print(plot_df)

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("T-B cell in CSF (per ml)")
ax.set_ylim(-5, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "count_per_ml"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "count_per_ml"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["count_per_ml"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 2.3*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_TB_multiplet_Counts_pr_ml.pdf")
plt.show()

In [ ]:
 ### --------- Monocyte --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select B cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["T-PBla"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
plot_df["csf_vol"] = plot_df["donor"].map(csf_ml)

## normalize
plot_df["count_per_ml"] = plot_df["count"] / plot_df["csf_vol"]

## Remove na
plot_df = plot_df.dropna(subset="count_per_ml")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

print(plot_df)

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("T-plasmablasT cell in CSF (per ml)")
ax.set_ylim(-5, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "count_per_ml"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "count_per_ml"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["count_per_ml"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 2.3*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_TPBL_multiplet_Counts_pr_ml.pdf")
plt.show()

In [ ]:
 ### --------- T-X multiplet --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select B cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["T-B cell", "T-PBla"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
plot_df["csf_vol"] = plot_df["donor"].map(csf_ml)

## normalize
plot_df["count_per_ml"] = plot_df["count"] / plot_df["csf_vol"]

## Remove na
plot_df = plot_df.dropna(subset="count_per_ml")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

print(plot_df)

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("T-B cell in CSF (per ml)")
ax.set_ylim(-5, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "count_per_ml"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "count_per_ml"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["count_per_ml"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 2.3*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_TB_PBL_multiplet_Counts_pr_ml.pdf")
plt.show()

In [ ]:
 ### --------- T-X multiplet --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select B cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["T-MG/BAM"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
plot_df["csf_vol"] = plot_df["donor"].map(csf_ml)

## normalize
plot_df["count_per_ml"] = plot_df["count"] / plot_df["csf_vol"]

## Remove na
plot_df = plot_df.dropna(subset="count_per_ml")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

print(plot_df)

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("T-MG/BAM cells in CSF (per ml)")
ax.set_ylim(-5, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "count_per_ml"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "count_per_ml"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["count_per_ml"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 2.3*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_TMGBAM_multiplet_Counts_pr_ml.pdf")
plt.show()

In [ ]:
 ### --------- T-X multiplet --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select B cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["cDC"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
plot_df["csf_vol"] = plot_df["donor"].map(csf_ml)

## normalize
plot_df["count_per_ml"] = plot_df["count"] / plot_df["csf_vol"]

## Remove na
plot_df = plot_df.dropna(subset="count_per_ml")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

print(plot_df)

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("T-cDC cells in CSF (per ml)")
ax.set_ylim(-5, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "count_per_ml"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "count_per_ml"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["count_per_ml"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 2.3*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_TDCs_multiplet_Counts_pr_ml.pdf")
plt.show()

In [ ]:
 ### --------- T-X multiplet --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select B cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["T-Monocyte"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
plot_df["csf_vol"] = plot_df["donor"].map(csf_ml)

## normalize
plot_df["count_per_ml"] = plot_df["count"] / plot_df["csf_vol"]

## Remove na
plot_df = plot_df.dropna(subset="count_per_ml")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

print(plot_df)

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("T-Monocyte cells in CSF (per ml)")
ax.set_ylim(-5, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "count_per_ml"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "count_per_ml"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["count_per_ml"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 2.3*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_TMono_multiplet_Counts_pr_ml.pdf")
plt.show()

In [ ]:
 ### --------- T-X multiplet --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select B cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["T-NK cell"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
plot_df["csf_vol"] = plot_df["donor"].map(csf_ml)

## normalize
plot_df["count_per_ml"] = plot_df["count"] / plot_df["csf_vol"]

## Remove na
plot_df = plot_df.dropna(subset="count_per_ml")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

print(plot_df)

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("T-NK cell in CSF (per ml)")
ax.set_ylim(-5, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "count_per_ml"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "count_per_ml"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["count_per_ml"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 2.3*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_TNK_multiplet_Counts_pr_ml.pdf")
plt.show()

In [ ]:
 ### --------- T-X multiplet --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select B cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["T-pDC"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
plot_df["csf_vol"] = plot_df["donor"].map(csf_ml)

## normalize
plot_df["count_per_ml"] = plot_df["count"] / plot_df["csf_vol"]

## Remove na
plot_df = plot_df.dropna(subset="count_per_ml")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

print(plot_df)

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("T-pDC cells in CSF (per ml)")
ax.set_ylim(-5, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "count_per_ml"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "count_per_ml"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["count_per_ml"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 2.3*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_T-pDC_multiplet_Counts_pr_ml.pdf")
plt.show()

In [ ]:
 ### --------- T-X multiplet --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select B cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["T-pDC-MG/BAM"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
plot_df["csf_vol"] = plot_df["donor"].map(csf_ml)

## normalize
plot_df["count_per_ml"] = plot_df["count"] / plot_df["csf_vol"]

## Remove na
plot_df = plot_df.dropna(subset="count_per_ml")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

print(plot_df)

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("T-pDC-MG/BAM cells in CSF (per ml)")
ax.set_ylim(-5, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "count_per_ml"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "count_per_ml"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["count_per_ml"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 2.3*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_Tripple_TpDCBAM_multiplet_Counts_pr_ml.pdf")
plt.show()

# Percentage cell counts

In [ ]:
 ### --------- CD8 T cell --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

## Total counts
total_counts = obs_df["donor"].value_counts().reset_index()
total_counts = total_counts.rename(columns={"count" : "total"})

#Count all cell types
count_df = obs_df[["donor", "catpat", "final_ann"]].value_counts().reset_index()

## --- select T cell singlets
any_b = count_df[count_df["final_ann"].isin(["CD8 T cell"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## add total counts
plot_df = plot_df.merge(total_counts, how="left")

## normalize
plot_df["percentage_cell"] = 100*plot_df["count"] / plot_df["total"]

## Remove na
plot_df = plot_df.dropna(subset="percentage_cell")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("% CD8 T cell in CSF")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "percentage_cell"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "percentage_cell"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["percentage_cell"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 4*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_CD8T_pct_csf.pdf")
plt.show()

In [ ]:
 ### --------- CD4 T cell --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

## Total counts
total_counts = obs_df["donor"].value_counts().reset_index()
total_counts = total_counts.rename(columns={"count" : "total"})

#Count all cell types
count_df = obs_df[["donor", "catpat", "final_ann"]].value_counts().reset_index()

## --- select cell singlets
any_b = count_df[count_df["final_ann"].isin(["CD4 T cell"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## add total counts
plot_df = plot_df.merge(total_counts, how="left")

## normalize
plot_df["percentage_cell"] = 100*plot_df["count"] / plot_df["total"]

## Remove na
plot_df = plot_df.dropna(subset="percentage_cell")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("% CD4 T cell in CSF")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "percentage_cell"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "percentage_cell"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["percentage_cell"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 4*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_CD4T_pct_csf.pdf")
plt.show()

In [ ]:
 ### --------- T cell --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

## Total counts
total_counts = obs_df["donor"].value_counts().reset_index()
total_counts = total_counts.rename(columns={"count" : "total"})

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["T cell"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## add total counts
plot_df = plot_df.merge(total_counts, how="left")

## normalize
plot_df["percentage_cell"] = 100*plot_df["count"] / plot_df["total"]

## Remove na
plot_df = plot_df.dropna(subset="percentage_cell")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(2, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("% T cell in CSF")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "percentage_cell"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "percentage_cell"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["percentage_cell"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 4*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_T_pct_csf.pdf")
plt.show()

In [ ]:
 ### --------- B cell --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

## Total counts
total_counts = obs_df["donor"].value_counts().reset_index()
total_counts = total_counts.rename(columns={"count" : "total"})

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["PBla"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## add total counts
plot_df = plot_df.merge(total_counts, how="left")

## normalize
plot_df["percentage_cell"] = 100*plot_df["count"] / plot_df["total"]

## Remove na
plot_df = plot_df.dropna(subset="percentage_cell")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(1.75, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("% PBla B cell in CSF")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "percentage_cell"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "percentage_cell"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["percentage_cell"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 4*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_PBla_csf.pdf")
plt.show()

In [ ]:
 ### --------- B cell --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

## Total counts
total_counts = obs_df["donor"].value_counts().reset_index()
total_counts = total_counts.rename(columns={"count" : "total"})

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["B cell"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## add total counts
plot_df = plot_df.merge(total_counts, how="left")

## normalize
plot_df["percentage_cell"] = 100*plot_df["count"] / plot_df["total"]

## Remove na
plot_df = plot_df.dropna(subset="percentage_cell")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(1.75, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("% B cell in CSF")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "percentage_cell"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "percentage_cell"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["percentage_cell"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 4*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_CD20+B_pct_csf.pdf")
plt.show()

In [ ]:
 ### --------- B cell --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

## Total counts
total_counts = obs_df["donor"].value_counts().reset_index()
total_counts = total_counts.rename(columns={"count" : "total"})

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["B cell", "PBla"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## add total counts
plot_df = plot_df.merge(total_counts, how="left")

## normalize
plot_df["percentage_cell"] = 100*plot_df["count"] / plot_df["total"]

## Remove na
plot_df = plot_df.dropna(subset="percentage_cell")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(1.75, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("% B cell in CSF")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "percentage_cell"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "percentage_cell"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["percentage_cell"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 4*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_B_pct_csf.pdf")
plt.show()

In [ ]:
 ### --------- MG/BAM cells --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

## Total counts
total_counts = obs_df["donor"].value_counts().reset_index()
total_counts = total_counts.rename(columns={"count" : "total"})

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["MG/BAM"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## add total counts
plot_df = plot_df.merge(total_counts, how="left")

## normalize
plot_df["percentage_cell"] = 100*plot_df["count"] / plot_df["total"]

## Remove na
plot_df = plot_df.dropna(subset="percentage_cell")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(1.75, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("% MG/BAM in CSF")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "percentage_cell"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "percentage_cell"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["percentage_cell"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 4*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_MGBAM_pct_csf.pdf")
plt.show()

In [ ]:
 ### --------- Mono cells --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

## Total counts
total_counts = obs_df["donor"].value_counts().reset_index()
total_counts = total_counts.rename(columns={"count" : "total"})

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["Monocyte"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## add total counts
plot_df = plot_df.merge(total_counts, how="left")

## normalize
plot_df["percentage_cell"] = 100*plot_df["count"] / plot_df["total"]

## Remove na
plot_df = plot_df.dropna(subset="percentage_cell")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(1.75, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("% Monocyte in CSF")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "percentage_cell"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "percentage_cell"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["percentage_cell"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 4*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_Monocyte_pct_csf.pdf")
plt.show()

In [ ]:
 ### --------- Mono cells --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

## Total counts
total_counts = obs_df["donor"].value_counts().reset_index()
total_counts = total_counts.rename(columns={"count" : "total"})

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["NK cell"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## add total counts
plot_df = plot_df.merge(total_counts, how="left")

## normalize
plot_df["percentage_cell"] = 100*plot_df["count"] / plot_df["total"]

## Remove na
plot_df = plot_df.dropna(subset="percentage_cell")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(1.75, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("% NK cell in CSF")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "percentage_cell"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "percentage_cell"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["percentage_cell"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 4*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_NK_pct_csf.pdf")
plt.show()

In [ ]:
 ### --------- pDC --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

## Total counts
total_counts = obs_df["donor"].value_counts().reset_index()
total_counts = total_counts.rename(columns={"count" : "total"})

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["pDC"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## add total counts
plot_df = plot_df.merge(total_counts, how="left")

## normalize
plot_df["percentage_cell"] = 100*plot_df["count"] / plot_df["total"]

## Remove na
plot_df = plot_df.dropna(subset="percentage_cell")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(1.75, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("% pDC in CSF")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "percentage_cell"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "percentage_cell"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["percentage_cell"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 4*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_pDC_pct_csf.pdf")
plt.show()

In [ ]:
 ### ---------cDC --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

## Total counts
total_counts = obs_df["donor"].value_counts().reset_index()
total_counts = total_counts.rename(columns={"count" : "total"})

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["cDC"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## add total counts
plot_df = plot_df.merge(total_counts, how="left")

## normalize
plot_df["percentage_cell"] = 100*plot_df["count"] / plot_df["total"]

## Remove na
plot_df = plot_df.dropna(subset="percentage_cell")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(1.75, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("% cDC in CSF")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "percentage_cell"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "percentage_cell"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["percentage_cell"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 4*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_DCs_pct_csf.pdf")
plt.show()

## T-X Multiplet percentage of CSF

In [ ]:
 ### --------- T-X cells --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

## Total counts
total_counts = obs_df["donor"].value_counts().reset_index()
total_counts = total_counts.rename(columns={"count" : "total"})

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["T-B cell", "T-PBla"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## add total counts
plot_df = plot_df.merge(total_counts, how="left")

## normalize
plot_df["percentage_cell"] = 100*plot_df["count"] / plot_df["total"]

## Remove na
plot_df = plot_df.dropna(subset="percentage_cell")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(1.75, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("% T-B cell in CSF")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "percentage_cell"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "percentage_cell"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["percentage_cell"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 4*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_T-Bcells_doublets.pdf")
plt.show()

In [ ]:
 ### --------- T-X cells --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

## Total counts
total_counts = obs_df["donor"].value_counts().reset_index()
total_counts = total_counts.rename(columns={"count" : "total"})

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["T-PBla"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## add total counts
plot_df = plot_df.merge(total_counts, how="left")

## normalize
plot_df["percentage_cell"] = 100*plot_df["count"] / plot_df["total"]

## Remove na
plot_df = plot_df.dropna(subset="percentage_cell")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(1.75, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("% T-PBla in CSF")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "percentage_cell"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "percentage_cell"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["percentage_cell"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 4*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_T-PBla_pct_csf.pdf")
plt.show()

In [ ]:
 ### --------- T-X cells --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

## Total counts
total_counts = obs_df["donor"].value_counts().reset_index()
total_counts = total_counts.rename(columns={"count" : "total"})

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["T-B cell"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## add total counts
plot_df = plot_df.merge(total_counts, how="left")

## normalize
plot_df["percentage_cell"] = 100*plot_df["count"] / plot_df["total"]

## Remove na
plot_df = plot_df.dropna(subset="percentage_cell")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(1.75, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("% T-B cell in CSF")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "percentage_cell"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "percentage_cell"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["percentage_cell"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 4*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_T-B cell_pct_csf.pdf")
plt.show()

In [ ]:
 ### --------- T-X cells --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

## Total counts
total_counts = obs_df["donor"].value_counts().reset_index()
total_counts = total_counts.rename(columns={"count" : "total"})

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["T-NK cell"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## add total counts
plot_df = plot_df.merge(total_counts, how="left")

## normalize
plot_df["percentage_cell"] = 100*plot_df["count"] / plot_df["total"]

## Remove na
plot_df = plot_df.dropna(subset="percentage_cell")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(1.75, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("% T-NK cell in CSF")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "percentage_cell"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "percentage_cell"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["percentage_cell"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 4*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_TNK_cells_pct_csf.pdf")
plt.show()

In [ ]:
 ### --------- T-X cells --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

## Total counts
total_counts = obs_df["donor"].value_counts().reset_index()
total_counts = total_counts.rename(columns={"count" : "total"})

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["T-cDC"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## add total counts
plot_df = plot_df.merge(total_counts, how="left")

## normalize
plot_df["percentage_cell"] = 100*plot_df["count"] / plot_df["total"]

## Remove na
plot_df = plot_df.dropna(subset="percentage_cell")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(1.75, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("% T-cDC in CSF")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "percentage_cell"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "percentage_cell"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["percentage_cell"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 4*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_T-cDC_pct_csf.pdf")
plt.show()

In [ ]:
 ### --------- T-X cells --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

## Total counts
total_counts = obs_df["donor"].value_counts().reset_index()
total_counts = total_counts.rename(columns={"count" : "total"})

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["T-Monocyte"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## add total counts
plot_df = plot_df.merge(total_counts, how="left")

## normalize
plot_df["percentage_cell"] = 100*plot_df["count"] / plot_df["total"]

## Remove na
plot_df = plot_df.dropna(subset="percentage_cell")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(1.75, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("% T-Monocyte in CSF")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "percentage_cell"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "percentage_cell"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["percentage_cell"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 4*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_T-Monocyte_pct_csf.pdf")
plt.show()

In [ ]:
 ### --------- T-X cells --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

## Total counts
total_counts = obs_df["donor"].value_counts().reset_index()
total_counts = total_counts.rename(columns={"count" : "total"})

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["T-MG/BAM"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## add total counts
plot_df = plot_df.merge(total_counts, how="left")

## normalize
plot_df["percentage_cell"] = 100*plot_df["count"] / plot_df["total"]

## Remove na
plot_df = plot_df.dropna(subset="percentage_cell")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(1.75, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("% T-MG/BAM in CSF")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "percentage_cell"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "percentage_cell"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["percentage_cell"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 4*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_T-MGBAM_pct_csf.pdf")
plt.show()

In [ ]:
 ### --------- T-X cells --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

## Total counts
total_counts = obs_df["donor"].value_counts().reset_index()
total_counts = total_counts.rename(columns={"count" : "total"})

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["T-pDC"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## add total counts
plot_df = plot_df.merge(total_counts, how="left")

## normalize
plot_df["percentage_cell"] = 100*plot_df["count"] / plot_df["total"]

## Remove na
plot_df = plot_df.dropna(subset="percentage_cell")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(1.75, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("% T-pDC in CSF")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "percentage_cell"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "percentage_cell"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["percentage_cell"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 4*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_TpDC_pct_csf.pdf")
plt.show()

In [ ]:
 ### --------- T-X cells --------

## Make obs_df
obs_df = CSF_data["gex"].obs.copy()

## Filter
obs_df = obs_df[obs_df["catpat"] != "IC"]

## Total counts
total_counts = obs_df["donor"].value_counts().reset_index()
total_counts = total_counts.rename(columns={"count" : "total"})

#Count all cell types
count_df = obs_df[["donor", "catpat", "Recluster_ann2"]].value_counts().reset_index()

## --- select cell singlets
any_b = count_df[count_df["Recluster_ann2"].isin(["T-pDC-MG/BAM"])]
plot_df = any_b.groupby(["donor"])["count"].sum().reset_index()

## Remake catpat from d_ann
d_ann = count_df[["donor", "catpat"]].drop_duplicates()
plot_df = plot_df.merge(d_ann, how = "left")

## add total counts
plot_df = plot_df.merge(total_counts, how="left")

## normalize
plot_df["percentage_cell"] = 100*plot_df["count"] / plot_df["total"]

## Remove na
plot_df = plot_df.dropna(subset="percentage_cell")

## Drop unused categories
for col in plot_df.select_dtypes(include="category").columns:
 plot_df[col] = plot_df[col].cat.remove_unused_categories()

plt.figure(figsize=(1.75, 2))

ax = sb.barplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    errorbar="se",      # standard error
    capsize=0.2,
    facecolor="none",      # no fill
    edgecolor="black",
    linewidth=0.5,
)

palette = {"NIC": "white", "MS": "black"}

sb.stripplot(
    data=plot_df,
    x="catpat",
    y="percentage_cell",
    order=["NIC", "MS"],
    hue="catpat",
    palette=palette,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("% T-pDC-MG/BAM in CSF")
ax.set_ylim(0, None)

sb.despine()

## add statistics
# Mann-Whitney U test
nic = plot_df.loc[plot_df["catpat"] == "NIC", "percentage_cell"]
ms = plot_df.loc[plot_df["catpat"] == "MS", "percentage_cell"]

stat, p = mannwhitneyu(ms, nic, alternative="two-sided", method="exact")  

# Position for annotation
y = plot_df["percentage_cell"].max()
h = y * 0.08  # height of bracket

# Bracket
ax.plot([0, 0, 1, 1], [y+h, y+2*h, y+2*h, y+h],
        lw=1, c="black")

# P-value
if p < 0.001:
    label = "***"
elif p < 0.01:
    label = "**"
elif p < 0.05:
    label = "*"
else:
    label = f"p = {p:.3f}"

ax.text(
    0.5,
    y + 4*h,
    label,
    ha="center",
    va="bottom",
    fontsize=10,
)

# Increase ylim to make room for annotation
ax.set_ylim(0, y + 3.5*h)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/101_TpDCBAM_pct_csf.pdf")
plt.show()

# Define singlet and multiple frequencies

In [ ]:
CSF_data["gex"].obs["cat_multiplet"] = pd.Categorical(np.where(CSF_data["gex"].obs["Recluster_ann2"].str.contains("T-"), "multiplet", "singlet"))
CSF_data["gex"].obs["cat_multiplet"].value_counts()

In [ ]:
# plt.rcParams["figure.figsize"] = (6, 6)
# fig = sc.pl.umap(CSF_data["gex"], color=["Recluster_ann2", "cat_multiplet"], legend_fontweight='bold', cmap="RdBu_r", frameon= False , vmin=0, legend_fontoutline=2, legend_loc = "on data", size=20, return_fig = True, show=False)

# # # Rasterize all scatter collections
# for ax in fig.axes:
#     for coll in ax.collections:
#         coll.set_rasterized(True)
# plt.show()

# ## 
# plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

In [ ]:
#### Define an annotated obs dataframe with multiplet and singlets as well as their cluster and donor information
obs_df = CSF_data["gex"].obs[["donor", "Recluster_ann2", "cat_multiplet", "chain_pairing"]].copy()

## Exclude Pt4 and Pt5
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt4"])]

### We went away froma defining T-T doublets because we do not have the same sensitivity and specifcity in detecting them as the other cluster-based approach
# ### Define T-cell doublets
# obs_df["Recluster_ann2"] = np.where(
#     (obs_df["Recluster_ann2"] == "T cell") & (obs_df["chain_pairing"] == "two full chains"),
#     "T-T cell",
#     obs_df["Recluster_ann2"]
# )

# ### update cat_multiplet
# obs_df["cat_multiplet"] = np.where(
#     (obs_df["Recluster_ann2"] == "T-T cell"),
#     "multiplet",
#     obs_df["cat_multiplet"]
# )

mapping_dict = {
    "T-cDC": "cDC",
    "T-Monocyte": "Monocyte",
    "T-MG/BAM": "MG/BAM",
    "T-B cell": "B cell",
    "T-PBla":"PBla",
    "T-pDC": "pDC",
    "T-NK cell": "NK cell",
    "T cell" : "T-T cell", 
}

# Add new consensus column
obs_df["consensus_col"] = obs_df["Recluster_ann2"].map(mapping_dict)

# Replace NA with Re_cluster_ann
obs_df["consensus_col"] = np.where(obs_df["consensus_col"].isna(), obs_df["Recluster_ann2"], obs_df["consensus_col"])
obs_df.groupby(["cat_multiplet", "consensus_col"]).size()



In [ ]:
#Group annotation
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]
IC_list = ["Pt5", "Pt4"]

#Match the lists defined above to define "RRMS" and "Control"
obs_df["catpat"] = pd.Categorical(np.where(obs_df["donor"].isin(patient_list), "MS", np.where(obs_df["donor"].isin(control_list), "NIC", np.where(obs_df["donor"].isin(IC_list), "IC", "Error"))))



In [ ]:
## transform consensus col and cat_multiplet into categories
obs_df["donor"] = pd.Categorical(obs_df["donor"])
obs_df["catpat"] = pd.Categorical(obs_df["catpat"])
obs_df["cat_multiplet"] = pd.Categorical(obs_df["cat_multiplet"])
obs_df["consensus_col"] = pd.Categorical(obs_df["consensus_col"])

## Summarize
summarized_df = obs_df[["donor", "catpat", "cat_multiplet", "consensus_col"]]
summarized_df = summarized_df.groupby(["donor", "catpat", "cat_multiplet", "consensus_col"], observed=True).size().reset_index().rename(columns={0:"cell_count"})

#Remove ex-doublets
summarized_df["consensus_col"] = pd.Categorical(summarized_df["consensus_col"])
summarized_df = summarized_df[summarized_df["consensus_col"] != "T-T cell"]
summarized_df = summarized_df[summarized_df["consensus_col"] != "Ex-multiplet"]
summarized_df = summarized_df[summarized_df["consensus_col"] != "T-pDC-MG/BAM"]
summarized_df["consensus_col"] = summarized_df["consensus_col"].cat.remove_unused_categories()

# Summarize further for proportions data
proportion_df = summarized_df.groupby(["cat_multiplet", "consensus_col"], as_index=False)["cell_count"].sum()

# ----------- Plot

# Pivot for stacked barplot
df = proportion_df
pivot_df = df.pivot(index="consensus_col", columns="cat_multiplet", values="cell_count").fillna(0)

# Ensure singlets are on the bottom
pivot_df = pivot_df[["singlet", "multiplet"]]

# Normalize each bar to sum = 1
pivot_df = pivot_df.div(pivot_df.sum(axis=1), axis=0)

# Sort by multiplet proportion (descending)
pivot_df = pivot_df.sort_values("multiplet", ascending=True)

# Define colors (bottom → top)
colors = ["#4C72B0", "#DD8452"]  # singlets = blue, multiplet = orange

# Plot
pivot_df.plot(
    kind="bar",
    stacked=True,
    width=0.8,
    figsize=(6,4),
    edgecolor="black",
    color=colors
)

plt.ylabel("Proprotion of each cell type")
plt.xlabel(" ")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Category", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

plt.savefig(
    "..\\..\\05_plots\\02_Overview_and_doublet_analysis\\39_singlets_vs_multiplet_proportions.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()

In [ ]:
## transform consensus col and cat_multiplet into categories
obs_df["donor"] = pd.Categorical(obs_df["donor"])
obs_df["catpat"] = pd.Categorical(obs_df["catpat"])
obs_df["cat_multiplet"] = pd.Categorical(obs_df["cat_multiplet"])
obs_df["consensus_col"] = pd.Categorical(obs_df["consensus_col"])

## Summarize
summarized_df = obs_df[["donor", "catpat", "cat_multiplet", "consensus_col"]]
summarized_df = summarized_df.groupby(["donor", "catpat", "cat_multiplet", "consensus_col"], observed=True).size().reset_index().rename(columns={0:"cell_count"})

#DO NOT REMOVE ANY CELLS - Because this will change the proportion within multiplet and proportion within singlets 
summarized_df["consensus_col"] = pd.Categorical(summarized_df["consensus_col"])
# summarized_df = summarized_df[summarized_df["consensus_col"] != "T-T cell"]
# summarized_df = summarized_df[summarized_df["consensus_col"] != "Ex-multiplet"]
# summarized_df = summarized_df[summarized_df["consensus_col"] != "T-pDC-MG/BAM"]
summarized_df["consensus_col"] = summarized_df["consensus_col"].cat.remove_unused_categories()

# Make a donor handle
summarized_df["donor_handle"] = summarized_df["donor"].astype(str) + "_" + summarized_df["catpat"].astype(str)

# Reduce the dataframe columns 
summarized_df_reduced = summarized_df[["donor_handle", "cat_multiplet", "consensus_col", "cell_count"]]

# Group only by observed combinations
proportion_df = summarized_df_reduced.groupby(
    ["donor_handle", "cat_multiplet", "consensus_col"],
    as_index=False,
    observed=False
)["cell_count"].sum()

proportion_df.head(10)

In [ ]:
# Example dataframe
df = proportion_df.copy()

# Exclude pt4 and pt4
df = df[~df["donor_handle"].isin(["Pt4_IC", "Pt5_IC"])]

#Group annotation
patient_list = ["Pt11_MS", "Pt12_MS", "Pt14_MS", "Pt17_MS", "Pt20_MS"]
control_list = ["Pt1_NIC", "Pt8_NIC", "Pt18_NIC", "Pt19_NIC", "Pt21_NIC"]
IC_list = ["Pt5", "Pt4"]

#Match the lists defined above to define "RRMS" and "Control"
df["catpat"] = pd.Categorical(np.where(df["donor_handle"].isin(patient_list), "MS", np.where(df["donor_handle"].isin(control_list), "NIC","Error")))

# Normalize to percentages per donor
df["prop"] = df.groupby(["donor_handle", "cat_multiplet"])["cell_count"].transform(lambda x: 100 * (x / x.sum()))

# Facet setup
consensus_cols = df['consensus_col'].unique()
n_cols = 9
n_rows = int(np.ceil(len(consensus_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(2*n_cols, 3*n_rows), sharey=False)
axes = axes.flatten()

# Color and offset mapping
color_map = {"singlet": "#4C72B0", "multiplet": "#DD8452"}
offset = {"singlet": -0.25, "multiplet": 0.25}
marker_map = { "NIC": "o",  "MS": "s"}

# Prepare legend handles (for a common legend)
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker='o', color='w', label='Singlets',
                          markerfacecolor=color_map['singlet'], markersize=8),
                   Line2D([0], [0], marker='o', color='w', label='multiplet',
                          markerfacecolor=color_map['multiplet'], markersize=8)]

for ax, col in zip(axes, consensus_cols):
    sub = df[df['consensus_col'] == col]
    
    # Loop through donors
    donors = sub['donor_handle'].unique()
    for donor in donors:
        donor_sub = sub[sub['donor_handle'] == donor]
        x_vals = [offset[cat] for cat in donor_sub['cat_multiplet']]
        y_vals = donor_sub['prop'].values
        colors = [color_map[cat] for cat in donor_sub['cat_multiplet']]
        
        # Scatter points
        ax.scatter(x_vals, y_vals, c=colors, s=50)
        
        # Connect paired points
        if len(y_vals) == 2:
            ax.plot(x_vals, y_vals, color='gray', alpha=0.5, zorder=0)
    
    # Wilcoxon test
    pval_text = ""
    try:
        wide = sub.pivot(index='donor_handle', columns='cat_multiplet', values='prop')
        if "singlet" in wide.columns and "multiplet" in wide.columns:
            stat, pval = wilcoxon(wide["singlet"], wide["multiplet"])
            pval_text = f"\np = {pval:.2e}"
    except ValueError:
        pass
    
    # Title with p-value
    ax.set_title(f"{col}{pval_text}", fontsize=10)
    
    # Remove x-axis labels, keep ticks (optional)
    ax.set_xticks([])
    ax.set_xlim(-0.5, 0.5)

# Turn off unused axes
for ax in axes[len(consensus_cols):]:
    ax.axis('off')

fig.supylabel("Percentage within donor")
# Add common legend to the right
fig.legend(handles=legend_elements, loc='center right', fontsize=10)
plt.tight_layout(rect=[0,0,0.85,1])  # leave space on the right for legend

plt.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/93_paired_analysis_singlet_vs_multiplet.pdf",
    bbox_inches="tight",
    dpi=300
)

plt.show()


In [ ]:
from matplotlib.lines import Line2D

# Color and marker mapping
catpat_palette = {"NIC": "#000000", "MS": "#999999"}
color_map = {"singlet": "#4C72B0", "multiplet": "#DD8452"}
marker_map = {"NIC": "o", "MS": "s"}

# Prepare legend (color + meaning)
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='NIC',
           markerfacecolor=catpat_palette['NIC'], markersize=8),
    Line2D([0], [0], marker='o', color='w', label='MS',
           markerfacecolor=catpat_palette['MS'], markersize=8),
]

fig, axes = plt.subplots(n_rows, n_cols, figsize=(2*n_cols, 3*n_rows), sharey=False)
axes = axes.flatten()

for ax, col in zip(axes, consensus_cols):
    sub = df[df['consensus_col'] == col]

    for donor in sub['donor_handle'].unique():
        donor_sub = sub[sub['donor_handle'] == donor]

        x_vals = [offset[cat] for cat in donor_sub['cat_multiplet']]
        y_vals = donor_sub['prop'].values

        # one catpat per donor
        group = donor_sub['catpat'].iloc[0]
        colors = [color_map[cat] for cat in donor_sub['cat_multiplet']]
        marker = marker_map[group]

        # scatter points
        ax.scatter(
            x_vals,
            y_vals,
            c=colors,
            marker=marker,
            s=50,
            edgecolor="black",
            linewidth=0.3
        )

        # connect paired points
        if len(y_vals) == 2:
            ax.plot(x_vals, y_vals, color='gray', alpha=0.5, zorder=0)

    # Wilcoxon test
    pval_text = ""
    try:
        wide = sub.pivot(index='donor_handle', columns='cat_multiplet', values='prop')
        if "singlet" in wide.columns and "multiplet" in wide.columns:
            stat, pval = wilcoxon(wide["singlet"], wide["multiplet"])
            pval_text = f"\np = {pval:.2e}"
    except ValueError:
        pass

    ax.set_title(f"{col}{pval_text}", fontsize=10)

    ax.set_xticks([])
    ax.set_xlim(-0.5, 0.5)

# Turn off unused axes
for ax in axes[len(consensus_cols):]:
    ax.axis('off')

fig.supylabel("Percentage within donor")

fig.legend(handles=legend_elements, loc='center right', fontsize=10)
plt.tight_layout(rect=[0, 0, 0.85, 1])
plt.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/93_paired_analysis_singlet_vs_multiplet.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()

# Differential abundance

In [ ]:
# --- Prepare obs_df ---
obs_df = CSF_data["gex"].obs.copy()
obs_df = obs_df[obs_df["Compartment"] == "CSF"]
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt4"])]
obs_df["catpat"] = obs_df["catpat"].cat.remove_unused_categories()
obs_df["donor"] = obs_df["donor"].cat.remove_unused_categories() 

#Greyscale palette
palette = sb.color_palette("Greys", 3)
catpat_palette = dict(zip(["NIC", "IC", "MS"], palette))

# Leiden cluster
cls = "Recluster_ann2"

# Calculate counts per donor and cluster
obs_df = (
    obs_df[["donor", cls, "catpat"]]
    .groupby(["donor", cls])
    .size()
    .reset_index(name="count")
)

# Total per donor
obs_df["total"] = obs_df.groupby("donor")["count"].transform("sum")

# Percentage per donor
obs_df["cluster_pct"] = 100 * obs_df["count"] / obs_df["total"]

# Annotate catpat
patient_list = ["Pt4", "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt5", "Pt8", "Pt18", "Pt19", "Pt21"]
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "Error")
    )
)

#Group annotation
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]
IC_list = ["Pt5", "Pt4"]

#Match the lists defined above to define "RRMS" and "Control"
obs_df["catpat"] = pd.Categorical(np.where(obs_df["donor"].isin(patient_list), "MS", np.where(obs_df["donor"].isin(control_list), "NIC", np.where(obs_df["donor"].isin(IC_list), "IC", "Error"))))

## Make certain that catpat has the right order
obs_df["catpat"] = pd.Categorical(
    obs_df["catpat"],
    categories=["NIC", "MS"],
    ordered=True
)

## Transform for log-scale
obs_df["cluster_pct"] = obs_df["cluster_pct"] + 1e-2

# --- Statistics: Mann–Whitney U per cluster ---
pvals = {}
clusters = obs_df[cls].cat.categories if hasattr(obs_df[cls], "cat") else obs_df[cls].unique()

for cl in clusters:
    sub = obs_df[obs_df[cls] == cl]
    rrms = sub.loc[sub["catpat"] == "MS", "cluster_pct"]
    ctrl = sub.loc[sub["catpat"].isin(["NIC"]), "cluster_pct"]

    if (len(rrms) > 1) and (len(ctrl) > 1):
        _, p = mannwhitneyu(rrms, ctrl, alternative="two-sided", method="exact")
        pvals[cl] = p
    else:
        pvals[cl] = np.nan


# --- Plot ---
plt.figure(figsize=(7, 5))

ax = sb.boxplot(
    data=obs_df,
    x=cls,
    y="cluster_pct",
    hue="catpat",
    dodge=True,
    fill=True,
    showfliers=False,
    linewidth=0.5,
    palette=catpat_palette
)

sb.stripplot(
    data=obs_df,
    x=cls,
    y="cluster_pct",
    hue="catpat",
    dodge=True,
    color="black", 
    jitter=0.2,
    alpha=1,
    size=5,
    linewidth=0.5,
    palette=catpat_palette,
    ax=ax
)

# yscale
plt.yscale("log")

# Fix double legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:3], labels[:3], title="catpat", frameon=False)

ax.set_xlabel("Leiden cluster")
ax.set_ylabel("Percentage of cells")
plt.xticks(rotation=45, ha="right")

# --- Correction or multiple hypothesis testing ---
pvals_df = pd.DataFrame({
    cls: list(pvals.keys()),
    "p_value": list(pvals.values())
})

pvals_df["p_adj"] = multipletests(
    pvals_df["p_value"].fillna(1),
    method="fdr_bh"
)[1]

def p_to_star(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

p_lookup = dict(zip(pvals_df[cls], pvals_df["p_adj"]))

# --- Annotate p-values ---
ymax = obs_df.groupby(cls)["cluster_pct"].max()

for i, cl in enumerate(clusters):
    p = p_lookup.get(cl, np.nan)

    if np.isnan(p):
        continue

    y = ymax[cl] * 1.08

    ax.text(
        i, y,
        p_to_star(p),
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )

    ax.plot(
        [i - 0.2, i + 0.2],
        [y * 0.98, y * 0.98],
        lw=1,
        c="black"
    )

sb.despine()
plt.grid(axis="y")
plt.tight_layout()

# Save
plt.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/47_CellAbundances_CSF.pdf",
    bbox_inches="tight",
    dpi=400,
)
plt.show()

print(pvals_df)


In [ ]:
obs_df[(obs_df["catpat"]== "NIC") & (obs_df["Recluster_ann2"] == "T-B cell")]

In [ ]:
# --- Prepare obs_df ---
obs_df = CSF_data["gex"].obs.copy()
obs_df = obs_df[obs_df["Compartment"] == "CSF"]
obs_df = obs_df[obs_df["cat_multiplet"] == "multiplet"]
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt4"])]
obs_df["catpat"] = obs_df["catpat"].cat.remove_unused_categories()
obs_df["donor"] = obs_df["donor"].cat.remove_unused_categories() 

#Greyscale palette
palette = sb.color_palette("Greys", 3)
catpat_palette = dict(zip(["NIC", "IC", "MS"], palette))

# Leiden cluster
cls = "Recluster_ann2"

# Calculate counts per donor and cluster
obs_df = (
    obs_df[["donor", cls, "catpat"]]
    .groupby(["donor", cls])
    .size()
    .reset_index(name="count")
)

# Total per donor
obs_df["total"] = obs_df.groupby("donor")["count"].transform("sum")

# Percentage per donor
obs_df["cluster_pct"] = 100 * obs_df["count"] / obs_df["total"]

# Annotate catpat
patient_list = ["Pt4", "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt5", "Pt8", "Pt18", "Pt19", "Pt21"]
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "Error")
    )
)

#Group annotation
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]
IC_list = ["Pt5", "Pt4"]

#Match the lists defined above to define "RRMS" and "Control"
obs_df["catpat"] = pd.Categorical(np.where(obs_df["donor"].isin(patient_list), "MS", np.where(obs_df["donor"].isin(control_list), "NIC", np.where(obs_df["donor"].isin(IC_list), "IC", "Error"))))

## Make certain that catpat has the right order
obs_df["catpat"] = pd.Categorical(
    obs_df["catpat"],
    categories=["NIC", "MS"],
    ordered=True
)

## Transform for log-scale
obs_df["cluster_pct"] = obs_df["cluster_pct"] + 1e-2

# --- Statistics: Mann–Whitney U per cluster ---
pvals = {}
clusters = obs_df[cls].cat.categories if hasattr(obs_df[cls], "cat") else obs_df[cls].unique()

for cl in clusters:
    sub = obs_df[obs_df[cls] == cl]
    rrms = sub.loc[sub["catpat"] == "MS", "cluster_pct"]
    ctrl = sub.loc[sub["catpat"].isin(["NIC"]), "cluster_pct"]

    if (len(rrms) > 1) and (len(ctrl) > 1):
        _, p = mannwhitneyu(rrms, ctrl, alternative="two-sided")
        pvals[cl] = p
    else:
        pvals[cl] = np.nan


# --- Plot ---
plt.figure(figsize=(9, 5))

ax = sb.boxplot(
    data=obs_df,
    x=cls,
    y="cluster_pct",
    hue="catpat",
    dodge=True,
    fill=True,
    showfliers=False,
    linewidth=0.5,
    palette=catpat_palette
)

sb.stripplot(
    data=obs_df,
    x=cls,
    y="cluster_pct",
    hue="catpat",
    dodge=True,
    color="black", 
    jitter=0.2,
    alpha=1,
    size=5,
    linewidth=0.5,
    palette=catpat_palette,
    ax=ax
)

# yscale
plt.yscale("log")

# Fix double legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:3], labels[:3], title="catpat", frameon=False)

ax.set_xlabel("Leiden cluster")
ax.set_ylabel("Percentage of cells")
plt.xticks(rotation=45, ha="right")

# --- Correction or multiple hypothesis testing ---
pvals_df = pd.DataFrame({
    cls: list(pvals.keys()),
    "p_value": list(pvals.values())
})

pvals_df["p_adj"] = multipletests(
    pvals_df["p_value"].fillna(1),
    method="fdr_bh"
)[1]

def p_to_star(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

p_lookup = dict(zip(pvals_df[cls], pvals_df["p_adj"]))

# --- Annotate p-values ---
ymax = obs_df.groupby(cls)["cluster_pct"].max()

for i, cl in enumerate(clusters):
    p = p_lookup.get(cl, np.nan)

    if np.isnan(p):
        continue

    y = ymax[cl] * 1.08

    ax.text(
        i, y,
        p_to_star(p),
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )

    ax.plot(
        [i - 0.2, i + 0.2],
        [y * 0.98, y * 0.98],
        lw=1,
        c="black"
    )

sb.despine()
plt.grid(axis="y")
plt.tight_layout()

# Save
plt.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/47_CellAbundances_CSF.pdf",
    bbox_inches="tight",
    dpi=400,
)
plt.show()

print(pvals_df)


# XY plot -DEGs T-B cell, B cell and T cell

### Preprocessing (doublet simulation)
Returns **csf_with_doublets**, the final AnnData object that we will use for differential gene expression

In [ ]:
# -------------------------
# 1. Restrict to selected donors with B cell
# -------------------------
obs_df = CSF_data["gex"].obs.copy()

# Make a donor ist of allowed donors
allowed_donors = obs_df[
    obs_df["Recluster_ann2"].isin(["B cell", "T-B cell"])
]["donor"].unique().tolist()

# filter data based on those donors
csf_dat_subset = CSF_data.copy()
mu.pp.filter_obs(
    csf_dat_subset["gex"],
    "donor",
    lambda x: x.isin(allowed_donors)
)


In [ ]:
# -------------------------
# 2. Donor-aware T+B simulation
# -------------------------
np.random.seed(42)

gex = csf_dat_subset["gex"]
donors = gex.obs["donor"].unique()

sim_doublets_per_donor = []

for donor in donors:
    sub = gex[gex.obs["donor"] == donor]

    APC_cells = sub.obs.query("Recluster_ann2 == 'B cell'").index
    T_cells = sub.obs.query("Recluster_ann2 == 'T cell'").index

    if len(APC_cells) == 0 or len(T_cells) == 0:
        print(f"⚠️ Donor {donor}: Missing T or PBs. Skipping.")
        continue

    n = min(len(APC_cells), len(T_cells))

    APC_sample_idx = sub.obs.loc[APC_cells].sample(n=n, random_state=42).index
    T_sample_idx = sub.obs.loc[T_cells].sample(n=n, random_state=42).index

    APC_subset = sub[APC_sample_idx].copy()
    T_subset = sub[T_sample_idx].copy()

    # Random 1:1 pairing
    T_idx = np.arange(n)
    APC_idx = np.arange(n)
    np.random.shuffle(T_idx)
    np.random.shuffle(APC_idx)
    pairs = list(zip(T_idx, APC_idx))

    # Build simulated matrix
    X_doublets = sparse.vstack([T_subset.X[t] + APC_subset.X[b] for t, b in pairs])
    sim_adata = ad.AnnData(
        X=X_doublets,
        var=T_subset.var.copy()
    )
    sim_adata.obs["Recluster_ann2"] = "Simulated doublets"
    sim_adata.obs["donor"] = donor
    sim_adata.obs.index = [f"{donor}_simTPB_{i}" for i in range(n)]
    sim_doublets_per_donor.append(sim_adata)

if len(sim_doublets_per_donor) == 0:
    raise ValueError("No simulated doublets created.")

sim_all = ad.concat(sim_doublets_per_donor, join="outer", merge="same")

# -------------------------
# 3. Concatenate observed + simulated
# -------------------------
csf_with_doublets = ad.concat(
    {"Observed": gex, "Simulated doublets": sim_all},
    label="source",
    join="outer",
    merge="same"
)

# Normalize + log1p
sc.pp.normalize_total(csf_with_doublets, target_sum=1e4)
sc.pp.log1p(csf_with_doublets)

In [ ]:
## --------------
## Step 3: Generate a umap for simulated vs observed data
## --------------
# sc.pp.pca(csf_with_doublets)
# sc.pp.neighbors(csf_with_doublets)
# sc.tl.umap(csf_with_doublets)

In [ ]:
## --------------
## Step 4: Report on simulated T-B doublets
## --------------
plt.rcParams["figure.figsize"] = (5, 5)
fig = sc.pl.umap(csf_with_doublets, color=["Recluster_ann2", "source"], legend_fontweight='bold', cmap="RdBu_r", frameon= False , vmin=0, legend_fontoutline=2, legend_loc = "on data", size=20, return_fig = True, show=False)

# # Rasterize all scatter collections
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/XX_simulated_TBdoublets_umap1.pdf", 
    bbox_inches="tight",
    dpi=300
)
plt.show()

## 
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

### Make a list of genes expressed in at least 15% of cells (or interacting cells)

In [ ]:
csf_with_doublets.obs["Recluster_ann2"].value_counts()

In [ ]:
## --------------
## Step 1: Generate lists of expressed genes (>X%) for each of the 3 populations of observed cells T cell, B cell and interacting cells
## --------------


## --- Expression in T cell singlets
adata = csf_with_doublets[csf_with_doublets.obs["Recluster_ann2"] == "T cell"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_1 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 



## --- Expression in B cell singlets
adata = csf_with_doublets[csf_with_doublets.obs["Recluster_ann2"] == "B cell"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_2 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 


## --- Expression in T-B cell
adata = csf_with_doublets[csf_with_doublets.obs["Recluster_ann2"] == "T-B cell"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_3 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 

#The total set of genes
gene_list_10pct_expressed = set(gene_list_10pct_1+gene_list_10pct_2+gene_list_10pct_3)
print(len(gene_list_10pct_expressed))

### T vs B cell - differentially expressed genes

In [ ]:
# %%%
# Calculate differentially expressed genes between T cell in CSf and B cell in CSF
# %%% 
sc.tl.rank_genes_groups(csf_with_doublets, groupby="Recluster_ann2", groups=["T cell"], reference="B cell", key_added="T_vs_Bcells")

# %%%% 
# Export all differentially expressed genes
# %%%%
TvBcells_DEG = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Bcells")
TvBcells_DEG.to_csv("../../06_csv_outputs/DEG/3_DoubletAnalysis_DEGs_TandBcells.csv")

# ### This introduces NAs
# sc.tl.filter_rank_genes_groups(
#     csf_with_doublets,
#     min_in_group_fraction=0.2,  #fraction positive in population of interest should be above 0.2.
#     max_out_group_fraction=0.99, #fraction positive !! outside !! the population of interest be less than 0.9
#     min_fold_change=0,      #Since a defaul min_fold_change is enforces, to retain negative logfold changes compare_abs has to be True
#     compare_abs=True,    #Since a defaul min_fold_change is enforces, to retain negative logfold changes compare_abs has to be True
#     key="T_vs_Bcells",
#     key_added="T_vs_Bcells (w. filtering)",
# )

In [ ]:
sc.pl.rank_genes_groups(csf_with_doublets, key="T_vs_Bcells", n_genes=50)

### Simulated vs observed T-B cell - differentially expressed genes

In [ ]:
# %%%
# Calculate differentially expressed genes between random T+B cell vs T-B cell doublets
# %%% 
sc.tl.rank_genes_groups(csf_with_doublets, groupby="Recluster_ann2", groups=["T-B cell"], reference="Simulated doublets", key_added="sampled_vs_T_B_doublets")

# %%%% 
# Export all differentially expressed genes
# %%%%
sampledTBcells_DEG = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="sampled_vs_T_B_doublets")
TvBcells_DEG.to_csv("../../06_csv_outputs/DEG/3_DoubletAnalysis_DEGs_simulated_vs_observed_TC20Bcells.csv")



In [ ]:
sc.pl.rank_genes_groups(csf_with_doublets, key="sampled_vs_T_B_doublets", n_genes=30)

### Make a combined data frame to plot log2FCs 

In [ ]:
### lists to filter on later - these are differentially expressed genes that passes the significance filter
gene_list1 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Bcells")
gene_list1 = gene_list1[gene_list1["names"].notna()]["names"]

gene_list2 = sc.get.rank_genes_groups_df(csf_with_doublets, group = "T-B cell", key ="sampled_vs_T_B_doublets")[["names", "logfoldchanges", "pvals_adj"]]
gene_list2 = gene_list2[gene_list2["names"].notna()]["names"]

combined_list = pd.concat([gene_list1, gene_list2]).drop_duplicates()
print(len(combined_list))
      
### Reduce this gene list to only concern genes that are expressed in at least X pct of T and B cell
combined_list = combined_list[combined_list.isin(gene_list_10pct_expressed)]
print(len(combined_list))

In [ ]:
# %%%%%% 
# Combined the DEGs into 1 plot
# %%%%%

#Subset 1 comes from T cell singlets vs B cell singlets
subset1 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Bcells")[["names", "logfoldchanges", "pvals_adj"]]
subset1 = subset1.rename(columns={"logfoldchanges":"TvB_lfc", "pvals_adj":"pvals_adj_TvB_lfc"})
subset1 = subset1.set_index("names")

#Subset 2 comes from sampled T cell singlet + sampled B cell (all observed B cell) versus T-B cell doublets. 
subset2 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="sampled_vs_T_B_doublets")
subset2 = subset2[["names", "scores", "logfoldchanges", "pvals_adj"]]
subset2 = subset2.set_index("names")

#The dataframes are now joined via indexes
plot_df = subset2.join(subset1).reset_index()

# .. and filtered based on the combined gene list
plot_df = plot_df[plot_df["names"].isin(combined_list)]

# ... and focus on statistical significant differences (cuts the number of dots from 5000+ to around 3000)
plot_df = plot_df[(plot_df["pvals_adj_TvB_lfc"]<0.05) | (plot_df["pvals_adj"]<0.05)]

# .... annotation for staitical significance
plot_df["stat_cat"] = np.where((plot_df["pvals_adj"]<0.05) & (plot_df["pvals_adj_TvB_lfc"]<0.05), "p.adj < 0.05 in both", "Irrellevant")

#Save the data for later heatmap
heatmap_CD20B = plot_df[plot_df["stat_cat"] == "p.adj < 0.05 in both"]

from adjustText import adjust_text

#Palette
color_dic = {
    "p.adj < 0.05 in both" : "#FF0000",
    "Irrellevant" : "#808080",
}

# Scatter plot
plt.figure(figsize=(6, 5))
# Background points
sb.scatterplot(
    data=plot_df[plot_df["stat_cat"] == "Irrellevant"],
    x="TvB_lfc",
    y="logfoldchanges",
    color="#808080",
    s=30,
    alpha=0.25,
    linewidth=0
)

# Significant points on top
sb.scatterplot(
    data=plot_df[plot_df["stat_cat"] == "p.adj < 0.05 in both"],
    x="TvB_lfc",
    y="logfoldchanges",
    color="#FF0000",
    s=30,
    alpha=1,
    linewidth=0
)
# Identify top genes in T-B cell (highest logfoldchanges)
top_y = plot_df.nlargest(30, "logfoldchanges")

# Add reference lines
plt.axhline(y=0, color="black", alpha=0.75, linestyle='--', linewidth=1)
plt.axvline(x=0, color="black", alpha=0.75, linestyle='--', linewidth=1)

# Annotate the top_y genes
texts = []
for _, row in top_y.iterrows():
    texts.append(
        plt.text(row["TvB_lfc"], row["logfoldchanges"], row["names"],
                 fontsize=8, color="black")
    )

# Optional: automatically adjust text to prevent overlap (ggrepel-style)
adjust_text(texts, arrowprops=dict(arrowstyle="-", color="gray", lw=0.5))

# Labels and formatting
plt.ylabel("<-- Simulated T+B cell singlets\nvs\nobserved T-B cell -->")
plt.xlabel("<-- B cell   vs    T cell -->")
plt.title("T and B cell genes\ndifferentially expressed in T-B cell")

sb.despine()
plt.tight_layout()

plt.show()

In [ ]:
from brokenaxes import brokenaxes
from adjustText import adjust_text

# Create figure first
fig = plt.figure(figsize=(6, 5))

# Create broken axes
bax = brokenaxes(
    xlims=((-17, 12), (25, 32)),
    hspace=0.05,
    fig=fig
)

# Scatter plot
colors = plot_df["stat_cat"].map(color_dic)

# Background
sub = plot_df[plot_df["stat_cat"] == "Irrellevant"]
bax.scatter(
    sub["TvB_lfc"],
    sub["logfoldchanges"],
    color="#808080",
    s=30,
    alpha=0.25
)

# Foreground
sub = plot_df[plot_df["stat_cat"] == "p.adj < 0.05 in both"]
bax.scatter(
    sub["TvB_lfc"],
    sub["logfoldchanges"],
    color="#FF0000",
    s=30,
    alpha=1
)

# ---- Identify genes ----
top_y1 = plot_df[(plot_df["stat_cat"] == "p.adj < 0.05 in both") & (plot_df["TvB_lfc"] > 0.5)].nlargest(30, "logfoldchanges")["names"]
top_y2 = plot_df[(plot_df["stat_cat"] == "p.adj < 0.05 in both") & (plot_df["TvB_lfc"] < -0.5)].nlargest(10, "logfoldchanges")["names"]
top_x1 = plot_df[(plot_df["stat_cat"] == "p.adj < 0.05 in both") & (plot_df["TvB_lfc"] > 0.5)].nsmallest(5, "logfoldchanges")["names"]
top_x2 = plot_df[(plot_df["stat_cat"] == "p.adj < 0.05 in both") & (plot_df["TvB_lfc"] < -0.5)].nsmallest(5, "logfoldchanges")["names"]
highlight_genes = set(top_y1).union(top_y2).union(top_x1).union(top_x2)
# highlight_genes = highlight_genes | set(["EOMES", "CD8A", "TCF7", "TOX", "JUNB", "PRF1", "TIGIT", "CD96", "NELL2", "CD28", "CD40LG", "CD4", "FRCL5", "RANK1", "FCRLA", "TNFRSF13B",
#                                         "MS4A1", "CD19", "COBLL1", "GZMK", "ITGAX", "PLXNC1", "PLXNC1", "CDKN1A", "MPZL1"])

highlight_genes = ["CST7", "HLA-DRA", "HLA-DRB1", "CD74", "HLA-DPA1"]
# Filter dataframe ONCE (important fix)
label_df = plot_df[plot_df["names"].isin(highlight_genes)]

# Reference lines
for ax in bax.axs:
    ax.axhline(y=0, color="black", alpha=0.75, linestyle="--", linewidth=1)
    ax.axvline(x=0, color="black", alpha=0.75, linestyle="--", linewidth=1)

# ---- Annotate ----
texts_by_ax = {ax: [] for ax in bax.axs}

for _, row in label_df.iterrows():

    for ax in bax.axs:

        xmin, xmax = ax.get_xlim()
        ymin, ymax = ax.get_ylim()

        if (
            xmin <= row["TvB_lfc"] <= xmax
            and ymin <= row["logfoldchanges"] <= ymax
        ):

            txt = ax.text(
                row["TvB_lfc"],
                row["logfoldchanges"],
                row["names"],
                fontsize=8,
                color="black"
            )

            texts_by_ax[ax].append(txt)
            break
        
# Labels
bax.set_ylabel(
    "<-- Simulated T+B cell singlets\nvs\nobserved T-B cell -->"
)
bax.set_xlabel("<-- CD20+ B   vs    T cell -->")

plt.suptitle(
    "T and B cell genes\ndifferentially expressed in T-B cell",
    y=1.02
)
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_TBcells_doublets_DEGs_simulated.pdf")
plt.show()

In [ ]:
plot_df[plot_df["names"].str.contains("HLA-")]

In [ ]:
TB_data = csf_with_doublets[csf_with_doublets.obs["Recluster_ann2"].isin(["T cell", "B cell", "T-B cell", "Simulated doublets"])].copy()

# Define desired category order
desired_order = ["T cell", "B cell", "Simulated doublets", "T-B cell"]

# Reorder categories
TB_data.obs["Recluster_ann2"] = TB_data.obs["Recluster_ann2"].cat.reorder_categories(
    desired_order, ordered=True
)

markers = {
     "T-cell side": ["GZMA", "CCL5",  "GZMK",   "CST7", "PRF1", "DUSP2", "ID2", "NKG7", "CD8A", "CD8B", "KLRK1",  "CCL4", "AOAH", "GZMH",  "XCL2"],
     # "NS" : ["GZMB", "CD4", "CD40", "CD40LG", "TNFSF4", "TNFRSF4"],
     # "Shared" : ["TYROBP", "ABCB10", "RGS1"], 
     "B-cells side" : ["CD74", "HLA-DRA", "HLA-DRB1", "LRRK1"], 
    # "Downregulated" : ["FLT3LG",  "MAL","MYC", "HLA-A", "HLA-B", "HLA-C"]
}

dp = sc.pl.dotplot(
   TB_data,
   markers,
    groupby="Recluster_ann2",
    standard_scale="var",
    cmap="RdBu_r",
    return_fig=True,
    show=False,
)

dp.add_totals()
dp.style(dot_edge_color="black", dot_edge_lw=0.5)

# Build the figure
# dp.make_figure()
dp.show()

# Save
dp.fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/94_CSF_reclustered2_dotplot.pdf",
    bbox_inches="tight",
    dpi=400,
)

In [ ]:
# Create figure and axis first
fig, ax = plt.subplots(figsize=(7, 2))

dp = sc.pl.matrixplot(
    TB_data,
    markers,
    groupby="Recluster_ann2",
    standard_scale="var",
    cmap="RdBu_r",
    ax=ax,
    return_fig=True,
    show=False,
)

dp.add_totals().style(edge_color='black').show()

fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/94_CSF_reclustered2_matrixplot.pdf",
    bbox_inches="tight",
    dpi=400,
)

# Pathway analysis

## GSEAPY

In [ ]:
## List of available libraries
names = gp.get_library_name()
names

### KEGG

In [ ]:
#Subset 1 comes from T cell singlets vs B cell singlets
subset1 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Bcells")[["names", "logfoldchanges", "pvals_adj"]]
subset1 = subset1.rename(columns={"logfoldchanges":"TvB_lfc", "pvals_adj":"pvals_adj_TvB_lfc"})
subset1 = subset1.set_index("names")

#Subset 2 comes from sampled T cell singlet + sampled B cell (all observed B cell) versus T-B cell doublets. 
subset2 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="sampled_vs_T_B_doublets")
subset2 = subset2[["names", "scores", "logfoldchanges", "pvals_adj"]]
subset2 = subset2.set_index("names")

#The dataframes are now joined via indexes
plot_df = subset2.join(subset1).reset_index()

# .. and filtered based on the combined gene list
plot_df = plot_df[plot_df["names"].isin(combined_list)]

# ... and focus on statistical significant differences (cuts the number of dots from 5000+ to around 3000)
plot_df = plot_df[(plot_df["pvals_adj_TvB_lfc"]<0.05) | (plot_df["pvals_adj"]<0.05)]

# .... annotation for staitical significance
plot_df["stat_cat"] = np.where((plot_df["pvals_adj"]<0.05) & (plot_df["pvals_adj_TvB_lfc"]<0.05), "p.adj < 0.05 in both", "Irrellevant")

## Run enrichment (online)
deg = plot_df[(plot_df["pvals_adj"] < 0.05) & (plot_df["logfoldchanges"] > 0)]["names"].tolist()

## A ranked list
rnk = (
    plot_df[["names", "scores"]]
    .dropna()
    .drop_duplicates("names")
    .sort_values("scores", ascending=False)
)


pre_mult = gp.prerank(
    rnk=rnk,    # single ranked list
    gene_sets="KEGG_2016",
    method='multilevel',            # use fgsea multilevel p-values
    eps=1e-90,                      # smallest p-value resolved; 0 => machine precision eps (float) – Only used when method='multilevel'. Lower boundary for the estimated p-value; p-values smaller than eps are reported as eps. Set to 0 to estimate p-values as small as machine precision allows. Default: 1e-50.
    threads=4,
    min_size=15,                    #Default gene set size max is 15 (gets rid of many disease GOBP gene sets)
    max_size=500,                   #Defailt gene set size max is 500
    sample_size=101,                #sample_size (int) – Only used when method='multilevel'. Sample size for the multilevel split step of the fgsea algorithm; larger values give more accurate (but slower) tail p-value estimates. Default: 101.
    seed=6,
    outdir=None,
)
pre_mult.res2d.head(10)


In [ ]:
df_up = pre_mult.res2d.copy()
df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("NES", ascending=False).head(20)["Term"].tolist()

In [ ]:
from gseapy import gseaplot
terms = 'TNF signaling pathway Homo sapiens hsa04668'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.show()

In [ ]:
from gseapy import gseaplot
terms =  'Toll-like receptor signaling pathway Homo sapiens hsa04620'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.show()

In [ ]:
from gseapy import gseaplot
terms =  'Chemokine signaling pathway Homo sapiens hsa04062'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.show()

In [ ]:
df_dn = pre_mult.res2d.copy()
df_dn[(df_dn["NES"] < 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()

In [ ]:
terms =  'Ribosome Homo sapiens hsa03010'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.show()

In [ ]:
plot_df = pd.concat([df_up, df_dn])

## Filter terms
list_up = df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()
list_dn = df_dn[(df_dn["NES" ]< 0) & (df_dn["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(10)["Term"].tolist()
list_total = list(set(list_up + list_dn))
plot_df = plot_df[plot_df["Term"].isin(list_total)]

## Sort by NES
plot_df = plot_df.sort_values("NES", ascending=False)

## Percentage overlap
plot_df["Overlap %"] = (
    plot_df["Tag %"]
    .str.split("/")
    .apply(lambda x: int(x[0]) / int(x[1]) * 100)
)

# Plot
plt.figure(figsize=(4, 5))

sb.scatterplot(
    data=plot_df,
    x="NES",
    y="Term",
    size="Overlap %",
    hue="FDR q-val",
    palette="RdBu",
    sizes=(50, 400),
    edgecolor="black"
)

plt.xlabel("NES")
plt.ylabel("")
plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    borderaxespad=0
)

plt.tight_layout()
plt.show()

### GOBP

In [ ]:
#Subset 1 comes from T cell singlets vs B cell singlets
subset1 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Bcells")[["names", "logfoldchanges", "pvals_adj"]]
subset1 = subset1.rename(columns={"logfoldchanges":"TvB_lfc", "pvals_adj":"pvals_adj_TvB_lfc"})
subset1 = subset1.set_index("names")

#Subset 2 comes from sampled T cell singlet + sampled B cell (all observed B cell) versus T-B cell doublets. 
subset2 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="sampled_vs_T_B_doublets")
subset2 = subset2[["names", "scores", "logfoldchanges", "pvals_adj"]]
subset2 = subset2.set_index("names")

#The dataframes are now joined via indexes
plot_df = subset2.join(subset1).reset_index()

# .. and filtered based on the combined gene list
plot_df = plot_df[plot_df["names"].isin(combined_list)]

# ... and focus on statistical significant differences (cuts the number of dots from 5000+ to around 3000)
plot_df = plot_df[(plot_df["pvals_adj_TvB_lfc"]<0.05) | (plot_df["pvals_adj"]<0.05)]

# .... annotation for staitical significance
plot_df["stat_cat"] = np.where((plot_df["pvals_adj"]<0.05) & (plot_df["pvals_adj_TvB_lfc"]<0.05), "p.adj < 0.05 in both", "Irrellevant")

## Run enrichment (online)
deg = plot_df[(plot_df["pvals_adj"] < 0.05) & (plot_df["logfoldchanges"] > 0)]["names"].tolist()

## A ranked list
rnk = (
    plot_df[["names", "scores"]]
    .dropna()
    .drop_duplicates("names")
    .sort_values("scores", ascending=False)
)


pre_mult = gp.prerank(
    rnk=rnk,    # single ranked list
    gene_sets="GO_Biological_Process_2026",
    method='multilevel',            # use fgsea multilevel p-values
    eps=1e-90,                      # smallest p-value resolved; 0 => machine precision eps (float) – Only used when method='multilevel'. Lower boundary for the estimated p-value; p-values smaller than eps are reported as eps. Set to 0 to estimate p-values as small as machine precision allows. Default: 1e-50.
    threads=4,
    min_size=15,                    #Default gene set size max is 15 (gets rid of many disease GOBP gene sets)
    max_size=500,                   #Defailt gene set size max is 500
    sample_size=101,                #sample_size (int) – Only used when method='multilevel'. Sample size for the multilevel split step of the fgsea algorithm; larger values give more accurate (but slower) tail p-value estimates. Default: 101.
    seed=6,
    outdir=None,
)
pre_mult.res2d.head(2)


In [ ]:
df_up = pre_mult.res2d.copy()
df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()

In [ ]:
df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(3)["Lead_genes"].tolist()

In [ ]:
from gseapy import gseaplot
terms = 'T Cell Activation (GO:0042110)'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_TcellActivation.pdf", format="pdf", dpi=400)
plt.show()

## PROBLEM: EBI considers this term obsolete

In [ ]:
df_dn = pre_mult.res2d.copy()
df_dn[(df_dn["NES"] < 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()

In [ ]:
terms =  'Cytoplasmic Translation (GO:0002181)'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_Cytoplasmic Translation.pdf", format="pdf", dpi=400)
plt.show()


#### Adding manually annotated gene sets

In [ ]:
plot_df = pd.concat([df_up, df_dn]).drop_duplicates()

## Filter terms
list_up = df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(5)["Term"].tolist()
list_dn = df_dn[(df_dn["NES" ]< 0) & (df_dn["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(5)["Term"].tolist()
## Find terms with antigen presentation, apoptosis, caspases, and GC
list_apop = plot_df[plot_df["Term"].str.contains("Apop") |
    plot_df["Term"].str.contains("Caspase") |
    plot_df["Term"].str.contains("Granzyme")].sort_values("FDR q-val")["Term"].tolist()
list_total = list(set(list_up + list_dn+list_apop))

In [ ]:

plot_df = pd.concat([df_up, df_dn])

## Filter terms
plot_df = plot_df[plot_df["Term"].isin(list_total)]

## Sort by NES
plot_df = plot_df.sort_values("NES", ascending=False)

## Percentage overlap
plot_df["Overlap %"] = (
    plot_df["Tag %"]
    .str.split("/")
    .apply(lambda x: int(x[0]) / int(x[1]) * 100)
)

# Plot
plt.figure(figsize=(4, 6))

sb.scatterplot(
    data=plot_df,
    x="NES",
    y="Term",
    size="Overlap %",
    hue="FDR q-val",
    palette="RdBu",
    sizes=(50, 400),
    edgecolor="black"
)

plt.title("GOBP")
plt.xlabel("NES")
plt.ylabel("")
plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    borderaxespad=0
)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_PATHWAYS_GOBP_TB.pdf", format="pdf", dpi=400)
plt.show()

# XY plot - T-PBla, T cell and plasmablast singlets

### Preprocessing (doublet simulation)
Returns **csf_with_doublets**, the final AnnData object that we will use for differential gene expression

In [ ]:
# -------------------------
# 1. Restrict to selected donors with B cell
# -------------------------
obs_df = CSF_data["gex"].obs.copy()

# Make a donor ist of allowed donors
allowed_donors = obs_df[
    obs_df["Recluster_ann2"].isin(["PBla", "T-PBla"])
]["donor"].unique().tolist()

# filter data based on those donors
csf_dat_subset = CSF_data.copy()
mu.pp.filter_obs(
    csf_dat_subset["gex"],
    "donor",
    lambda x: x.isin(allowed_donors)
)


In [ ]:
# -------------------------
# 2. Donor-aware T+B simulation
# -------------------------
# np.random.seed(41)

gex = csf_dat_subset["gex"]
donors = gex.obs["donor"].unique()

sim_doublets_per_donor = []

for donor in donors:
    sub = gex[gex.obs["donor"] == donor]

    APC_cells = sub.obs.query("Recluster_ann2 == 'PBla'").index
    T_cells = sub.obs.query("Recluster_ann2 == 'T cell'").index

    if len(APC_cells) == 0 or len(T_cells) == 0:
        print(f"⚠️ Donor {donor}: Missing T or PBs. Skipping.")
        continue

    n = min(len(APC_cells), len(T_cells))

    APC_sample_idx = sub.obs.loc[APC_cells].sample(n=n, random_state=39).index
    T_sample_idx = sub.obs.loc[T_cells].sample(n=n, random_state=39).index

    APC_subset = sub[APC_sample_idx].copy()
    T_subset = sub[T_sample_idx].copy()

    # Random 1:1 pairing
    T_idx = np.arange(n)
    APC_idx = np.arange(n)
    np.random.shuffle(T_idx)
    np.random.shuffle(APC_idx)
    pairs = list(zip(T_idx, APC_idx))

    # Build simulated matrix
    X_doublets = sparse.vstack([T_subset.X[t] + APC_subset.X[b] for t, b in pairs])
    sim_adata = ad.AnnData(
        X=X_doublets,
        var=T_subset.var.copy()
    )
    sim_adata.obs["Recluster_ann2"] = "Simulated doublets"
    sim_adata.obs["donor"] = donor
    sim_adata.obs.index = [f"{donor}_simTPB_{i}" for i in range(n)]
    sim_doublets_per_donor.append(sim_adata)

if len(sim_doublets_per_donor) == 0:
    raise ValueError("No simulated doublets created.")

sim_all = ad.concat(sim_doublets_per_donor, join="outer", merge="same")

# -------------------------
# 3. Concatenate observed + simulated
# -------------------------
csf_with_doublets = ad.concat(
    {"Observed": gex, "Simulated doublets": sim_all},
    label="source",
    join="outer",
    merge="same"
)

# Normalize + log1p
sc.pp.normalize_total(csf_with_doublets, target_sum=1e4)
sc.pp.log1p(csf_with_doublets)

In [ ]:
## --------------
## Step 3: Generate a umap for simulated vs observed data
## --------------
# sc.pp.pca(csf_with_doublets)
# sc.pp.neighbors(csf_with_doublets)
# sc.tl.umap(csf_with_doublets)

In [ ]:
# ## --------------
# ## Step 4: Report on simulated T-B doublets
# ## --------------
# plt.rcParams["figure.figsize"] = (5, 5)
# fig = sc.pl.umap(csf_with_doublets, color=["Recluster_ann2", "source"], legend_fontweight='bold', cmap="RdBu_r", frameon= False , vmin=0, legend_fontoutline=2, legend_loc = "on data", size=20, return_fig = True, show=False)

# # # Rasterize all scatter collections
# for ax in fig.axes:
#     for coll in ax.collections:
#         coll.set_rasterized(True)

# fig.savefig(
#     "../../05_plots/02_Overview_and_doublet_analysis/XX_simulated_TBdoublets_umap1.pdf", 
#     bbox_inches="tight",
#     dpi=300
# )
# plt.show()

# ## 
# plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

### Make a list of genes expressed in at least 15% of cells (or interacting cells)

In [ ]:
csf_with_doublets.obs["Recluster_ann2"].value_counts()

In [ ]:
## --------------
## Step 1: Generate lists of expressed genes (>X%) for each of the 3 populations of observed cells T cell, B cell and interacting cells
## --------------


## --- Expression in T cell singlets
adata = csf_with_doublets[csf_with_doublets.obs["Recluster_ann2"] == "T cell"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_1 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 



## --- Expression in B cell singlets
adata = csf_with_doublets[csf_with_doublets.obs["Recluster_ann2"] == "PBla"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_2 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 


## --- Expression in T-PBla
adata = csf_with_doublets[csf_with_doublets.obs["Recluster_ann2"] == "T-PBla"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_3 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 

#The total set of genes
gene_list_10pct_expressed = set(gene_list_10pct_1+gene_list_10pct_2+gene_list_10pct_3)
print(len(gene_list_10pct_expressed))

### T vs Pbla cell - differentially expressed genes

In [ ]:
# %%%
# Calculate differentially expressed genes between T cell in CSf and B cell in CSF
# %%% 
sc.tl.rank_genes_groups(csf_with_doublets, groupby="Recluster_ann2", groups=["T cell"], reference="PBla", key_added="T_vs_Bcells")

# %%%% 
# Export all differentially expressed genes
# %%%%
TvBcells_DEG = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Bcells")
TvBcells_DEG.to_csv("../../06_csv_outputs/DEG/3_DoubletAnalysis_DEGs_TPBcells.csv")

# ### This introduces NAs
# sc.tl.filter_rank_genes_groups(
#     csf_with_doublets,
#     min_in_group_fraction=0.2,  #fraction positive in population of interest should be above 0.2.
#     max_out_group_fraction=0.99, #fraction positive !! outside !! the population of interest be less than 0.9
#     min_fold_change=0,      #Since a defaul min_fold_change is enforces, to retain negative logfold changes compare_abs has to be True
#     compare_abs=True,    #Since a defaul min_fold_change is enforces, to retain negative logfold changes compare_abs has to be True
#     key="T_vs_Bcells",
#     key_added="T_vs_Bcells (w. filtering)",
# )

In [ ]:
sc.pl.rank_genes_groups(csf_with_doublets, key="T_vs_Bcells", n_genes=50)

### Simulated vs observed T-B cell - differentially expressed genes

In [ ]:
# %%%
# Calculate differentially expressed genes between random T+B cell vs T-B cell doublets
# %%% 
sc.tl.rank_genes_groups(csf_with_doublets, groupby="Recluster_ann2", groups=["T-PBla"], reference="Simulated doublets", key_added="sampled_vs_T_B_doublets")

# %%%% 
# Export all differentially expressed genes
# %%%%
sampledTBcells_DEG = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="sampled_vs_T_B_doublets")
TvBcells_DEG.to_csv("../../06_csv_outputs/DEG/3_DoubletAnalysis_DEGs_simulated_vs_observed_TPBcells.csv")

# #Filter it
# ### This introduces NAs
# sc.tl.filter_rank_genes_groups(
#     csf_with_doublets,
#     min_in_group_fraction=0.2,  #fraction positive in population of interest should be above 0.2.
#     max_out_group_fraction=0.99, #fraction positive !! outside !! the population of interest be less than 0.9
#     min_fold_change=0,      #Since a defaul min_fold_change is enforces, to retain negative logfold changes compare_abs has to be True
#     compare_abs=True,    #Since a defaul min_fold_change is enforces, to retain negative logfold changes compare_abs has to be True
#     key="sampled_vs_T_B_doublets",
#     key_added="sampled_vs_T_B_doublets (w. filtering)",
# )


In [ ]:
sc.pl.rank_genes_groups(csf_with_doublets, key="sampled_vs_T_B_doublets", n_genes=30)

### Make a combined data frame to plot log2FCs 

In [ ]:
### lists to filter on later - these are differentially expressed genes that passes the significance filter
gene_list1 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Bcells")
gene_list1 = gene_list1[gene_list1["names"].notna()]["names"]

gene_list2 = sc.get.rank_genes_groups_df(csf_with_doublets, group = "T-PBla", key ="sampled_vs_T_B_doublets")[["names", "logfoldchanges", "pvals_adj"]]
gene_list2 = gene_list2[gene_list2["names"].notna()]["names"]

combined_list = pd.concat([gene_list1, gene_list2]).drop_duplicates()
print(len(combined_list))
      
### Reduce this gene list to only concern genes that are expressed in at least X pct of T and B cell
combined_list = combined_list[combined_list.isin(gene_list_10pct_expressed)]
print(len(combined_list))

In [ ]:
subset1

In [ ]:
# %%%%%% 
# Combined the DEGs into 1 plot
# %%%%%

#Subset 1 comes from T cell singlets vs B cell singlets
subset1 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Bcells")[["names", "logfoldchanges", "pvals_adj"]]
subset1 = subset1.rename(columns={"logfoldchanges":"TvB_lfc", "pvals_adj":"pvals_adj_TvB_lfc"})
subset1 = subset1.set_index("names")

#Subset 2 comes from sampled T cell singlet + sampled B cell (all observed B cell) versus T-B cell doublets. 
subset2 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="sampled_vs_T_B_doublets")
subset2 = subset2[["names", "logfoldchanges", "pvals_adj"]]
subset2 = subset2.set_index("names")

#The dataframes are now joined via indexes
plot_df = subset2.join(subset1).reset_index()

# .. and filtered based on the combined gene list
plot_df = plot_df[plot_df["names"].isin(combined_list)]

# ... and focus on statistical significant differences (cuts the number of dots from 5000+ to around 3000)
plot_df = plot_df[(plot_df["pvals_adj_TvB_lfc"]<0.05) | (plot_df["pvals_adj"]<0.05)]

# .... annotation for staitical significance
plot_df["stat_cat"] = np.where((plot_df["pvals_adj"]<0.05) & (plot_df["pvals_adj_TvB_lfc"]<0.05), "p.adj < 0.05 in both", "Irrellevant")

#Save the data for later heatmap
heatmap_PB = plot_df[plot_df["stat_cat"] == "p.adj < 0.05 in both"]

from adjustText import adjust_text

#Palette
color_dic = {
    "p.adj < 0.05 in both" : "#FF0000",
    "Irrellevant" : "#808080",
}

# Scatter plot
plt.figure(figsize=(6, 5))
# Background points
sb.scatterplot(
    data=plot_df[plot_df["stat_cat"] == "Irrellevant"],
    x="TvB_lfc",
    y="logfoldchanges",
    color="#808080",
    s=30,
    alpha=0.25,
    linewidth=0
)

# Significant points on top
sb.scatterplot(
    data=plot_df[plot_df["stat_cat"] == "p.adj < 0.05 in both"],
    x="TvB_lfc",
    y="logfoldchanges",
    color="#FF0000",
    s=30,
    alpha=1,
    linewidth=0
)
# Identify top genes in T-B cell (highest logfoldchanges)
top_y = plot_df[plot_df["stat_cat"]== "p.adj < 0.05 in both"].nlargest(30, "logfoldchanges")

# Add reference lines
plt.axhline(y=0, color="black", alpha=0.75, linestyle='--', linewidth=1)
plt.axvline(x=0, color="black", alpha=0.75, linestyle='--', linewidth=1)

# Annotate the top_y genes
texts = []
for _, row in top_y.iterrows():
    texts.append(
        plt.text(row["TvB_lfc"], row["logfoldchanges"], row["names"],
                 fontsize=8, color="black")
    )

# Optional: automatically adjust text to prevent overlap (ggrepel-style)
adjust_text(texts, arrowprops=dict(arrowstyle="-", color="gray", lw=0.5))

# Labels and formatting
plt.ylabel("<-- Simulated T+B cell singlets\nvs\nobserved T-PBla -->")
plt.xlabel("<-- PBla   vs    T cell -->")
plt.title("T and B cell genes\ndifferentially expressed in T-B cell")

sb.despine()
plt.tight_layout()

plt.show()

In [ ]:
from brokenaxes import brokenaxes
from adjustText import adjust_text

# Create figure first
fig = plt.figure(figsize=(6, 5))

# Create broken axes
bax = brokenaxes(
    xlims=((-32, -25), (-17, 10)),
    hspace=0.05,
    fig=fig
)

# Scatter plot
colors = plot_df["stat_cat"].map(color_dic)

# Background
sub = plot_df[plot_df["stat_cat"] == "Irrellevant"]
bax.scatter(
    sub["TvB_lfc"],
    sub["logfoldchanges"],
    color="#808080",
    s=30,
    alpha=0.25
)

# Foreground
sub = plot_df[plot_df["stat_cat"] == "p.adj < 0.05 in both"]
bax.scatter(
    sub["TvB_lfc"],
    sub["logfoldchanges"],
    color="#FF0000",
    s=30,
    alpha=1
)

# ---- Identify genes ----
top_y1 = plot_df[(plot_df["stat_cat"] == "p.adj < 0.05 in both") & (plot_df["TvB_lfc"] > 0.5)].nlargest(30, "logfoldchanges")["names"]
top_y2 = plot_df[(plot_df["stat_cat"] == "p.adj < 0.05 in both") & (plot_df["TvB_lfc"] < -0.5)].nlargest(10, "logfoldchanges")["names"]
top_x1 = plot_df[(plot_df["stat_cat"] == "p.adj < 0.05 in both") & (plot_df["TvB_lfc"] > 0.5)].nsmallest(5, "logfoldchanges")["names"]
top_x2 = plot_df[(plot_df["stat_cat"] == "p.adj < 0.05 in both") & (plot_df["TvB_lfc"] < -0.5)].nsmallest(5, "logfoldchanges")["names"]
highlight_genes = set(top_y1).union(top_y2).union(top_x1).union(top_x2)
highlight_genes = highlight_genes | set(["SDC1", "IGHG4", "AC099524.1", "LRRK1", "HLA-DRA", "HLA-DRB1", "CDT1", "GMNN"])
highlight_genes = set(["SDC1", "IGHG4", "AC099524.1", "LRRK1", "HLA-DRA", "HLA-DRB1", "METTL1", "IGHG1", "BLNK"])

# Filter dataframe ONCE (important fix)
label_df = plot_df[plot_df["names"].isin(highlight_genes)]

# Reference lines
for ax in bax.axs:
    ax.axhline(y=0, color="black", alpha=0.75, linestyle="--", linewidth=1)
    ax.axvline(x=0, color="black", alpha=0.75, linestyle="--", linewidth=1)

# ---- Annotate ----
texts_by_ax = {ax: [] for ax in bax.axs}

for _, row in label_df.iterrows():

    for ax in bax.axs:

        xmin, xmax = ax.get_xlim()
        ymin, ymax = ax.get_ylim()

        if (
            xmin <= row["TvB_lfc"] <= xmax
            and ymin <= row["logfoldchanges"] <= ymax
        ):

            txt = ax.text(
                row["TvB_lfc"],
                row["logfoldchanges"],
                row["names"],
                fontsize=8,
                color="black"
            )

            texts_by_ax[ax].append(txt)
            break
        
# Labels
bax.set_ylabel(
    "<-- Simulated T+PB singlets\nvs\nobserved T-PB cell -->"
)
bax.set_xlabel("<-- PB   vs    T cell -->")

plt.suptitle(
    "T and PB cell genes\ndifferentially expressed in T-PB cell",
    y=1.02
)
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_TPBcells_doublets_DEGs_simulated.pdf")
plt.show()

In [ ]:
plot_df[(plot_df["TvB_lfc"]<0)].sort_values("logfoldchanges", ascending=False).head(20)

In [ ]:
plot_df[plot_df["names"] == "BLNK"]

In [ ]:
TB_data = csf_with_doublets[csf_with_doublets.obs["Recluster_ann2"].isin(["T cell", "PBla", "T-PBla", "Simulated doublets"])].copy()

# Define desired category order
desired_order = ["T cell", "PBla", "Simulated doublets", "T-PBla"]

# Reorder categories
TB_data.obs["Recluster_ann2"] = TB_data.obs["Recluster_ann2"].cat.reorder_categories(
    desired_order, ordered=True
)

markers = {
     "T-cell side": ["GZMK", "CST7", "CCL5",   "CD8B", "NKG7",  "CD8A","CCL4", "GZMH","AOAH"],
    # "Other": [ "DERL3", "JCHAIN", "SDC1", "MZB1", "CD38"],
     "B-cells side" : [  "IGHG1", "HLA-DRB1", "LRRK1", "HLA-DRA",  "IGHG4", "AC099524.1", "METTL1"], 
     "Downregulated" : ["CD4", "CD40LG",  "LTB"]
}

# markers = {
#      "T-cell side": ["GZMA", "CCL5",  "GZMK",   "CST7", "PRF1", "DUSP2", "NKG7", "CD8A", "CD8B", "KLRK1",  "CCL4", "AOAH", "GZMH",  "XCL2"],
#      # "NS" : ["GZMB", "CD4", "CD40", "CD40LG", "TNFSF4", "TNFRSF4"],
#      # "Shared" : ["TYROBP", "ABCB10", "RGS1"], 
#      "B-cells side" : ["CD74", "HLA-DRA", "HLA-DRB1", "LRRK1"], 
#     # "Downregulated" : ["FLT3LG",  "MAL","MYC", "HLA-A", "HLA-B", "HLA-C"]
# }


dp = sc.pl.dotplot(
   TB_data,
   markers,
    groupby="Recluster_ann2",
    standard_scale="var",
    cmap="RdBu_r",
    return_fig=True,
    show=False,
)

dp.add_totals()
dp.style(dot_edge_color="black", dot_edge_lw=0.5)

# Build the figure
# dp.make_figure()
dp.show()

# Save
dp.fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/94_CSF_reclustered2_dotplot.pdf",
    bbox_inches="tight",
    dpi=400,
)

In [ ]:
# Create figure and axis first
fig, ax = plt.subplots(figsize=(7, 2))
dp = sc.pl.matrixplot(
    TB_data,
    markers,
    groupby="Recluster_ann2",
    standard_scale="var",
    cmap="RdBu_r",
    ax=ax,
    return_fig=True,
    show=False,
)

dp.add_totals().style(edge_color='black').show()

fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/94_CSF_reclustered2_matrixplot.pdf",
    bbox_inches="tight",
    dpi=400,
)

## Save DEGs

In [ ]:
heatmap_PB.head(10)

# Pathway analysis 

## GSEAPY

In [ ]:
## List of available libraries
names = gp.get_library_name()
names

### KEGG

In [ ]:
#Subset 1 comes from T cell singlets vs B cell singlets
subset1 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Bcells")[["names", "logfoldchanges", "pvals_adj"]]
subset1 = subset1.rename(columns={"logfoldchanges":"TvB_lfc", "pvals_adj":"pvals_adj_TvB_lfc"})
subset1 = subset1.set_index("names")

#Subset 2 comes from sampled T cell singlet + sampled B cell (all observed B cell) versus T-B cell doublets. 
subset2 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="sampled_vs_T_B_doublets")
subset2 = subset2[["names", "scores", "logfoldchanges", "pvals_adj"]]
subset2 = subset2.set_index("names")

#The dataframes are now joined via indexes
plot_df = subset2.join(subset1).reset_index()

# .. and filtered based on the combined gene list
plot_df = plot_df[plot_df["names"].isin(combined_list)]

# ... and focus on statistical significant differences (cuts the number of dots from 5000+ to around 3000)
plot_df = plot_df[(plot_df["pvals_adj_TvB_lfc"]<0.05) | (plot_df["pvals_adj"]<0.05)]

# .... annotation for staitical significance
plot_df["stat_cat"] = np.where((plot_df["pvals_adj"]<0.05) & (plot_df["pvals_adj_TvB_lfc"]<0.05), "p.adj < 0.05 in both", "Irrellevant")

## Run enrichment (online)
deg = plot_df[(plot_df["pvals_adj"] < 0.05) & (plot_df["logfoldchanges"] > 0)]["names"].tolist()

## A ranked list
rnk = (
    plot_df[["names", "scores"]]
    .dropna()
    .drop_duplicates("names")
    .sort_values("scores", ascending=False)
)


pre_mult = gp.prerank(
    rnk=rnk,    # single ranked list
    gene_sets="KEGG_2016",
    method='multilevel',            # use fgsea multilevel p-values
    eps=1e-90,                      # smallest p-value resolved; 0 => machine precision eps (float) – Only used when method='multilevel'. Lower boundary for the estimated p-value; p-values smaller than eps are reported as eps. Set to 0 to estimate p-values as small as machine precision allows. Default: 1e-50.
    threads=4,
    min_size=15,                    #Default gene set size max is 15 (gets rid of many disease GOBP gene sets)
    max_size=500,                   #Defailt gene set size max is 500
    sample_size=101,                #sample_size (int) – Only used when method='multilevel'. Sample size for the multilevel split step of the fgsea algorithm; larger values give more accurate (but slower) tail p-value estimates. Default: 101.
    seed=6,
    outdir=None,
)
pre_mult.res2d.head(10)


In [ ]:
df_up = pre_mult.res2d.copy()
df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()

In [ ]:
from gseapy import gseaplot
terms = 'Protein processing in endoplasmic reticulum Homo sapiens hsa04141'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.show()

In [ ]:
from gseapy import gseaplot
terms =  'Proteasome Homo sapiens hsa03050'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.show()

In [ ]:
from gseapy import gseaplot
terms =  'Citrate cycle (TCA cycle) Homo sapiens hsa00020'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.show()

In [ ]:
from gseapy import gseaplot
terms =  'Oxidative phosphorylation Homo sapiens hsa00190'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.show()

In [ ]:
from gseapy import gseaplot
terms =  'Antigen processing and presentation Homo sapiens hsa04612'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.show()

In [ ]:
df_dn = pre_mult.res2d.copy()
df_dn[(df_dn["NES"] < 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()

In [ ]:
terms =  'Ribosome Homo sapiens hsa03010'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.show()

In [ ]:
terms =  'T cell receptor signaling pathway Homo sapiens hsa04660'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.show()

In [ ]:
plot_df = pd.concat([df_up, df_dn])

## Filter terms
list_up = df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()
list_dn = df_dn[(df_dn["NES" ]< 0) & (df_dn["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(10)["Term"].tolist()
list_total = list(set(list_up + list_dn))
plot_df = plot_df[plot_df["Term"].isin(list_total)]

## Sort by NES
plot_df = plot_df.sort_values("NES", ascending=False)

## Percentage overlap
plot_df["Overlap %"] = (
    plot_df["Tag %"]
    .str.split("/")
    .apply(lambda x: int(x[0]) / int(x[1]) * 100)
)

# Plot
plt.figure(figsize=(10, 8))

sb.scatterplot(
    data=plot_df,
    x="NES",
    y="Term",
    size="Overlap %",
    hue="FDR q-val",
    palette="RdBu",
    sizes=(50, 400),
    edgecolor="black"
)

plt.xlabel("NES")
plt.ylabel("")
plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    borderaxespad=0
)

plt.tight_layout()
plt.show()

### GOBP

In [ ]:
#Subset 1 comes from T cell singlets vs B cell singlets
subset1 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Bcells")[["names", "logfoldchanges", "pvals_adj"]]
subset1 = subset1.rename(columns={"logfoldchanges":"TvB_lfc", "pvals_adj":"pvals_adj_TvB_lfc"})
subset1 = subset1.set_index("names")

#Subset 2 comes from sampled T cell singlet + sampled B cell (all observed B cell) versus T-B cell doublets. 
subset2 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="sampled_vs_T_B_doublets")
subset2 = subset2[["names", "scores", "logfoldchanges", "pvals_adj"]]
subset2 = subset2.set_index("names")

#The dataframes are now joined via indexes
plot_df = subset2.join(subset1).reset_index()

# .. and filtered based on the combined gene list
plot_df = plot_df[plot_df["names"].isin(combined_list)]

# ... and focus on statistical significant differences (cuts the number of dots from 5000+ to around 3000)
plot_df = plot_df[(plot_df["pvals_adj_TvB_lfc"]<0.05) | (plot_df["pvals_adj"]<0.05)]

# .... annotation for staitical significance
plot_df["stat_cat"] = np.where((plot_df["pvals_adj"]<0.05) & (plot_df["pvals_adj_TvB_lfc"]<0.05), "p.adj < 0.05 in both", "Irrellevant")

## Run enrichment (online)
deg = plot_df[(plot_df["pvals_adj"] < 0.05) & (plot_df["logfoldchanges"] > 0)]["names"].tolist()

## A ranked list
rnk = (
    plot_df[["names", "scores"]]
    .dropna()
    .drop_duplicates("names")
    .sort_values("scores", ascending=False)
)


pre_mult = gp.prerank(
    rnk=rnk,    # single ranked list
    gene_sets="GO_Biological_Process_2026",
    method='multilevel',            # use fgsea multilevel p-values
    eps=1e-90,                      # smallest p-value resolved; 0 => machine precision eps (float) – Only used when method='multilevel'. Lower boundary for the estimated p-value; p-values smaller than eps are reported as eps. Set to 0 to estimate p-values as small as machine precision allows. Default: 1e-50.
    threads=4,
    min_size=15,                    #Default gene set size max is 15 (gets rid of many disease GOBP gene sets)
    max_size=500,                   #Defailt gene set size max is 500
    sample_size=101,                #sample_size (int) – Only used when method='multilevel'. Sample size for the multilevel split step of the fgsea algorithm; larger values give more accurate (but slower) tail p-value estimates. Default: 101.
    seed=6,
    outdir=None,
)
pre_mult.res2d.head(10)


In [ ]:
df_up = pre_mult.res2d.copy()
df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()

In [ ]:
from gseapy import gseaplot
terms = 'Protein Exit From Endoplasmic Reticulum (GO:0032527)'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_ProteinExitER_GSEA.pdf", format="pdf", dpi=400)
plt.show()

## PROBLEM: EBI considers this term obsolete

In [ ]:
from gseapy import gseaplot
terms = 'Endoplasmic Reticulum Unfolded Protein Response (GO:0030968)'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_unfolded_protein_response.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
terms =  'Response to Type I Interferon (GO:0034340)'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_ISG_Type1.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
terms =  'Oxidative Phosphorylation (GO:0006119)'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_OXPHOS.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
terms =  'Glycoprotein Biosynthetic Process (GO:0009101)'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_glycoprotein_biosynthesis.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
terms =  'Regulation of Proteasomal Protein Catabolic Process (GO:0061136)'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.show()

In [ ]:
terms =  'Antigen Processing and Presentation of Peptide Antigen via MHC Class II (GO:0002495)'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_AntigenpresentationMHCIIs.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
df_dn = pre_mult.res2d.copy()
df_dn[(df_dn["NES"] < 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()

In [ ]:
terms =  'Cytoplasmic Translation (GO:0002181)'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.show()

In [ ]:
plot_df = pd.concat([df_up, df_dn])

## Filter terms
list_up = df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()
list_dn = df_dn[(df_dn["NES" ]< 0) & (df_dn["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(10)["Term"].tolist()
list_total = list(set(list_up + list_dn))
plot_df = plot_df[plot_df["Term"].isin(list_total)]

## Sort by NES
plot_df = plot_df.sort_values("NES", ascending=False)

## Percentage overlap
plot_df["Overlap %"] = (
    plot_df["Tag %"]
    .str.split("/")
    .apply(lambda x: int(x[0]) / int(x[1]) * 100)
)

# Plot
plt.figure(figsize=(12, 8))

sb.scatterplot(
    data=plot_df,
    x="NES",
    y="Term",
    size="Overlap %",
    hue="FDR q-val",
    palette="RdBu",
    sizes=(50, 400),
    edgecolor="black"
)

plt.xlabel("NES")
plt.ylabel("")
plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    borderaxespad=0
)

plt.tight_layout()
plt.show()

#### Adding manually annotated gene sets

In [ ]:
plot_df = pd.concat([df_up, df_dn]).drop_duplicates()

## Filter terms
list_up = df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()
list_dn = df_dn[(df_dn["NES" ]< 0) & (df_dn["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(10)["Term"].tolist()
## Find terms with antigen presentation, apoptosis, caspases, and GC
list_apop = plot_df[plot_df["Term"].str.contains("Apop") |
    plot_df["Term"].str.contains("Caspase") |
    plot_df["Term"].str.contains("Granzyme")].sort_values("FDR q-val")["Term"].tolist()
list_total = list(set(list_up + list_dn+list_apop))

In [ ]:

plot_df = pd.concat([df_up, df_dn])

## Filter terms
plot_df = plot_df[plot_df["Term"].isin(list_total)]

## Sort by NES
plot_df = plot_df.sort_values("NES", ascending=False)

## Percentage overlap
plot_df["Overlap %"] = (
    plot_df["Tag %"]
    .str.split("/")
    .apply(lambda x: int(x[0]) / int(x[1]) * 100)
)

# Plot
plt.figure(figsize=(12, 12))

sb.scatterplot(
    data=plot_df,
    x="NES",
    y="Term",
    size="Overlap %",
    hue="FDR q-val",
    palette="RdBu",
    sizes=(50, 400),
    edgecolor="black"
)

plt.title("GOBP")
plt.xlabel("NES")
plt.ylabel("")
plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    borderaxespad=0
)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_PATHWAYS_GOBP_PBLa.pdf", format="pdf", dpi=400)
plt.show()

### GOMF

In [ ]:
#Subset 1 comes from T cell singlets vs B cell singlets
subset1 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Bcells")[["names", "logfoldchanges", "pvals_adj"]]
subset1 = subset1.rename(columns={"logfoldchanges":"TvB_lfc", "pvals_adj":"pvals_adj_TvB_lfc"})
subset1 = subset1.set_index("names")

#Subset 2 comes from sampled T cell singlet + sampled B cell (all observed B cell) versus T-B cell doublets. 
subset2 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="sampled_vs_T_B_doublets")
subset2 = subset2[["names", "scores", "logfoldchanges", "pvals_adj"]]
subset2 = subset2.set_index("names")

#The dataframes are now joined via indexes
plot_df = subset2.join(subset1).reset_index()

# .. and filtered based on the combined gene list
plot_df = plot_df[plot_df["names"].isin(combined_list)]

# ... and focus on statistical significant differences (cuts the number of dots from 5000+ to around 3000)
plot_df = plot_df[(plot_df["pvals_adj_TvB_lfc"]<0.05) | (plot_df["pvals_adj"]<0.05)]

# .... annotation for staitical significance
plot_df["stat_cat"] = np.where((plot_df["pvals_adj"]<0.05) & (plot_df["pvals_adj_TvB_lfc"]<0.05), "p.adj < 0.05 in both", "Irrellevant")

## Run enrichment (online)
deg = plot_df[(plot_df["pvals_adj"] < 0.05) & (plot_df["logfoldchanges"] > 0)]["names"].tolist()

## A ranked list
rnk = (
    plot_df[["names", "scores"]]
    .dropna()
    .drop_duplicates("names")
    .sort_values("scores", ascending=False)
)


pre_mult = gp.prerank(
    rnk=rnk,    # single ranked list
    gene_sets="GO_Molecular_Function_2026",
    method='multilevel',            # use fgsea multilevel p-values
    eps=1e-90,                      # smallest p-value resolved; 0 => machine precision eps (float) – Only used when method='multilevel'. Lower boundary for the estimated p-value; p-values smaller than eps are reported as eps. Set to 0 to estimate p-values as small as machine precision allows. Default: 1e-50.
    threads=4,
    min_size=15,                    #Default gene set size max is 15 (gets rid of many disease GOBP gene sets)
    max_size=500,                   #Defailt gene set size max is 500
    sample_size=101,                #sample_size (int) – Only used when method='multilevel'. Sample size for the multilevel split step of the fgsea algorithm; larger values give more accurate (but slower) tail p-value estimates. Default: 101.
    seed=6,
    outdir=None,
)
pre_mult.res2d.head(10)


In [ ]:
df_up = pre_mult.res2d.copy()
df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()

In [ ]:
df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(5)

In [ ]:
from gseapy import gseaplot
terms = 'Endopeptidase Regulator Activity (GO:0061135)'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.show()

In [ ]:
df_dn = pre_mult.res2d.copy()
df_dn[(df_dn["NES"] < 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()

In [ ]:
terms =  "MRNA 5'-UTR Binding (GO:0048027)"

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.show()

In [ ]:
plot_df = pd.concat([df_up, df_dn])

## Filter terms
list_up = df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()
list_dn = df_dn[(df_dn["NES" ]< 0) & (df_dn["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(10)["Term"].tolist()
list_total = list(set(list_up + list_dn))
plot_df = plot_df[plot_df["Term"].isin(list_total)]

## Sort by NES
plot_df = plot_df.sort_values("NES", ascending=False)

## Percentage overlap
plot_df["Overlap %"] = (
    plot_df["Tag %"]
    .str.split("/")
    .apply(lambda x: int(x[0]) / int(x[1]) * 100)
)

# Plot
plt.figure(figsize=(10, 8))

sb.scatterplot(
    data=plot_df,
    x="NES",
    y="Term",
    size="Overlap %",
    hue="FDR q-val",
    palette="RdBu",
    sizes=(50, 400),
    edgecolor="black"
)

plt.xlabel("NES")
plt.ylabel("")
plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    borderaxespad=0
)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_PATHWAYS_GOMF_PBLa.pdf", dpi=400, format="pdf")
plt.show()

In [ ]:
#Subset 1 comes from T cell singlets vs B cell singlets
subset1 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Bcells")[["names", "logfoldchanges", "pvals_adj"]]
subset1 = subset1.rename(columns={"logfoldchanges":"TvB_lfc", "pvals_adj":"pvals_adj_TvB_lfc"})
subset1 = subset1.set_index("names")

#Subset 2 comes from sampled T cell singlet + sampled B cell (all observed B cell) versus T-B cell doublets. 
subset2 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="sampled_vs_T_B_doublets")
subset2 = subset2[["names", "scores", "logfoldchanges", "pvals_adj"]]
subset2 = subset2.set_index("names")

#The dataframes are now joined via indexes
plot_df = subset2.join(subset1).reset_index()

# .. and filtered based on the combined gene list
plot_df = plot_df[plot_df["names"].isin(combined_list)]

# ... and focus on statistical significant differences (cuts the number of dots from 5000+ to around 3000)
plot_df = plot_df[(plot_df["pvals_adj_TvB_lfc"]<0.05) | (plot_df["pvals_adj"]<0.05)]

# .... annotation for staitical significance
plot_df["stat_cat"] = np.where((plot_df["pvals_adj"]<0.05) & (plot_df["pvals_adj_TvB_lfc"]<0.05), "p.adj < 0.05 in both", "Irrellevant")

## Run enrichment (online)
deg = plot_df[(plot_df["pvals_adj"] < 0.05) & (plot_df["logfoldchanges"] > 0)]["names"].tolist()

## A ranked list
rnk = (
    plot_df[["names", "scores"]]
    .dropna()
    .drop_duplicates("names")
    .sort_values("scores", ascending=False)
)


pre_mult = gp.prerank(
    rnk=rnk,    # single ranked list
    gene_sets="Gene_Perturbations_from_GEO_up",
    method='multilevel',            # use fgsea multilevel p-values
    eps=1e-90,                      # smallest p-value resolved; 0 => machine precision eps (float) – Only used when method='multilevel'. Lower boundary for the estimated p-value; p-values smaller than eps are reported as eps. Set to 0 to estimate p-values as small as machine precision allows. Default: 1e-50.
    threads=4,
    min_size=15,                    #Default gene set size max is 15 (gets rid of many disease GOBP gene sets)
    max_size=500,                   #Defailt gene set size max is 500
    sample_size=101,                #sample_size (int) – Only used when method='multilevel'. Sample size for the multilevel split step of the fgsea algorithm; larger values give more accurate (but slower) tail p-value estimates. Default: 101.
    seed=6,
    outdir=None,
)
pre_mult.res2d.head(10)


In [ ]:
df_up = pre_mult.res2d.copy()
df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()

In [ ]:
df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(10)

In [ ]:
from gseapy import gseaplot
terms = 'ERO1L OE human GSE40601 sample 2348'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.show()

In [ ]:
df_dn = pre_mult.res2d.copy()
df_dn[(df_dn["NES"] < 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()

In [ ]:
terms =  "NFKB1 Inactivation  (IKK Inhibition) human GSE20667 sample 2576"

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.show()

In [ ]:
plot_df = pd.concat([df_up, df_dn])

## Filter terms
list_up = df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()
list_dn = df_dn[(df_dn["NES" ]< 0) & (df_dn["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(10)["Term"].tolist()
list_total = list(set(list_up + list_dn))
plot_df = plot_df[plot_df["Term"].isin(list_total)]

## Sort by NES
plot_df = plot_df.sort_values("NES", ascending=False)

## Percentage overlap
plot_df["Overlap %"] = (
    plot_df["Tag %"]
    .str.split("/")
    .apply(lambda x: int(x[0]) / int(x[1]) * 100)
)

# Plot
plt.figure(figsize=(12, 8))

sb.scatterplot(
    data=plot_df,
    x="NES",
    y="Term",
    size="Overlap %",
    hue="FDR q-val",
    palette="RdBu",
    sizes=(50, 400),
    edgecolor="black"
)

plt.xlabel("NES")
plt.ylabel("")
plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    borderaxespad=0
)

plt.tight_layout()
plt.show()

## gprofiler

In [ ]:
## List of available libraries
names = gp.get_library_name()
names

### Pathways UP

In [ ]:
#Subset 1 comes from T cell singlets vs B cell singlets
subset1 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Bcells")[["names", "logfoldchanges", "pvals_adj"]]
subset1 = subset1.rename(columns={"logfoldchanges":"TvB_lfc", "pvals_adj":"pvals_adj_TvB_lfc"})
subset1 = subset1.set_index("names")

#Subset 2 comes from sampled T cell singlet + sampled B cell (all observed B cell) versus T-B cell doublets. 
subset2 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="sampled_vs_T_B_doublets")
subset2 = subset2[["names", "scores", "logfoldchanges", "pvals_adj"]]
subset2 = subset2.set_index("names")

#The dataframes are now joined via indexes
plot_df = subset2.join(subset1).reset_index()

# .. and filtered based on the combined gene list
plot_df = plot_df[plot_df["names"].isin(combined_list)]

# ... and focus on statistical significant differences (cuts the number of dots from 5000+ to around 3000)
plot_df = plot_df[(plot_df["pvals_adj_TvB_lfc"]<0.05) | (plot_df["pvals_adj"]<0.05)]

# .... annotation for staitical significance
plot_df["stat_cat"] = np.where((plot_df["pvals_adj"]<0.05) & (plot_df["pvals_adj_TvB_lfc"]<0.05), "p.adj < 0.05 in both", "Irrellevant")

## Run enrichment (online)
deg = plot_df[(plot_df["pvals_adj"] < 0.05) & (plot_df["logfoldchanges"] > 0)]["names"].tolist()

gene_list = deg                            # or a Python list / Series
gene_sets1 = "KEGG_2026"        # several, comma-separated
gene_sets2 = "GO_Molecular_Function_2026"        # several, comma-separated
gene_sets3 = "GO_Biological_Process_2026"        # several, comma-separated
gene_sets4 = "Reactome_Pathways_2024"        # several, comma-separated
gene_sets5 = "MSigDB_Hallmark_2020"        # several, comma-separated
gene_sets6 = "BioCarta_2016"        # several, comma-separated
gene_sets7 = "MSigDB_Computational"        # several, comma-separated

## Enrich online without a custom background
enr1dat = gp.enrichr(
    gene_list=gene_list,                              # or "./tests/data/gene_list.txt"
    gene_sets=gene_sets1,
    organism='human',                                 # set to your organism, e.g. 'Yeast'
    outdir=None,                                       # keep results in memory only
)

## Enrich online without a custom background
enr2dat = gp.enrichr(
    gene_list=gene_list,                              # or "./tests/data/gene_list.txt"
    gene_sets=gene_sets2,
    organism='human',                                 # set to your organism, e.g. 'Yeast'
    outdir=None,                                       # keep results in memory only
)

## Enrich online without a custom background
enr3dat = gp.enrichr(
    gene_list=gene_list,                              # or "./tests/data/gene_list.txt"
    gene_sets=gene_sets3,
    organism='human',                                 # set to your organism, e.g. 'Yeast'
    outdir=None,                                       # keep results in memory only
)

## Enrich online without a custom background
enr4dat = gp.enrichr(
    gene_list=gene_list,                              # or "./tests/data/gene_list.txt"
    gene_sets=gene_sets4,
    organism='human',                                 # set to your organism, e.g. 'Yeast'
    outdir=None,                                       # keep results in memory only
)

## Enrich online without a custom background
enr5dat = gp.enrichr(
    gene_list=gene_list,                              # or "./tests/data/gene_list.txt"
    gene_sets=gene_sets5,
    organism='human',                                 # set to your organism, e.g. 'Yeast'
    outdir=None,                                       # keep results in memory only
)

## Enrich online without a custom background
enr6dat = gp.enrichr(
    gene_list=gene_list,                              # or "./tests/data/gene_list.txt"
    gene_sets=gene_sets6,
    organism='human',                                 # set to your organism, e.g. 'Yeast'
    outdir=None,                                       # keep results in memory only
)

## Enrich online without a custom background
enr7dat = gp.enrichr(
    gene_list=gene_list,                              # or "./tests/data/gene_list.txt"
    gene_sets=gene_sets7,
    organism='human',                                 # set to your organism, e.g. 'Yeast'
    outdir=None,                                       # keep results in memory only
)

In [ ]:
enr1 = enr1dat.results
enr2 = enr2dat.results
enr3 = enr3dat.results
enr4 = enr4dat.results
enr5 = enr5dat.results
# enr6 = enr6.results
# enr7 = enr7.results

In [ ]:
total_enr = pd.concat([enr1, enr2, enr3, enr4, enr5])
total_enr.sort_values("Adjusted P-value", ascending=True).head(10)

### Top terms

In [ ]:
# Bar plot grouped by library, with an explicit color per group.
ax = barplot(
    total_enr, 
    column="Adjusted P-value",
    group='Gene_set',      # group bars by library (multi-library comparison)
    size=10,
    top_term=10,
    figsize=(3, 5),
    color={'GO_Biological_Process_2026': 'blue', 'GO_Molecular_Function_2026': 'navy', 
          "KEGG_2026" : "salmon", 
          "Reactome_Pathways_2024" : "teal", 
          "MSigDB_Hallmark_2020" : "grey"},
)

ax.tick_params(axis="y", labelsize=8)

plt.show()

### Pathways down

In [ ]:
#Subset 1 comes from T cell singlets vs B cell singlets
subset1 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Bcells")[["names", "logfoldchanges", "pvals_adj"]]
subset1 = subset1.rename(columns={"logfoldchanges":"TvB_lfc", "pvals_adj":"pvals_adj_TvB_lfc"})
subset1 = subset1.set_index("names")

#Subset 2 comes from sampled T cell singlet + sampled B cell (all observed B cell) versus T-B cell doublets. 
subset2 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="sampled_vs_T_B_doublets")
subset2 = subset2[["names", "scores", "logfoldchanges", "pvals_adj"]]
subset2 = subset2.set_index("names")

#The dataframes are now joined via indexes
plot_df = subset2.join(subset1).reset_index()

# .. and filtered based on the combined gene list
plot_df = plot_df[plot_df["names"].isin(combined_list)]

# ... and focus on statistical significant differences (cuts the number of dots from 5000+ to around 3000)
plot_df = plot_df[(plot_df["pvals_adj_TvB_lfc"]<0.05) | (plot_df["pvals_adj"]<0.05)]

# .... annotation for staitical significance
plot_df["stat_cat"] = np.where((plot_df["pvals_adj"]<0.05) & (plot_df["pvals_adj_TvB_lfc"]<0.05), "p.adj < 0.05 in both", "Irrellevant")

## Run enrichment (online)
deg = plot_df[(plot_df["pvals_adj"] < 0.05) & (plot_df["logfoldchanges"] < 0)]["names"].tolist()

gene_list = deg                            # or a Python list / Series
gene_sets1 = "KEGG_2026"        # several, comma-separated
gene_sets2 = "GO_Molecular_Function_2026"        # several, comma-separated
gene_sets3 = "GO_Biological_Process_2026"        # several, comma-separated
gene_sets4 = "Reactome_Pathways_2024"        # several, comma-separated
gene_sets5 = "MSigDB_Hallmark_2020"        # several, comma-separated
gene_sets6 = "BioCarta_2016"        # several, comma-separated
gene_sets7 = "MSigDB_Computational"        # several, comma-separated

## Enrich online without a custom background
enr1 = gp.enrichr(
    gene_list=gene_list,                              # or "./tests/data/gene_list.txt"
    gene_sets=gene_sets1,
    organism='human',                                 # set to your organism, e.g. 'Yeast'
    outdir=None,                                       # keep results in memory only
)

## Enrich online without a custom background
enr2 = gp.enrichr(
    gene_list=gene_list,                              # or "./tests/data/gene_list.txt"
    gene_sets=gene_sets2,
    organism='human',                                 # set to your organism, e.g. 'Yeast'
    outdir=None,                                       # keep results in memory only
)

## Enrich online without a custom background
enr3 = gp.enrichr(
    gene_list=gene_list,                              # or "./tests/data/gene_list.txt"
    gene_sets=gene_sets3,
    organism='human',                                 # set to your organism, e.g. 'Yeast'
    outdir=None,                                       # keep results in memory only
)

## Enrich online without a custom background
enr4 = gp.enrichr(
    gene_list=gene_list,                              # or "./tests/data/gene_list.txt"
    gene_sets=gene_sets4,
    organism='human',                                 # set to your organism, e.g. 'Yeast'
    outdir=None,                                       # keep results in memory only
)

## Enrich online without a custom background
enr5 = gp.enrichr(
    gene_list=gene_list,                              # or "./tests/data/gene_list.txt"
    gene_sets=gene_sets5,
    organism='human',                                 # set to your organism, e.g. 'Yeast'
    outdir=None,                                       # keep results in memory only
)

## Enrich online without a custom background
enr6 = gp.enrichr(
    gene_list=gene_list,                              # or "./tests/data/gene_list.txt"
    gene_sets=gene_sets6,
    organism='human',                                 # set to your organism, e.g. 'Yeast'
    outdir=None,                                       # keep results in memory only
)

## Enrich online without a custom background
enr7 = gp.enrichr(
    gene_list=gene_list,                              # or "./tests/data/gene_list.txt"
    gene_sets=gene_sets7,
    organism='human',                                 # set to your organism, e.g. 'Yeast'
    outdir=None,                                       # keep results in memory only
)

In [ ]:
enr1 = enr1.results
enr2 = enr2.results
enr3 = enr3.results
enr4 = enr4.results
enr5 = enr5.results
# enr6 = enr6.results
# enr7 = enr7.results

In [ ]:
total_enr = pd.concat([enr1, enr2, enr3, enr4, enr5])
total_enr.sort_values("Adjusted P-value", ascending=True).head(10)

### Bottom terms

In [ ]:
# Bar plot grouped by library, with an explicit color per group.
ax = barplot(
    total_enr, 
    column="Adjusted P-value",
    group='Gene_set',      # group bars by library (multi-library comparison)
    size=10,
    top_term=5,
    figsize=(3, 5),
    color={'GO_Biological_Process_2026': 'blue', 'GO_Molecular_Function_2026': 'navy', 
          "KEGG_2026" : "salmon", 
          "Reactome_Pathways_2024" : "teal", 
          "MSigDB_Hallmark_2020" : "grey"},
)

ax.tick_params(axis="y", labelsize=8)

plt.show()

## decoupler dc.tl.ulm()

In [ ]:
#Subset 1 comes from T cell singlets vs B cell singlets
subset1 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Bcells")[["names", "logfoldchanges", "pvals_adj"]]
subset1 = subset1.rename(columns={"logfoldchanges":"TvB_lfc", "pvals_adj":"pvals_adj_TvB_lfc"})
subset1 = subset1.set_index("names")

#Subset 2 comes from sampled T cell singlet + sampled B cell (all observed B cell) versus T-B cell doublets. 
subset2 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="sampled_vs_T_B_doublets")
subset2 = subset2[["names", "scores", "logfoldchanges", "pvals_adj"]]
subset2 = subset2.set_index("names")

#The dataframes are now joined via indexes
plot_df = subset2.join(subset1).reset_index()

# .. and filtered based on the combined gene list
plot_df = plot_df[plot_df["names"].isin(combined_list)]

# ... and focus on statistical significant differences (cuts the number of dots from 5000+ to around 3000)
plot_df = plot_df[(plot_df["pvals_adj_TvB_lfc"]<0.05) | (plot_df["pvals_adj"]<0.05)]

# .... annotation for staitical significance
plot_df["stat_cat"] = np.where((plot_df["pvals_adj"]<0.05) & (plot_df["pvals_adj_TvB_lfc"]<0.05), "p.adj < 0.05 in both", "Irrellevant")

## Run enrichment (online)
# deg = plot_df[(plot_df["pvals_adj"] < 0.05) & (plot_df["pvals_adj_TvB_lfc"] < 0.05)]
deg = plot_df[(plot_df["pvals_adj"] < 0.05)]
deg = deg.set_index("names")

## The tutorial asks to use a statistics that takes into account significance and not just log2fc. I will use "scores" for that. 
### https://decoupler.readthedocs.io/en/latest/notebooks/scell/rna_psbk.html
### They also do it here: https://www.sc-best-practices.org/conditions/gsea_pathway.html
data = deg[["scores"]].T.rename(index={"stat": "observed.vs.simulated"})

# extract scores 
data = (
    deg
    # Sort by absolute score
    .sort_values("scores", ascending=False)[
        # Format for decoupler
        ["scores"]
    ]
    .rename_axis(["Obs_vs_sim"], axis=1)
)
data.tail(10)

In [ ]:
dc.op.show_resources()

In [ ]:
# Retrieving via python
# msigdb = dc.op.resource("MSigDB")

In [ ]:
msigdb["collection"].value_counts()

### Hallmark pathways

In [ ]:
# Get reactome pathways
reactome = msigdb.query("collection == 'hallmark'")
# Filter duplicates
reactome = reactome[~reactome.duplicated(("geneset", "genesymbol"))]

# Filtering genesets to match behaviour of fgsea
geneset_size = reactome.groupby("geneset").size()
gsea_genesets = geneset_size.index[(geneset_size > 15) & (geneset_size < 500)]

## Prepare the net db
reactome_renamed = reactome[reactome["geneset"].isin(gsea_genesets)]
reactome_renamed = reactome_renamed.rename(columns={"genesymbol" : "target", "geneset" : "source"})

In [ ]:
# We will use the scores from the Wilcox test - these take into account the significance of a gene
scores, pvals = dc.mt.ulm(
    data.T,
    reactome_renamed,
)

gsea_results = (
    pd.concat({"score": scores.T, "pval": pvals.T}, axis=1)
    .droplevel(level=1, axis=1)
    .sort_values("pval")
)

gsea_results.head(30)

g_res1 = gsea_results.copy()
g_res1["source"] = "hallmark"

In [ ]:
# Filter by sign padj
msk = (pvals.T < 0.05).iloc[:, 0]
scores = scores.loc[:, msk]

dc.pl.barplot(data=scores, name="scores", figsize=(6, 3))
plt.show()

In [ ]:
_, le = dc.pl.leading_edge(
    data,
    stat="scores",
    net=reactome_renamed,
    name="HALLMARK_P53_PATHWAY",
)
print("leading edge:", le[:10])
plt.show()

In [ ]:
_, le = dc.pl.leading_edge(
    data,
    stat="scores",
    net=reactome_renamed,
    name="HALLMARK_UNFOLDED_PROTEIN_RESPONSE",
)
print("leading edge:", le[:10])
plt.show()

### Immunesigdb pathways

In [ ]:
# Get reactome pathways
reactome = msigdb.query("collection == 'immunesigdb'")
# Filter duplicates
reactome = reactome[~reactome.duplicated(("geneset", "genesymbol"))]

# Filtering genesets to match behaviour of fgsea
geneset_size = reactome.groupby("geneset").size()
gsea_genesets = geneset_size.index[(geneset_size > 15) & (geneset_size < 500)]

## Prepare the net db
reactome_renamed = reactome[reactome["geneset"].isin(gsea_genesets)]
reactome_renamed = reactome_renamed.rename(columns={"genesymbol" : "target", "geneset" : "source"})

In [ ]:
# We will use the scores from the Wilcox test - these take into account the significance of a gene
scores, pvals = dc.mt.ulm(
    data.T,
    reactome_renamed,
)

gsea_results = (
    pd.concat({"score": scores.T, "pval": pvals.T}, axis=1)
    .droplevel(level=1, axis=1)
    .sort_values("pval")
)

# gsea_results = gsea_results.reset_index()[(gsea_results.reset_index()["index"].str.contains("BCELL")) | (gsea_results.reset_index()["index"].str.contains("PLASMA")) | (gsea_results.reset_index()["index"].str.contains("TCELL"))]
# gsea_results = gsea_results[gsea_results["pval"]< 0.05]
# gsea_list = gsea_results["index"].tolist()

gsea_results = gsea_results[gsea_results.index.str.contains("_UP")]
g_res2 = gsea_results.copy()
g_res2["source"] = "immunesigdb"

In [ ]:
g_res2.sort_values("score", ascending=False).tail(20)

In [ ]:
# Filter by sign padj
# msk = ((pvals.T < 0.05).iloc[:, 0]) & ((scores.T > 0).iloc[:, 0])
msk = gsea_results.nlargest(5, "score").index.union(gsea_results.nsmallest(5, "score").index)
# msk = scores.columns.isin(gsea_list)
scores = scores.loc[:, msk]

dc.pl.barplot(data=scores, name="scores", figsize=(5, 6))
plt.show()

In [ ]:
_, le = dc.pl.leading_edge(
    data,
    stat="scores",
    net=reactome_renamed,
    name="GSE6269_HEALTHY_VS_FLU_INF_PBMC_UP",
)
print("leading edge:", le[:10])
plt.show()

In [ ]:
_, le = dc.pl.leading_edge(
    data,
    stat="scores",
    net=reactome_renamed,
    name="GSE6269_HEALTHY_VS_FLU_INF_PBMC_UP",
)
print("leading edge:", le[:10])
plt.show()

### KEGG pathways

In [ ]:
# Get reactome pathways
reactome = msigdb.query("collection == 'kegg_pathways'")
# Filter duplicates
reactome = reactome[~reactome.duplicated(("geneset", "genesymbol"))]

# Filtering genesets to match behaviour of fgsea
geneset_size = reactome.groupby("geneset").size()
gsea_genesets = geneset_size.index[(geneset_size > 15) & (geneset_size < 500)]

## Prepare the net db
reactome_renamed = reactome[reactome["geneset"].isin(gsea_genesets)]
reactome_renamed = reactome_renamed.rename(columns={"genesymbol" : "target", "geneset" : "source"})

In [ ]:
# We will use the scores from the Wilcox test - these take into account the significance of a gene
scores, pvals = dc.mt.ulm(
    data.T,
    reactome_renamed,
)

gsea_results = (
    pd.concat({"score": scores.T, "pval": pvals.T}, axis=1)
    .droplevel(level=1, axis=1)
    .sort_values("pval")
)

gsea_results.head(30)
g_res3 = gsea_results.copy()
g_res3["source"] = "kegg_pathways"

In [ ]:
# Filter by sign padj
msk = (pvals.T < 0.05).iloc[:, 0]
scores = scores.loc[:, msk]

dc.pl.barplot(data=scores, name="scores", figsize=(6, 3))
plt.show()

### go_biological_process pathways

In [ ]:
# Get reactome pathways
reactome = msigdb.query("collection == 'go_biological_process'")
# Filter duplicates
reactome = reactome[~reactome.duplicated(("geneset", "genesymbol"))]

# Filtering genesets to match behaviour of fgsea
geneset_size = reactome.groupby("geneset").size()
gsea_genesets = geneset_size.index[(geneset_size > 15) & (geneset_size < 500)]

## Prepare the net db
reactome_renamed = reactome[reactome["geneset"].isin(gsea_genesets)]
reactome_renamed = reactome_renamed.rename(columns={"genesymbol" : "target", "geneset" : "source"})

In [ ]:
# We will use the scores from the Wilcox test - these take into account the significance of a gene
scores, pvals = dc.mt.ulm(
    data.T,
    reactome_renamed,
)

gsea_results = (
    pd.concat({"score": scores.T, "pval": pvals.T}, axis=1)
    .droplevel(level=1, axis=1)
    .sort_values("pval")
)

gsea_results.head(30)
g_res4 = gsea_results.copy()
g_res4["source"] = "go_biological_process"

In [ ]:
# Filter by sign padj
# msk = (pvals.T < 0.05).iloc[:, 0]
msk = gsea_results.nlargest(5, "score").index.union(gsea_results.nsmallest(5, "score").index)
scores = scores.loc[:, msk]

dc.pl.barplot(data=scores, name="scores", figsize=(6, 6))
plt.show()

### chemical_and_genetic_perturbations 

In [ ]:
# Get reactome pathways
reactome = msigdb.query("collection == 'chemical_and_genetic_perturbations'")
# Filter duplicates
reactome = reactome[~reactome.duplicated(("geneset", "genesymbol"))]

# Filtering genesets to match behaviour of fgsea
geneset_size = reactome.groupby("geneset").size()
gsea_genesets = geneset_size.index[(geneset_size > 15) & (geneset_size < 500)]

## Prepare the net db
reactome_renamed = reactome[reactome["geneset"].isin(gsea_genesets)]
reactome_renamed = reactome_renamed.rename(columns={"genesymbol" : "target", "geneset" : "source"})

In [ ]:
# We will use the scores from the Wilcox test - these take into account the significance of a gene
scores, pvals = dc.mt.ulm(
    data.T,
    reactome_renamed,
)

gsea_results = (
    pd.concat({"score": scores.T, "pval": pvals.T}, axis=1)
    .droplevel(level=1, axis=1)
    .sort_values("pval")
)

gsea_results.head(30)
g_res5 = gsea_results.copy()
g_res5["source"] = "chemical_and_genetic_perturbations"

In [ ]:
g_res5.sort_values("score", ascending=False)

In [ ]:
# Filter by sign padj
# msk = (pvals.T < 0.05).iloc[:, 0]
msk = gsea_results.nlargest(5, "score").index.union(gsea_results.nsmallest(5, "score").index)
scores = scores.loc[:, msk]

dc.pl.barplot(data=scores, name="scores", figsize=(8, 6))
plt.show()

In [ ]:
_, le = dc.pl.leading_edge(
    data,
    stat="scores",
    net=reactome_renamed,
    name="PASQUALUCCI_LYMPHOMA_BY_GC_STAGE_UP",
)
print("leading edge:", le[:10])
plt.show()

### Reactome

In [ ]:
# Get reactome pathways
reactome = msigdb.query("collection == 'reactome_pathways'")
# Filter duplicates
reactome = reactome[~reactome.duplicated(("geneset", "genesymbol"))]

# Filtering genesets to match behaviour of fgsea
geneset_size = reactome.groupby("geneset").size()
gsea_genesets = geneset_size.index[(geneset_size > 15) & (geneset_size < 500)]

## Prepare the net db
reactome_renamed = reactome[reactome["geneset"].isin(gsea_genesets)]
reactome_renamed = reactome_renamed.rename(columns={"genesymbol" : "target", "geneset" : "source"})

In [ ]:
# We will use the scores from the Wilcox test - these take into account the significance of a gene
scores, pvals = dc.mt.ulm(
    data.T,
    reactome_renamed,
)

gsea_results = (
    pd.concat({"score": scores.T, "pval": pvals.T}, axis=1)
    .droplevel(level=1, axis=1)
    .sort_values("pval")
)

gsea_results.head(30)
g_res6 = gsea_results.copy()
g_res6["source"] = "Reactome"

In [ ]:
# Filter by sign padj
# msk = (pvals.T < 0.05).iloc[:, 0]
msk = gsea_results.nlargest(5, "score").index.union(gsea_results.nsmallest(5, "score").index)
scores = scores.loc[:, msk]

dc.pl.barplot(data=scores, name="scores", figsize=(10, 6))
plt.show()

### Reactome

In [ ]:
# Get reactome pathways
reactome = msigdb.query("collection == 'reactome_pathways'")
# Filter duplicates
reactome = reactome[~reactome.duplicated(("geneset", "genesymbol"))]

# Filtering genesets to match behaviour of fgsea
geneset_size = reactome.groupby("geneset").size()
gsea_genesets = geneset_size.index[(geneset_size > 15) & (geneset_size < 500)]

## Prepare the net db
reactome_renamed = reactome[reactome["geneset"].isin(gsea_genesets)]
reactome_renamed = reactome_renamed.rename(columns={"genesymbol" : "target", "geneset" : "source"})

In [ ]:
# We will use the scores from the Wilcox test - these take into account the significance of a gene
scores, pvals = dc.mt.ulm(
    data.T,
    reactome_renamed,
)

gsea_results = (
    pd.concat({"score": scores.T, "pval": pvals.T}, axis=1)
    .droplevel(level=1, axis=1)
    .sort_values("pval")
)

gsea_results.head(30)

In [ ]:
# Filter by sign padj
msk = (pvals.T < 0.05).iloc[:, 0]
scores = scores.loc[:, msk]

dc.pl.barplot(data=scores, name="scores", figsize=(6, 6))
plt.show()

### Top terms

In [ ]:
total_dc = pd.concat([g_res1, g_res2, g_res3, g_res4, g_res5, g_res6])
top5 = (
    total_dc
    .sort_values("score", ascending=False)
    .groupby("source")
    .head(5)
)

#Only significant ones 
top5 = top5[top5["pval"]<0.05]

# Make a copy
plot_df = top5.reset_index().copy()

# -log10 p-value for coloring
plot_df["-log10(pval)"] = -np.log10(plot_df["pval"])

# Sort so highest score is at the top
plot_df = plot_df.sort_values("score", ascending=False)

## copy
plot_df_up = plot_df.copy()

fig, ax = plt.subplots(figsize=(10, 8))

sb.barplot(
    data=plot_df,
    x="score",
    y="index",
    hue="-log10(pval)",
    palette="RdBu_r",
    dodge=False,
    legend=True,
    ax=ax
)

ax.set_xlabel("Score", fontsize=14)
ax.set_ylabel("Gene set", fontsize=14)
ax.tick_params(axis="y", labelsize=10)

plt.tight_layout()
plt.show()

In [ ]:
total_dc = pd.concat([g_res1, g_res2, g_res3, g_res4, g_res5, g_res6])
top5 = (
    total_dc
    .sort_values("score", ascending=False)
    .groupby("source")
    .tail(5)
)

#Only significant ones 
top5 = top5[top5["pval"]<0.05]

## Adjust the p-value to lowest value if 0
top5["pval"] = np.where(top5["pval"] == 0, 4.063766e-44, top5["pval"])

# Make a copy
plot_df = top5.reset_index().copy()

# -log10 p-value for coloring
plot_df["-log10(pval)"] = -np.log10(plot_df["pval"])

# Sort so highest score is at the bottom
plot_df = plot_df.sort_values("score", ascending=False)

## copy
plot_df_dn = plot_df.copy()

fig, ax = plt.subplots(figsize=(10, 8))

sb.barplot(
    data=plot_df,
    x="score",
    y="index",
    hue="-log10(pval)",
    palette="RdBu_r",
    dodge=False,
    legend=True,
    ax=ax
)

ax.set_xlabel("Score", fontsize=14)
ax.set_ylabel("Gene set", fontsize=14)
ax.tick_params(axis="y", labelsize=10)

plt.tight_layout()
plt.show()

In [ ]:
## all
plot_df_all = pd.concat([plot_df_up, plot_df_dn])

fig, ax = plt.subplots(figsize=(10, 8))

sb.barplot(
    data=plot_df_all,
    x="score",
    y="index",
    hue="-log10(pval)",
    palette="RdBu_r",
    dodge=False,
    legend=True,
    ax=ax
)

ax.set_xlabel("Score", fontsize=14)
ax.set_ylabel("Gene set", fontsize=14)
ax.tick_params(axis="y", labelsize=10)

plt.tight_layout()
plt.show()

# XY plot - T-MG/BAM, T cell and MG/BAM singlets

### Preprocessing (doublet simulation)
Returns **csf_with_doublets**, the final AnnData object that we will use for differential gene expression

In [ ]:
# -------------------------
# 1. Restrict to selected donors with MG/BAM and T-MG/BAM
# -------------------------
obs_df = CSF_data["gex"].obs.copy()

# Make a donor ist of allowed donors
allowed_donors = obs_df[
    obs_df["Recluster_ann2"].isin(["MG/BAM", "T-MG/BAM"])
]["donor"].unique().tolist()

# filter data based on those donors
csf_dat_subset = CSF_data.copy()
mu.pp.filter_obs(
    csf_dat_subset["gex"],
    "donor",
    lambda x: x.isin(allowed_donors)
)


In [ ]:
obs_df["Recluster_ann2"].value_counts()

In [ ]:
# -------------------------
# 2. Donor-aware T+X simulation
# -------------------------

gex = csf_dat_subset["gex"]
donors = gex.obs["donor"].unique()

sim_doublets_per_donor = []

for donor in donors:
    sub = gex[gex.obs["donor"] == donor]

    APC_cells = sub.obs.query("Recluster_ann2 == 'MG/BAM'").index
    T_cells = sub.obs.query("Recluster_ann2 == 'T cell'").index

    if len(APC_cells) == 0 or len(T_cells) == 0:
        print(f"⚠️ Donor {donor}: Missing T or MG/BAM. Skipping.")
        continue

    n = min(len(APC_cells), len(T_cells))

    APC_sample_idx = sub.obs.loc[APC_cells].sample(n=n, random_state=39).index
    T_sample_idx = sub.obs.loc[T_cells].sample(n=n, random_state=39).index

    APC_subset = sub[APC_sample_idx].copy()
    T_subset = sub[T_sample_idx].copy()

    # Random 1:1 pairing
    T_idx = np.arange(n)
    APC_idx = np.arange(n)
    np.random.shuffle(T_idx)
    np.random.shuffle(APC_idx)
    pairs = list(zip(T_idx, APC_idx))

    # Build simulated matrix
    X_doublets = sparse.vstack([T_subset.X[t] + APC_subset.X[b] for t, b in pairs])
    sim_adata = ad.AnnData(
        X=X_doublets,
        var=T_subset.var.copy()
    )
    sim_adata.obs["Recluster_ann2"] = "Simulated doublets"
    sim_adata.obs["donor"] = donor
    sim_adata.obs.index = [f"{donor}_simTPB_{i}" for i in range(n)]
    sim_doublets_per_donor.append(sim_adata)

if len(sim_doublets_per_donor) == 0:
    raise ValueError("No simulated doublets created.")

sim_all = ad.concat(sim_doublets_per_donor, join="outer", merge="same")

# -------------------------
# 3. Concatenate observed + simulated
# -------------------------
csf_with_doublets = ad.concat(
    {"Observed": gex, "Simulated doublets": sim_all},
    label="source",
    join="outer",
    merge="same"
)

# Normalize + log1p
sc.pp.normalize_total(csf_with_doublets, target_sum=1e4)
sc.pp.log1p(csf_with_doublets)

### Make a list of genes expressed in at least 15% of cells (or interacting cells)

In [ ]:
csf_with_doublets.obs["Recluster_ann2"].value_counts()

In [ ]:
## --------------
## Step 1: Generate lists of expressed genes (>X%) for each of the 3 populations of observed cells T cell, B cell and interacting cells
## --------------


## --- Expression in T cell singlets
adata = csf_with_doublets[csf_with_doublets.obs["Recluster_ann2"] == "T cell"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_1 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 



## --- Expression in B cell singlets
adata = csf_with_doublets[csf_with_doublets.obs["Recluster_ann2"] == "MG/BAM"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_2 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 


## --- Expression in T-MG/BAM
adata = csf_with_doublets[csf_with_doublets.obs["Recluster_ann2"] == "T-MG/BAM"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_3 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 

#The total set of genes
gene_list_10pct_expressed = set(gene_list_10pct_1+gene_list_10pct_2+gene_list_10pct_3)
print(len(gene_list_10pct_expressed))

### T vs MG/BAM cells - differentially expressed genes

In [ ]:
# %%%
# Calculate differentially expressed genes between T cell in CSf and B cell in CSF
# %%% 
sc.tl.rank_genes_groups(csf_with_doublets, groupby="Recluster_ann2", groups=["T cell"], reference="MG/BAM", key_added="T_vs_Xcells")

# %%%% 
# Export all differentially expressed genes
# %%%%
TvBcells_DEG = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Xcells")
TvBcells_DEG.to_csv("../../06_csv_outputs/DEG/3_DoubletAnalysis_DEGs_TMGBAMcells.csv")

# ### This introduces NAs
# sc.tl.filter_rank_genes_groups(
#     csf_with_doublets,
#     min_in_group_fraction=0.2,  #fraction positive in population of interest should be above 0.2.
#     max_out_group_fraction=0.99, #fraction positive !! outside !! the population of interest be less than 0.9
#     min_fold_change=0,      #Since a defaul min_fold_change is enforces, to retain negative logfold changes compare_abs has to be True
#     compare_abs=True,    #Since a defaul min_fold_change is enforces, to retain negative logfold changes compare_abs has to be True
#     key="T_vs_Bcells",
#     key_added="T_vs_Bcells (w. filtering)",
# )

In [ ]:
sc.pl.rank_genes_groups(csf_with_doublets, key="T_vs_Xcells", n_genes=50)

### Simulated vs observed T-B cell - differentially expressed genes

In [ ]:
# %%%
# Calculate differentially expressed genes between random T+B cell vs T-B cell doublets
# %%% 
sc.tl.rank_genes_groups(csf_with_doublets, groupby="Recluster_ann2", groups=["T-MG/BAM"], reference="Simulated doublets", key_added="sampled_vs_TX_doublets")

# %%%% 
# Export all differentially expressed genes
# %%%%
sampledTBcells_DEG = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="sampled_vs_TX_doublets")
TvBcells_DEG.to_csv("../../06_csv_outputs/DEG/3_DoubletAnalysis_DEGs_simulated_vs_observed_TPBcells.csv")

# #Filter it
# ### This introduces NAs
# sc.tl.filter_rank_genes_groups(
#     csf_with_doublets,
#     min_in_group_fraction=0.2,  #fraction positive in population of interest should be above 0.2.
#     max_out_group_fraction=0.99, #fraction positive !! outside !! the population of interest be less than 0.9
#     min_fold_change=0,      #Since a defaul min_fold_change is enforces, to retain negative logfold changes compare_abs has to be True
#     compare_abs=True,    #Since a defaul min_fold_change is enforces, to retain negative logfold changes compare_abs has to be True
#     key="sampled_vs_T_B_doublets",
#     key_added="sampled_vs_T_B_doublets (w. filtering)",
# )


In [ ]:
sc.pl.rank_genes_groups(csf_with_doublets, key="sampled_vs_TX_doublets", n_genes=30)

### Make a combined data frame to plot log2FCs 

In [ ]:
### lists to filter on later - these are differentially expressed genes that passes the significance filter
gene_list1 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Xcells")
gene_list1 = gene_list1[gene_list1["names"].notna()]["names"]

gene_list2 = sc.get.rank_genes_groups_df(csf_with_doublets, group = "T-MG/BAM", key ="sampled_vs_TX_doublets")[["names", "logfoldchanges", "pvals_adj"]]
gene_list2 = gene_list2[gene_list2["names"].notna()]["names"]

combined_list = pd.concat([gene_list1, gene_list2]).drop_duplicates()
print(len(combined_list))
      
### Reduce this gene list to only concern genes that are expressed in at least X pct of T and B cell
combined_list = combined_list[combined_list.isin(gene_list_10pct_expressed)]
print(len(combined_list))

In [ ]:
# %%%%%% 
# Combined the DEGs into 1 plot
# %%%%%

#Subset 1 comes from T cell singlets vs B cell singlets
subset1 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Xcells")[["names", "logfoldchanges", "pvals_adj"]]
subset1 = subset1.rename(columns={"logfoldchanges":"TvX_lfc", "pvals_adj":"pvals_adj_TvX_lfc"})
subset1 = subset1.set_index("names")

#Subset 2 comes from sampled T cell singlet + sampled B cell (all observed B cell) versus T-B cell doublets. 
subset2 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="sampled_vs_TX_doublets")
subset2 = subset2[["names", "scores", "logfoldchanges", "pvals_adj"]]
subset2 = subset2.set_index("names")

#The dataframes are now joined via indexes
plot_df = subset2.join(subset1).reset_index()

# .. and filtered based on the combined gene list
plot_df = plot_df[plot_df["names"].isin(combined_list)]

# ... and focus on statistical significant differences (cuts the number of dots from 5000+ to around 3000)
plot_df = plot_df[(plot_df["pvals_adj_TvX_lfc"]<0.05) | (plot_df["pvals_adj"]<0.05)]

# .... annotation for staitical significance
plot_df["stat_cat"] = np.where((plot_df["pvals_adj"]<0.05) & (plot_df["pvals_adj_TvX_lfc"]<0.05), "p.adj < 0.05 in both", "Irrellevant")

#Save the data for later heatmap
heatmap_TMG = plot_df[plot_df["stat_cat"] == "p.adj < 0.05 in both"]

from adjustText import adjust_text

#Palette
color_dic = {
    "p.adj < 0.05 in both" : "#FF0000",
    "Irrellevant" : "#808080",
}

# Scatter plot
plt.figure(figsize=(12, 10))
# Background points
sb.scatterplot(
    data=plot_df[plot_df["stat_cat"] == "Irrellevant"],
    x="TvX_lfc",
    y="logfoldchanges",
    color="#808080",
    s=30,
    alpha=0.25,
    linewidth=0
)

# Significant points on top
sb.scatterplot(
    data=plot_df[plot_df["stat_cat"] == "p.adj < 0.05 in both"],
    x="TvX_lfc",
    y="logfoldchanges",
    color="#FF0000",
    s=30,
    alpha=1,
    linewidth=0
)
# Identify top genes in T-B cell (highest logfoldchanges)
top_y = plot_df[plot_df["stat_cat"]== "p.adj < 0.05 in both"].nlargest(20, "logfoldchanges")
bot_y = plot_df[plot_df["stat_cat"]== "p.adj < 0.05 in both"].nsmallest(20, "logfoldchanges")
bot_x = plot_df[plot_df["stat_cat"]== "p.adj < 0.05 in both"].nsmallest(10, "TvX_lfc")
top_x = plot_df[plot_df["stat_cat"]== "p.adj < 0.05 in both"].nlargest(10, "TvX_lfc")
niche = plot_df[(plot_df["stat_cat"]== "p.adj < 0.05 in both") & (plot_df["logfoldchanges"]>0.25)].nlargest(20, "TvX_lfc")
other = plot_df[plot_df["names"].isin(["CXCL16"])]
highlight_genes = pd.concat([top_y, bot_y, bot_x, top_x, niche, other]).drop_duplicates()

# Add reference lines
plt.axhline(y=0, color="black", alpha=0.75, linestyle='--', linewidth=1)
plt.axvline(x=0, color="black", alpha=0.75, linestyle='--', linewidth=1)

# Annotate the top_y genes
texts = []
for _, row in highlight_genes.iterrows():
    texts.append(
        plt.text(row["TvX_lfc"], row["logfoldchanges"], row["names"],
                 fontsize=8, color="black")
    )

# Optional: automatically adjust text to prevent overlap (ggrepel-style)
adjust_text(texts, arrowprops=dict(arrowstyle="-", color="gray", lw=0.5))

# Labels and formatting
plt.ylabel("<-- Simulated T+X cell singlets\nvs\nobserved T-X -->")
plt.xlabel("<-- MG/BAM   vs    T cell -->")
plt.ylim(-1.5, 1.5)

sb.despine()
plt.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_TMGBAM_cells_doublets_DEGs_simulated.pdf")
plt.show()

In [ ]:
heatmap_TMG[(heatmap_TMG["logfoldchanges"] < -1) & (heatmap_TMG["TvX_lfc"] < 0)]

In [ ]:
# from brokenaxes import brokenaxes
# from adjustText import adjust_text

# # Create figure first
# fig = plt.figure(figsize=(6, 5))

# # Create broken axes
# bax = brokenaxes(
#     xlims=((-32, -25), (-17, 10)),
#     hspace=0.05,
#     fig=fig
# )

# # Scatter plot
# colors = plot_df["stat_cat"].map(color_dic)

# # Background
# sub = plot_df[plot_df["stat_cat"] == "Irrellevant"]
# bax.scatter(
#     sub["TvB_lfc"],
#     sub["logfoldchanges"],
#     color="#808080",
#     s=30,
#     alpha=0.25
# )

# # Foreground
# sub = plot_df[plot_df["stat_cat"] == "p.adj < 0.05 in both"]
# bax.scatter(
#     sub["TvB_lfc"],
#     sub["logfoldchanges"],
#     color="#FF0000",
#     s=30,
#     alpha=1
# )

# # ---- Identify genes ----
# top_y1 = plot_df[(plot_df["stat_cat"] == "p.adj < 0.05 in both") & (plot_df["TvB_lfc"] > 0.5)].nlargest(30, "logfoldchanges")["names"]
# top_y2 = plot_df[(plot_df["stat_cat"] == "p.adj < 0.05 in both") & (plot_df["TvB_lfc"] < -0.5)].nlargest(10, "logfoldchanges")["names"]
# top_x1 = plot_df[(plot_df["stat_cat"] == "p.adj < 0.05 in both") & (plot_df["TvB_lfc"] > 0.5)].nsmallest(5, "logfoldchanges")["names"]
# top_x2 = plot_df[(plot_df["stat_cat"] == "p.adj < 0.05 in both") & (plot_df["TvB_lfc"] < -0.5)].nsmallest(5, "logfoldchanges")["names"]
# highlight_genes = set(top_y1).union(top_y2).union(top_x1).union(top_x2)
# highlight_genes = highlight_genes | set(["SDC1", "IGHG4", "AC099524.1"])

# # Filter dataframe ONCE (important fix)
# label_df = plot_df[plot_df["names"].isin(highlight_genes)]

# # Reference lines
# for ax in bax.axs:
#     ax.axhline(y=0, color="black", alpha=0.75, linestyle="--", linewidth=1)
#     ax.axvline(x=0, color="black", alpha=0.75, linestyle="--", linewidth=1)

# # ---- Annotate ----
# texts_by_ax = {ax: [] for ax in bax.axs}

# for _, row in label_df.iterrows():

#     for ax in bax.axs:

#         xmin, xmax = ax.get_xlim()
#         ymin, ymax = ax.get_ylim()

#         if (
#             xmin <= row["TvB_lfc"] <= xmax
#             and ymin <= row["logfoldchanges"] <= ymax
#         ):

#             txt = ax.text(
#                 row["TvB_lfc"],
#                 row["logfoldchanges"],
#                 row["names"],
#                 fontsize=8,
#                 color="black"
#             )

#             texts_by_ax[ax].append(txt)
#             break
        
# # Labels
# bax.set_ylabel(
#     "<-- Simulated T+PB singlets\nvs\nobserved T-PB cell -->"
# )
# bax.set_xlabel("<-- PB   vs    T cell -->")

# plt.suptitle(
#     "T and PB cell genes\ndifferentially expressed in T-PB cell",
#     y=1.02
# )
# plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_TPBcells_doublets_DEGs_simulated.pdf")
# plt.show()

# Pathway analysis

## GSEAPY

In [ ]:
## List of available libraries
names = gp.get_library_name()
names

### KEGG

In [ ]:
# %%%%%% 
# Combined the DEGs into 1 plot
# %%%%%

#Subset 1 comes from T cell singlets vs B cell singlets
subset1 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Xcells")[["names", "logfoldchanges", "pvals_adj"]]
subset1 = subset1.rename(columns={"logfoldchanges":"TvX_lfc", "pvals_adj":"pvals_adj_TvX_lfc"})
subset1 = subset1.set_index("names")

#Subset 2 comes from sampled T cell singlet + sampled B cell (all observed B cell) versus T-B cell doublets. 
subset2 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="sampled_vs_TX_doublets")
subset2 = subset2[["names", "scores", "logfoldchanges", "pvals_adj"]]
subset2 = subset2.set_index("names")

#The dataframes are now joined via indexes
plot_df = subset2.join(subset1).reset_index()

# .. and filtered based on the combined gene list
plot_df = plot_df[plot_df["names"].isin(combined_list)]

# ... and focus on statistical significant differences (cuts the number of dots from 5000+ to around 3000)
plot_df = plot_df[(plot_df["pvals_adj_TvX_lfc"]<0.05) | (plot_df["pvals_adj"]<0.05)]

# .... annotation for staitical significance
plot_df["stat_cat"] = np.where((plot_df["pvals_adj"]<0.05) & (plot_df["pvals_adj_TvX_lfc"]<0.05), "p.adj < 0.05 in both", "Irrellevant")

## Run enrichment (online)
deg = plot_df[(plot_df["pvals_adj"] < 0.05) & (plot_df["logfoldchanges"] > 0)]["names"].tolist()

## A ranked list
rnk = (
    plot_df[["names", "scores"]]
    .dropna()
    .drop_duplicates("names")
    .sort_values("scores", ascending=False)
)


pre_mult = gp.prerank(
    rnk=rnk,    # single ranked list
    gene_sets="KEGG_2016",
    method='multilevel',            # use fgsea multilevel p-values
    eps=1e-90,                      # smallest p-value resolved; 0 => machine precision eps (float) – Only used when method='multilevel'. Lower boundary for the estimated p-value; p-values smaller than eps are reported as eps. Set to 0 to estimate p-values as small as machine precision allows. Default: 1e-50.
    threads=4,
    min_size=15,                    #Default gene set size max is 15 (gets rid of many disease GOBP gene sets)
    max_size=500,                   #Defailt gene set size max is 500
    sample_size=101,                #sample_size (int) – Only used when method='multilevel'. Sample size for the multilevel split step of the fgsea algorithm; larger values give more accurate (but slower) tail p-value estimates. Default: 101.
    seed=6,
    outdir=None,
)
pre_mult.res2d.head(10)


In [ ]:
df_up = pre_mult.res2d.copy()
df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("NES", ascending=False).head(20)["Term"].tolist()

In [ ]:
from gseapy import gseaplot
terms = 'Complement and coagulation cascades Homo sapiens hsa04610'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.show()

In [ ]:
df_dn = pre_mult.res2d.copy()
df_dn[(df_dn["NES"] < 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()

In [ ]:
terms =  'Ribosome Homo sapiens hsa03010'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.show()

In [ ]:
plot_df = pd.concat([df_up, df_dn])

## Filter terms
list_up = df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()
list_dn = df_dn[(df_dn["NES" ]< 0) & (df_dn["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(10)["Term"].tolist()
list_total = list(set(list_up + list_dn))
plot_df = plot_df[plot_df["Term"].isin(list_total)]

## Sort by NES
plot_df = plot_df.sort_values("NES", ascending=False)

## Percentage overlap
plot_df["Overlap %"] = (
    plot_df["Tag %"]
    .str.split("/")
    .apply(lambda x: int(x[0]) / int(x[1]) * 100)
)

# Plot
plt.figure(figsize=(4, 5))

sb.scatterplot(
    data=plot_df,
    x="NES",
    y="Term",
    size="Overlap %",
    hue="FDR q-val",
    palette="RdBu",
    sizes=(50, 400),
    edgecolor="black"
)

plt.xlabel("NES")
plt.ylabel("")
plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    borderaxespad=0
)

plt.tight_layout()
plt.show()

### GOBP

In [ ]:
#Subset 1 comes from T cell singlets vs B cell singlets
subset1 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Xcells")[["names", "logfoldchanges", "pvals_adj"]]
subset1 = subset1.rename(columns={"logfoldchanges":"TvX_lfc", "pvals_adj":"pvals_adj_TvX_lfc"})
subset1 = subset1.set_index("names")

#Subset 2 comes from sampled T cell singlet + sampled B cell (all observed B cell) versus T-B cell doublets. 
subset2 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="sampled_vs_TX_doublets")
subset2 = subset2[["names", "scores", "logfoldchanges", "pvals_adj"]]
subset2 = subset2.set_index("names")

#The dataframes are now joined via indexes
plot_df = subset2.join(subset1).reset_index()

# .. and filtered based on the combined gene list
plot_df = plot_df[plot_df["names"].isin(combined_list)]

# ... and focus on statistical significant differences (cuts the number of dots from 5000+ to around 3000)
plot_df = plot_df[(plot_df["pvals_adj_TvX_lfc"]<0.05) | (plot_df["pvals_adj"]<0.05)]

# .... annotation for staitical significance
plot_df["stat_cat"] = np.where((plot_df["pvals_adj"]<0.05) & (plot_df["pvals_adj_TvX_lfc"]<0.05), "p.adj < 0.05 in both", "Irrellevant")

## Run enrichment (online)
deg = plot_df[(plot_df["pvals_adj"] < 0.05) & (plot_df["logfoldchanges"] > 0)]["names"].tolist()

## A ranked list
rnk = (
    plot_df[["names", "scores"]]
    .dropna()
    .drop_duplicates("names")
    .sort_values("scores", ascending=False)
)


pre_mult = gp.prerank(
    rnk=rnk,    # single ranked list
    gene_sets="GO_Biological_Process_2026",
    method='multilevel',            # use fgsea multilevel p-values
    eps=1e-90,                      # smallest p-value resolved; 0 => machine precision eps (float) – Only used when method='multilevel'. Lower boundary for the estimated p-value; p-values smaller than eps are reported as eps. Set to 0 to estimate p-values as small as machine precision allows. Default: 1e-50.
    threads=4,
    min_size=15,                    #Default gene set size max is 15 (gets rid of many disease GOBP gene sets)
    max_size=500,                   #Defailt gene set size max is 500
    sample_size=101,                #sample_size (int) – Only used when method='multilevel'. Sample size for the multilevel split step of the fgsea algorithm; larger values give more accurate (but slower) tail p-value estimates. Default: 101.
    seed=6,
    outdir=None,
)
pre_mult.res2d.head(2)


In [ ]:
df_up = pre_mult.res2d.copy()
df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()

In [ ]:
df_up[df_up["Term"] == "Regulation of T Cell Activation (GO:0050863)"]["Lead_genes"].tolist()

In [ ]:
from gseapy import gseaplot
terms = 'Microglial Cell Activation (GO:0001774)'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_MGactivation.pdf", format="pdf", dpi=400)
plt.show()

## PROBLEM: EBI considers this term obsolete

In [ ]:
from gseapy import gseaplot
terms = 'Antigen Processing and Presentation of Peptide Antigen via MHC Class II (GO:0002495)'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_AgPresentation.pdf", format="pdf", dpi=400)
plt.show()

## PROBLEM: EBI considers this term obsolete

In [ ]:
from gseapy import gseaplot
terms = 'Regulation of T Cell Activation (GO:0050863)'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_TcellActivation.pdf", format="pdf", dpi=400)
plt.show()

## PROBLEM: EBI considers this term obsolete

In [ ]:
from gseapy import gseaplot
terms = 'Innate Immune Response Activating Cell Surface Receptor Signaling Pathway (GO:0002220)'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_InnateReceptors.pdf", format="pdf", dpi=400)
plt.show()

## PROBLEM: EBI considers this term obsolete

In [ ]:
df_dn = pre_mult.res2d.copy()
df_dn[(df_dn["NES"] < 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()

In [ ]:
terms =  'Translation (GO:0006412)'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_Cytoplasmic Translation.pdf", format="pdf", dpi=400)
plt.show()


#### Adding manually annotated gene sets

In [ ]:
plot_df = pd.concat([df_up, df_dn]).drop_duplicates()

## Filter terms
list_up = df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()
list_dn = df_dn[(df_dn["NES" ]< 0) & (df_dn["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(5)["Term"].tolist()
## Find terms with antigen presentation, apoptosis, caspases, and GC
list_apop = plot_df[plot_df["Term"].str.contains("Apop") |
    plot_df["Term"].str.contains("Caspase") |
    plot_df["Term"].str.contains("Granzyme")].sort_values("FDR q-val")["Term"].tolist()
list_total = list(set(list_up + list_dn+list_apop))

In [ ]:

plot_df = pd.concat([df_up, df_dn])

## Filter terms
plot_df = plot_df[plot_df["Term"].isin(list_total)]

## Sort by NES
plot_df = plot_df.sort_values("NES", ascending=False)

## Percentage overlap
plot_df["Overlap %"] = (
    plot_df["Tag %"]
    .str.split("/")
    .apply(lambda x: int(x[0]) / int(x[1]) * 100)
)

# Plot
plt.figure(figsize=(4, 12))

sb.scatterplot(
    data=plot_df,
    x="NES",
    y="Term",
    size="Overlap %",
    hue="FDR q-val",
    palette="RdBu",
    sizes=(50, 400),
    edgecolor="black"
)

plt.title("GOBP")
plt.xlabel("NES")
plt.ylabel("")
plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    borderaxespad=0
)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_PATHWAYS_GOBP_TMG.pdf", format="pdf", dpi=400)
plt.show()

### GOMF

In [ ]:
#Subset 1 comes from T cell singlets vs B cell singlets
subset1 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Xcells")[["names", "logfoldchanges", "pvals_adj"]]
subset1 = subset1.rename(columns={"logfoldchanges":"TvX_lfc", "pvals_adj":"pvals_adj_TvX_lfc"})
subset1 = subset1.set_index("names")

#Subset 2 comes from sampled T cell singlet + sampled B cell (all observed B cell) versus T-B cell doublets. 
subset2 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="sampled_vs_TX_doublets")
subset2 = subset2[["names", "scores", "logfoldchanges", "pvals_adj"]]
subset2 = subset2.set_index("names")

#The dataframes are now joined via indexes
plot_df = subset2.join(subset1).reset_index()

# .. and filtered based on the combined gene list
plot_df = plot_df[plot_df["names"].isin(combined_list)]

# ... and focus on statistical significant differences (cuts the number of dots from 5000+ to around 3000)
plot_df = plot_df[(plot_df["pvals_adj_TvX_lfc"]<0.05) | (plot_df["pvals_adj"]<0.05)]

# .... annotation for staitical significance
plot_df["stat_cat"] = np.where((plot_df["pvals_adj"]<0.05) & (plot_df["pvals_adj_TvX_lfc"]<0.05), "p.adj < 0.05 in both", "Irrellevant")

## Run enrichment (online)
deg = plot_df[(plot_df["pvals_adj"] < 0.05) & (plot_df["logfoldchanges"] > 0)]["names"].tolist()

## A ranked list
rnk = (
    plot_df[["names", "scores"]]
    .dropna()
    .drop_duplicates("names")
    .sort_values("scores", ascending=False)
)


pre_mult = gp.prerank(
    rnk=rnk,    # single ranked list
    gene_sets="GO_Molecular_Function_2026",
    method='multilevel',            # use fgsea multilevel p-values
    eps=1e-90,                      # smallest p-value resolved; 0 => machine precision eps (float) – Only used when method='multilevel'. Lower boundary for the estimated p-value; p-values smaller than eps are reported as eps. Set to 0 to estimate p-values as small as machine precision allows. Default: 1e-50.
    threads=4,
    min_size=15,                    #Default gene set size max is 15 (gets rid of many disease GOBP gene sets)
    max_size=500,                   #Defailt gene set size max is 500
    sample_size=101,                #sample_size (int) – Only used when method='multilevel'. Sample size for the multilevel split step of the fgsea algorithm; larger values give more accurate (but slower) tail p-value estimates. Default: 101.
    seed=6,
    outdir=None,
)
pre_mult.res2d.head(10)


In [ ]:
df_up = pre_mult.res2d.copy()
df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()

In [ ]:
df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(1)["Lead_genes"].tolist()

In [ ]:
df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(5)

In [ ]:
from gseapy import gseaplot
terms = 'Peptidase Inhibitor Activity (GO:0030414)'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_PATHWAYS_GOMF_TMG_peptideaseInh.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
from gseapy import gseaplot
terms = 'Serine-type Peptidase Activity (GO:0008236)'

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_PATHWAYS_GOMF_TMG_SerineProteases.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
df_dn = pre_mult.res2d.copy()
df_dn[(df_dn["NES"] < 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(20)["Term"].tolist()

In [ ]:
terms =  "MRNA Binding (GO:0003729)"

ax = gseaplot(rank_metric=pre_mult.ranking, term=terms,
         ofname=None, **pre_mult.results[terms])
plt.show()

In [ ]:
plot_df = pd.concat([df_up, df_dn])

## Filter terms
list_up = df_up[(df_up["NES"] > 0) & (df_up["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(5)["Term"].tolist()
list_dn = df_dn[(df_dn["NES" ]< 0) & (df_dn["NOM p-val"] < 0.05)].sort_values("FDR q-val").head(5)["Term"].tolist()
list_total = list(set(list_up + list_dn))
plot_df = plot_df[plot_df["Term"].isin(list_total)]

## Sort by NES
plot_df = plot_df.sort_values("NES", ascending=False)

## Percentage overlap
plot_df["Overlap %"] = (
    plot_df["Tag %"]
    .str.split("/")
    .apply(lambda x: int(x[0]) / int(x[1]) * 100)
)

# Plot
plt.figure(figsize=(2.5, 3))

sb.scatterplot(
    data=plot_df,
    x="NES",
    y="Term",
    size="Overlap %",
    hue="FDR q-val",
    palette="RdBu",
    sizes=(50, 400),
    edgecolor="black"
)

plt.xlabel("NES")
plt.ylabel("")
plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    borderaxespad=0
)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_PATHWAYS_GOMF_TMG.pdf", dpi=400, format="pdf")
plt.show()

# Receptor ligand pairs

In [ ]:
# Retrieving via python
cpdb = pd.read_csv(filepath_or_buffer="../../03_annotation/13_Receptor_ligand_interactions/CellPhoneDB_v5.csv")
cpdb.head(2)

In [ ]:
cpdb["directionality"].value_counts()

In [ ]:
## Split interactors into two columns
cpdb_reduced = cpdb[['interactors']].join(cpdb['interactors'].str.split('-', n=1, expand=True).rename(columns={0:'Receptor', 1:'Ligand'}))

## Make a reduced dataframe
cpdb_reduced = cpdb_reduced.join(cpdb_reduced['Receptor'].str.split('+', n=4, expand=True).rename(columns={0:'Receptor1', 1:'Receptor2', 2:'Receptor3', 3:'Receptor4'}))
cpdb_reduced = cpdb_reduced.join(cpdb_reduced['Ligand'].str.split('+', n=5, expand=True).rename(columns={0:'Ligand1', 1:'Ligand2', 2:'Ligand3', 3:'Ligand4', 4:'Ligand5'}))

## pivot to a longer list of genes belonging to a pair of interactors
long_df = (
    cpdb_reduced
    .set_index("interactors")
    .stack()
    .reset_index()
)

long_df.columns = ["interactors", "interaction_type", "gene"]

## Remove "Receptor" and "Ligand", which were the original genes
long_df = long_df[~long_df["interaction_type"].isin(["Receptor", "Ligand"])]


### Query T cell side

In [ ]:
### 
#Subset 1 comes from T cell singlets vs B cell singlets
subset1 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Xcells")[["names", "logfoldchanges", "pvals_adj"]]
subset1 = subset1.rename(columns={"logfoldchanges":"TvX_lfc", "pvals_adj":"pvals_adj_TvX_lfc"})
subset1 = subset1.set_index("names")

#Subset 2 comes from sampled T cell singlet + sampled B cell (all observed B cell) versus T-B cell doublets. 
subset2 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="sampled_vs_TX_doublets")
subset2 = subset2[["names", "scores", "logfoldchanges", "pvals_adj"]]
subset2 = subset2.set_index("names")

#The dataframes are now joined via indexes
plot_df = subset2.join(subset1).reset_index()

# .. and filtered based on the combined gene list
plot_df = plot_df[plot_df["names"].isin(combined_list)]

# ... and focus on statistical significant differences (cuts the number of dots from 5000+ to around 3000)
plot_df = plot_df[(plot_df["pvals_adj_TvX_lfc"]<0.05) | (plot_df["pvals_adj"]<0.05)]

# .... annotation for staitical significance
plot_df["stat_cat"] = np.where((plot_df["pvals_adj"]<0.05) & (plot_df["pvals_adj_TvX_lfc"]<0.05), "p.adj < 0.05 in both", "Irrellevant")

## Run enrichment (online)
deg = plot_df[(plot_df["pvals_adj"] < 0.05) & (plot_df["logfoldchanges"] > 0) & (plot_df["TvX_lfc"] > 0)]
deg = deg.set_index("names")

## Get a list of differentially expressed genes
deg_list = deg.reset_index()["names"].tolist()

## Filter / subset
deg_df = long_df[long_df["gene"].isin(deg_list)]

## Pivot wider
wide_df = deg_df.pivot_table(
    index="interactors",
    columns="interaction_type",
    values="gene",
    aggfunc=list
)

deg_df

# XY plot - T-cDC, T cell and DC singlets

### Preprocessing (doublet simulation)
Returns **csf_with_doublets**, the final AnnData object that we will use for differential gene expression

In [ ]:
# -------------------------
# 1. Restrict to selected donors with B cell
# -------------------------
obs_df = CSF_data["gex"].obs.copy()

# Make a donor ist of allowed donors
allowed_donors = obs_df[
    obs_df["Recluster_ann2"].isin(["cDC", "T-cDC"])
]["donor"].unique().tolist()

# filter data based on those donors
csf_dat_subset = CSF_data.copy()
mu.pp.filter_obs(
    csf_dat_subset["gex"],
    "donor",
    lambda x: x.isin(allowed_donors)
)


In [ ]:
obs_df["Recluster_ann2"].value_counts()

In [ ]:
# -------------------------
# 2. Donor-aware T+B simulation
# -------------------------
# np.random.seed(41)

gex = csf_dat_subset["gex"]
donors = gex.obs["donor"].unique()

sim_doublets_per_donor = []

for donor in donors:
    sub = gex[gex.obs["donor"] == donor]

    APC_cells = sub.obs.query("Recluster_ann2 == 'cDC'").index
    T_cells = sub.obs.query("Recluster_ann2 == 'T cell'").index

    if len(APC_cells) == 0 or len(T_cells) == 0:
        print(f"⚠️ Donor {donor}: Missing T or cDC. Skipping.")
        continue

    n = min(len(APC_cells), len(T_cells))

    APC_sample_idx = sub.obs.loc[APC_cells].sample(n=n, random_state=39).index
    T_sample_idx = sub.obs.loc[T_cells].sample(n=n, random_state=39).index

    APC_subset = sub[APC_sample_idx].copy()
    T_subset = sub[T_sample_idx].copy()

    # Random 1:1 pairing
    T_idx = np.arange(n)
    APC_idx = np.arange(n)
    np.random.shuffle(T_idx)
    np.random.shuffle(APC_idx)
    pairs = list(zip(T_idx, APC_idx))

    # Build simulated matrix
    X_doublets = sparse.vstack([T_subset.X[t] + APC_subset.X[b] for t, b in pairs])
    sim_adata = ad.AnnData(
        X=X_doublets,
        var=T_subset.var.copy()
    )
    sim_adata.obs["Recluster_ann2"] = "Simulated doublets"
    sim_adata.obs["donor"] = donor
    sim_adata.obs.index = [f"{donor}_simTPB_{i}" for i in range(n)]
    sim_doublets_per_donor.append(sim_adata)

if len(sim_doublets_per_donor) == 0:
    raise ValueError("No simulated doublets created.")

sim_all = ad.concat(sim_doublets_per_donor, join="outer", merge="same")

# -------------------------
# 3. Concatenate observed + simulated
# -------------------------
csf_with_doublets = ad.concat(
    {"Observed": gex, "Simulated doublets": sim_all},
    label="source",
    join="outer",
    merge="same"
)

# Normalize + log1p
sc.pp.normalize_total(csf_with_doublets, target_sum=1e4)
sc.pp.log1p(csf_with_doublets)

In [ ]:
## --------------
## Step 3: Generate a umap for simulated vs observed data
## --------------
# sc.pp.pca(csf_with_doublets)
# sc.pp.neighbors(csf_with_doublets)
# sc.tl.umap(csf_with_doublets)

In [ ]:
## --------------
## Step 4: Report on simulated T-B doublets
## --------------
plt.rcParams["figure.figsize"] = (5, 5)
fig = sc.pl.umap(csf_with_doublets, color=["Recluster_ann2", "source"], legend_fontweight='bold', cmap="RdBu_r", frameon= False , vmin=0, legend_fontoutline=2, legend_loc = "on data", size=20, return_fig = True, show=False)

# # Rasterize all scatter collections
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/XX_simulated_TDCoublets_umap1.pdf", 
    bbox_inches="tight",
    dpi=300
)
plt.show()

## 
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

### Make a list of genes expressed in at least 15% of cells (or interacting cells)

In [ ]:
csf_with_doublets.obs["Recluster_ann2"].value_counts()

In [ ]:
## --------------
## Step 1: Generate lists of expressed genes (>X%) for each of the 3 populations of observed cells T cell, B cell and interacting cells
## --------------


## --- Expression in T cell singlets
adata = csf_with_doublets[csf_with_doublets.obs["Recluster_ann2"] == "T cell"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_1 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 



## --- Expression in B cell singlets
adata = csf_with_doublets[csf_with_doublets.obs["Recluster_ann2"] == "cDC"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_2 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 


## --- Expression in T-cDC
adata = csf_with_doublets[csf_with_doublets.obs["Recluster_ann2"] == "T-cDC"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_3 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 

#The total set of genes
gene_list_10pct_expressed = set(gene_list_10pct_1+gene_list_10pct_2+gene_list_10pct_3)
print(len(gene_list_10pct_expressed))

### T vs cDC cells - differentially expressed genes

In [ ]:
# %%%
# Calculate differentially expressed genes between T cell in CSf and B cell in CSF
# %%% 
sc.tl.rank_genes_groups(csf_with_doublets, groupby="Recluster_ann2", groups=["T cell"], reference="cDC", key_added="T_vs_Xcells")

# %%%% 
# Export all differentially expressed genes
# %%%%
TvBcells_DEG = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Xcells")
TvBcells_DEG.to_csv("../../06_csv_outputs/DEG/3_DoubletAnalysis_DEGs_TMGBAMcells.csv")

# ### This introduces NAs
# sc.tl.filter_rank_genes_groups(
#     csf_with_doublets,
#     min_in_group_fraction=0.2,  #fraction positive in population of interest should be above 0.2.
#     max_out_group_fraction=0.99, #fraction positive !! outside !! the population of interest be less than 0.9
#     min_fold_change=0,      #Since a defaul min_fold_change is enforces, to retain negative logfold changes compare_abs has to be True
#     compare_abs=True,    #Since a defaul min_fold_change is enforces, to retain negative logfold changes compare_abs has to be True
#     key="T_vs_Bcells",
#     key_added="T_vs_Bcells (w. filtering)",
# )

In [ ]:
sc.pl.rank_genes_groups(csf_with_doublets, key="T_vs_Xcells", n_genes=50)

### Simulated vs observed T-B cell - differentially expressed genes

In [ ]:
# %%%
# Calculate differentially expressed genes between random T+B cell vs T-B cell doublets
# %%% 
sc.tl.rank_genes_groups(csf_with_doublets, groupby="Recluster_ann2", groups=["T-cDC"], reference="Simulated doublets", key_added="sampled_vs_TX_doublets")

# %%%% 
# Export all differentially expressed genes
# %%%%
sampledTBcells_DEG = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="sampled_vs_TX_doublets")
TvBcells_DEG.to_csv("../../06_csv_outputs/DEG/3_DoubletAnalysis_DEGs_simulated_vs_observed_TDCcells.csv")

# #Filter it
# ### This introduces NAs
# sc.tl.filter_rank_genes_groups(
#     csf_with_doublets,
#     min_in_group_fraction=0.2,  #fraction positive in population of interest should be above 0.2.
#     max_out_group_fraction=0.99, #fraction positive !! outside !! the population of interest be less than 0.9
#     min_fold_change=0,      #Since a defaul min_fold_change is enforces, to retain negative logfold changes compare_abs has to be True
#     compare_abs=True,    #Since a defaul min_fold_change is enforces, to retain negative logfold changes compare_abs has to be True
#     key="sampled_vs_T_B_doublets",
#     key_added="sampled_vs_T_B_doublets (w. filtering)",
# )


In [ ]:
sc.pl.rank_genes_groups(csf_with_doublets, key="sampled_vs_TX_doublets", n_genes=30)

### Make a combined data frame to plot log2FCs 

In [ ]:
### lists to filter on later - these are differentially expressed genes that passes the significance filter
gene_list1 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Xcells")
gene_list1 = gene_list1[gene_list1["names"].notna()]["names"]

gene_list2 = sc.get.rank_genes_groups_df(csf_with_doublets, group = "T-cDC", key ="sampled_vs_TX_doublets")[["names", "logfoldchanges", "pvals_adj"]]
gene_list2 = gene_list2[gene_list2["names"].notna()]["names"]

combined_list = pd.concat([gene_list1, gene_list2]).drop_duplicates()
print(len(combined_list))
      
### Reduce this gene list to only concern genes that are expressed in at least X pct of T and B cell
combined_list = combined_list[combined_list.isin(gene_list_10pct_expressed)]
print(len(combined_list))

In [ ]:
# %%%%%% 
# Combined the DEGs into 1 plot
# %%%%%

#Subset 1 comes from T cell singlets vs B cell singlets
subset1 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="T_vs_Xcells")[["names", "logfoldchanges", "pvals_adj"]]
subset1 = subset1.rename(columns={"logfoldchanges":"TvX_lfc", "pvals_adj":"pvals_adj_TvX_lfc"})
subset1 = subset1.set_index("names")

#Subset 2 comes from sampled T cell singlet + sampled B cell (all observed B cell) versus T-B cell doublets. 
subset2 = sc.get.rank_genes_groups_df(csf_with_doublets, group = None, key ="sampled_vs_TX_doublets")
subset2 = subset2[["names", "logfoldchanges", "pvals_adj"]]
subset2 = subset2.set_index("names")

#The dataframes are now joined via indexes
plot_df = subset2.join(subset1).reset_index()

# .. and filtered based on the combined gene list
plot_df = plot_df[plot_df["names"].isin(combined_list)]

# ... and focus on statistical significant differences (cuts the number of dots from 5000+ to around 3000)
plot_df = plot_df[(plot_df["pvals_adj_TvX_lfc"]<0.05) | (plot_df["pvals_adj"]<0.05)]

# .... annotation for staitical significance
plot_df["stat_cat"] = np.where((plot_df["pvals_adj"]<0.05) & (plot_df["pvals_adj_TvX_lfc"]<0.05), "p.adj < 0.05 in both", "Irrellevant")

#Save the data for later heatmap
heatmap_TDC = plot_df[plot_df["stat_cat"] == "p.adj < 0.05 in both"]

from adjustText import adjust_text

#Palette
color_dic = {
    "p.adj < 0.05 in both" : "#FF0000",
    "Irrellevant" : "#808080",
}

# Scatter plot
plt.figure(figsize=(6, 5))
# Background points
sb.scatterplot(
    data=plot_df[plot_df["stat_cat"] == "Irrellevant"],
    x="TvX_lfc",
    y="logfoldchanges",
    color="#808080",
    s=30,
    alpha=0.25,
    linewidth=0
)

# Significant points on top
sb.scatterplot(
    data=plot_df[plot_df["stat_cat"] == "p.adj < 0.05 in both"],
    x="TvX_lfc",
    y="logfoldchanges",
    color="#FF0000",
    s=30,
    alpha=1,
    linewidth=0
)
# Identify top genes in T-B cell (highest logfoldchanges)
top_Tcell = heatmap_TDC[(heatmap_TDC["logfoldchanges"] > 0.2) & (heatmap_TDC["TvX_lfc"] > 5)]
bottom_Tcell = heatmap_TDC[(heatmap_TDC["logfoldchanges"] < -0.2) & (heatmap_TDC["TvX_lfc"] > 5)]
top_y = plot_df[plot_df["stat_cat"]== "p.adj < 0.05 in both"].nlargest(30, "logfoldchanges")

# Add reference lines
plt.axhline(y=0, color="black", alpha=0.75, linestyle='--', linewidth=1)
plt.axvline(x=0, color="black", alpha=0.75, linestyle='--', linewidth=1)

# Annotate the top_y genes
texts = []
for _, row in bottom_Tcell.iterrows():
    texts.append(
        plt.text(row["TvX_lfc"], row["logfoldchanges"], row["names"],
                 fontsize=8, color="black")
    )

# Optional: automatically adjust text to prevent overlap (ggrepel-style)
adjust_text(texts, arrowprops=dict(arrowstyle="-", color="gray", lw=0.5))

# Labels and formatting
plt.ylabel("<-- Simulated T+X cell singlets\nvs\nobserved T-X -->")
plt.xlabel("<-- cDC   vs    T cell -->")

sb.despine()
plt.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_TPBcells_doublets_DEGs_simulated.pdf")
plt.show()

In [ ]:
heatmap_TDC[(heatmap_TDC["logfoldchanges"] > 0.2) & (heatmap_TDC["TvX_lfc"] > 5)]

In [ ]:
from brokenaxes import brokenaxes
from adjustText import adjust_text

# Create figure first
fig = plt.figure(figsize=(6, 5))

# Create broken axes
bax = brokenaxes(
    xlims=((-34, -27), (-15, 15), (25, 32)),
    hspace=0.05,
    fig=fig
)

# Scatter plot
colors = plot_df["stat_cat"].map(color_dic)

# Background
sub = plot_df[plot_df["stat_cat"] == "Irrellevant"]
bax.scatter(
    sub["TvX_lfc"],
    sub["logfoldchanges"],
    color="#808080",
    s=30,
    alpha=0.25
)

# Foreground
sub = plot_df[plot_df["stat_cat"] == "p.adj < 0.05 in both"]
bax.scatter(
    sub["TvX_lfc"],
    sub["logfoldchanges"],
    color="#FF0000",
    s=30,
    alpha=1
)

# ---- Identify genes ----
top_y1 = plot_df[(plot_df["stat_cat"] == "p.adj < 0.05 in both") & (plot_df["TvX_lfc"] > 0.5)].nlargest(30, "logfoldchanges")["names"]
top_y2 = plot_df[(plot_df["stat_cat"] == "p.adj < 0.05 in both") & (plot_df["TvX_lfc"] < -0.5)].nlargest(10, "logfoldchanges")["names"]
top_x1 = plot_df[(plot_df["stat_cat"] == "p.adj < 0.05 in both") & (plot_df["TvX_lfc"] > 0.5)].nsmallest(5, "logfoldchanges")["names"]
top_x2 = plot_df[(plot_df["stat_cat"] == "p.adj < 0.05 in both") & (plot_df["TvX_lfc"] < -0.5)].nsmallest(5, "logfoldchanges")["names"]
highlight_genes = set(top_y1).union(top_y2).union(top_x1).union(top_x2)

# Filter dataframe ONCE (important fix)
label_df = plot_df[plot_df["names"].isin(highlight_genes)]

# Reference lines
for ax in bax.axs:
    ax.axhline(y=0, color="black", alpha=0.75, linestyle="--", linewidth=1)
    ax.axvline(x=0, color="black", alpha=0.75, linestyle="--", linewidth=1)

# ---- Annotate ----
texts_by_ax = {ax: [] for ax in bax.axs}

for _, row in label_df.iterrows():

    for ax in bax.axs:

        xmin, xmax = ax.get_xlim()
        ymin, ymax = ax.get_ylim()

        if (
            xmin <= row["TvX_lfc"] <= xmax
            and ymin <= row["logfoldchanges"] <= ymax
        ):

            txt = ax.text(
                row["TvX_lfc"],
                row["logfoldchanges"],
                row["names"],
                fontsize=8,
                color="black"
            )

            texts_by_ax[ax].append(txt)
            break
        
# Labels
bax.set_ylabel(
    "<-- Simulated T+X singlets\nvs\nobserved T-X cells -->"
)
bax.set_xlabel("<-- cDC   vs    T cell -->")


plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/XX_TDCcells_doublets_DEGs_simulated.pdf")
plt.show()

In [ ]:
Tgenes = plot_df[(plot_df["TvX_lfc"]>5) & (plot_df["logfoldchanges"]>0)].sort_values("pvals_adj", ascending=True).head(20)["names"]
TgenesDown = plot_df[(plot_df["TvX_lfc"]>5) & (plot_df["logfoldchanges"]<0)].sort_values("pvals_adj", ascending=True).head(20)["names"]
Xgenes = plot_df[(plot_df["TvX_lfc"]<5)& (plot_df["logfoldchanges"]>0)].sort_values("pvals_adj", ascending=True).head(20)["names"]
XgenesDown = plot_df[(plot_df["TvX_lfc"]<5)& (plot_df["logfoldchanges"]<0)].sort_values("pvals_adj", ascending=True).head(20)["names"]

In [ ]:
TB_data = csf_with_doublets[csf_with_doublets.obs["Recluster_ann2"].isin(["T cell", "cDC", "T-cDC", "Simulated doublets"])].copy()

# Define desired category order
desired_order = ["T cell", "cDC", "Simulated doublets", "T-cDC"]

# Reorder categories
TB_data.obs["Recluster_ann2"] = TB_data.obs["Recluster_ann2"].cat.reorder_categories(
    desired_order, ordered=True
)

markers = {
     "T cell": Tgenes, 
     "cDC" : Xgenes, 
     "T down" : TgenesDown,
    "cDC down" : XgenesDown,
    "Other" : ["CD8A", "CD8B"]
}

dp = sc.pl.dotplot(
   TB_data,
   markers,
    groupby="Recluster_ann2",
    standard_scale="var",
    cmap="RdBu_r",
    return_fig=True,
    show=False,
)

dp.add_totals()
dp.style(dot_edge_color="black", dot_edge_lw=0.5)

# Build the figure
# dp.make_figure()
dp.show()

# Save
dp.fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/94_CSF_reclustered2_dotplot.pdf",
    bbox_inches="tight",
    dpi=400,
)

In [ ]:
# Create figure and axis first
fig, ax = plt.subplots(figsize=(14, 2))

dp = sc.pl.matrixplot(
    TB_data,
    markers,
    groupby="Recluster_ann2",
    standard_scale="var",
    cmap="RdBu_r",
    ax=ax,
    return_fig=True,
    show=False,
)

dp.add_totals().style(edge_color='black').show()

fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/94_CSF_reclustered2_matrixplot_TMG.pdf",
    bbox_inches="tight",
    dpi=400,
)

## Save DEGs

In [ ]:
heatmap_TMG.head(20)

# Overflow

# Virus-specific clonotypes with sister cells in PB & CSF - Pt20

In [ ]:
## Make a common list
TB_total_list = TPB_clone_list_all + T_B_clone_list_all

cd8data["gex"].obs["TB_highlight"] = pd.Categorical(np.where(cd8data["gex"].obs["clone_handle"].isin(TB_total_list), "T-B clones", "other"))
print("number of unique TB clones")
cd8data["gex"].obs[cd8data["gex"].obs["TB_highlight"] == "T-B clones"][["donor", "clone_handle"]].drop_duplicates()["donor"].value_counts()

In [ ]:
## Number of EBV-specific clones
TB_total_list = TPB_clone_list_all + T_B_clone_list_all

print("Number of EBV-specific clones") 
cd8data["gex"].obs[cd8data["gex"].obs["Virus_extended"] == "EBV"][["donor", "clone_handle"]].drop_duplicates()["donor"].value_counts()

In [ ]:
## Annotate where all T-B clones are in patient 20
cd8data["gex"].obs["TB_highlight"] = pd.Categorical(np.where((cd8data["gex"].obs["donor"] == "Pt20") & (cd8data["gex"].obs["clone_handle"].isin(TB_total_list)), "Pt20 T-B clones", 
                                                    np.where(cd8data["gex"].obs["donor"] == "Pt20", "Pt20 others", "Other donors")), 
                                                   ordered=True, 
                                                   categories = ["Pt20 T-B clones", 
                                                                "Pt20 others", 
                                                                "Other donors"])

In [ ]:
obs = cd8data["gex"].obs

# Get UMAP coordinates
umap = cd8data["gex"].obsm["X_umap"]

# Define masks
mask_other = obs["TB_highlight"] == "Other donors"
mask_pt20 = obs["TB_highlight"] == "Pt20 others"
mask_tb = obs["TB_highlight"] == "Pt20 T-B clones"

palette_highlight = {
    "Pt20 others" : "#A1A1A1", 
    "Pt20 T-B clones" : "#000000", 
    "Other donors" : "#D3D3D3", 
}


fig, ax = plt.subplots(figsize=(6,6))

# Layer 1: Other donors (bottom)
ax.scatter(
    umap[mask_other, 0],
    umap[mask_other, 1],
    s=1,
    c=palette_highlight["Other donors"],
    linewidths=0,
    rasterized=True
)

# Layer 2: Pt20 others
ax.scatter(
    umap[mask_pt20, 0],
    umap[mask_pt20, 1],
    s=5,
    c=palette_highlight["Pt20 others"],
    linewidths=0,
    rasterized=True
)

# Layer 3: Pt20 T-B clones (top)
ax.scatter(
    umap[mask_tb, 0],
    umap[mask_tb, 1],
    s=20,
    c=palette_highlight["Pt20 T-B clones"],
    linewidths=0,
    rasterized=True
)

ax.set_axis_off()

plt.tight_layout()

fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/XX_Highlight_CD20_TBclones.pdf",
    bbox_inches="tight",
    dpi=300
)

plt.show()

In [ ]:
## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["donor"].isin(["Pt20"])]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
Donor_all = obs_df[["L3_ann"]].value_counts().reset_index()
Donor_all["source"] = "Patient 20 CD8 T cell"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "EBV"]
obs_df = obs_df[obs_df["donor"].isin(["Pt20"])]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
EBV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
EBV_agspf["source"] = "EBV-specific clones"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "CMV"]
obs_df = obs_df[obs_df["donor"].isin(["Pt20"])]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
CMV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
CMV_agspf["source"] = "CMV-specific clones"

## Get the FLU-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "FLU-A"]
obs_df = obs_df[obs_df["donor"].isin(["Pt20"])]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
FLU_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
FLU_agspf["source"] = "FLU-A-specific clones"

# ## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(TB_total_list)]
obs_df = obs_df[obs_df["donor"].isin(["Pt20"])]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_all = obs_df[["L3_ann"]].value_counts().reset_index()
TB_all["source"] = "T-B clones"

## Get the clonotypes of interest (T_B_clone_list_agsp)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(T_B_clone_list_agsp)]
obs_df = obs_df[obs_df["donor"].isin(["Pt20"])]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
TB_agspf["source"] = "EBV-specific T-B clones"

## Concat, pivot and transform
count_df = pd.concat([Donor_all, CMV_agspf, FLU_agspf, EBV_agspf, TB_all, TB_agspf])
count_df = count_df.pivot(index = "source", columns = "L3_ann", values="count")
print("Number of cells for each category")
print(count_df.sum(axis=1))

#Transform into percentages
count_df = count_df.div(count_df.sum(axis=1), axis=0)*100


cluster_color_dict = {
    # --- Pastel colors / TN ---
    "0":  "#dbe9f6",  # very light blue
    "1":  "#c7e9c0",  # pastel green
    "2":  "#e2d9f3",  # light lavender
    "11": "#fdd0a2",  # pastel peach

    # --- TCMs ---
    "7":  "#74c476",  # green
    "13": "#31a354",  # dark green
    "21": "#3BFD16",  # neon

    # --- TEM ---
    "4":  "#fdae6b",  # light orange
    "8":  "#fd8d3c",  # orange

    # --- TEMRA ---
    "9":  "#fb6a4a",  # soft red
    "10": "#8B0000",  # dark red

    # --- TE ---
    "6":  "#C00000",  # red
    "12": "#843c39",  # brown red

    # --- Activated ---
    "17": "#f768a1",  # pink
    "19": "#c51b8a",  # 
    "20": "#FF1B76",  # 
    "22": "#d9d9d9",  # 

    # --- Gamma delta ---
    "15": "#737373",  # Dark grey

    # --- CSF blue / purple-blue gradient ---
    "3":  "#6baed6",  # light blue
    "5":  "#3182bd",  # medium blue
    "14": "#9e9ac8",  # light purple
    "16": "#756bb1",  # purple
    "18": "#1f4fbf",  # dark blue

}

# Annotate a dataframe that contains L3_ann too
L3ann_df = cd8data["gex"].obs[["leiden_1.8", "L3_ann"]].drop_duplicates().copy()
L3ann_df["cluster_colors"] = L3ann_df["leiden_1.8"].map(cluster_color_dict)

## Create L3_ann_dict
clusters = L3ann_df["L3_ann"]
colors = L3ann_df["cluster_colors"]
L3_colors = dict(zip(clusters, colors))

#plot distribution changes
count_df = count_df.reset_index()
count_df["source"] = pd.Categorical(count_df["source"], ordered=True, categories = ["Patient 20 CD8 T cell",  "CMV-specific clones", "FLU-A-specific clones", "T-B clones", "EBV-specific clones", "EBV-specific T-B clones"])

# Sort and keep source as index
count_df = count_df.sort_values("source").set_index("source")

fig, ax = plt.subplots(figsize=(3,3))

count_df.plot(
    kind="bar",
    stacked=True,
    color=L3_colors,
    edgecolor="white",
    linewidth=0.2,
    width=0.80,
    ax=ax
)

ax.set_ylim(0,100)
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
# ax.set_xticklabels([])

## export the legend once, and remove it again
# plt.legend(title="L3_ann", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.get_legend().remove()

ax.grid(False)

plt.title("Of PB CD8 T cell (Pt20)")
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/97_Phenotype_of_all_clones_outside_T-B.pdf")
plt.show()

In [ ]:
## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["donor"].isin(["Pt20"])]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
Donor_all = obs_df[["L3_ann"]].value_counts().reset_index()
Donor_all["source"] = "Patient 20 CD8 T cell"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "EBV"]
obs_df = obs_df[obs_df["donor"].isin(["Pt20"])]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
EBV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
EBV_agspf["source"] = "EBV-specific clones"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "CMV"]
obs_df = obs_df[obs_df["donor"].isin(["Pt20"])]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
CMV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
CMV_agspf["source"] = "CMV-specific clones"

## Get the FLU-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "FLU-A"]
obs_df = obs_df[obs_df["donor"].isin(["Pt20"])]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
FLU_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
FLU_agspf["source"] = "FLU-A-specific clones"

# ## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(TB_total_list)]
obs_df = obs_df[obs_df["donor"].isin(["Pt20"])]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_all = obs_df[["L3_ann"]].value_counts().reset_index()
TB_all["source"] = "T-B clones"

## Get the clonotypes of interest (T_B_clone_list_agsp)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(T_B_clone_list_agsp)]
obs_df = obs_df[obs_df["donor"].isin(["Pt20"])]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
TB_agspf["source"] = "EBV-specific T-B clones"

## Concat, pivot and transform
count_df = pd.concat([Donor_all, CMV_agspf, FLU_agspf, EBV_agspf, TB_all, TB_agspf])
count_df = count_df.pivot(index = "source", columns = "L3_ann", values="count")
print("Number of cells for each category")
print(count_df.sum(axis=1))

#Transform into percentages
count_df = count_df.div(count_df.sum(axis=1), axis=0)*100


cluster_color_dict = {
    # --- Pastel colors / TN ---
    "0":  "#dbe9f6",  # very light blue
    "1":  "#c7e9c0",  # pastel green
    "2":  "#e2d9f3",  # light lavender
    "11": "#fdd0a2",  # pastel peach

    # --- TCMs ---
    "7":  "#74c476",  # green
    "13": "#31a354",  # dark green
    "21": "#3BFD16",  # neon

    # --- TEM ---
    "4":  "#fdae6b",  # light orange
    "8":  "#fd8d3c",  # orange

    # --- TEMRA ---
    "9":  "#fb6a4a",  # soft red
    "10": "#8B0000",  # dark red

    # --- TE ---
    "6":  "#C00000",  # red
    "12": "#843c39",  # brown red

    # --- Activated ---
    "17": "#f768a1",  # pink
    "19": "#c51b8a",  # 
    "20": "#FF1B76",  # 
    "22": "#d9d9d9",  # 

    # --- Gamma delta ---
    "15": "#737373",  # Dark grey

    # --- CSF blue / purple-blue gradient ---
    "3":  "#6baed6",  # light blue
    "5":  "#3182bd",  # medium blue
    "14": "#9e9ac8",  # light purple
    "16": "#756bb1",  # purple
    "18": "#1f4fbf",  # dark blue

}

# Annotate a dataframe that contains L3_ann too
L3ann_df = cd8data["gex"].obs[["leiden_1.8", "L3_ann"]].drop_duplicates().copy()
L3ann_df["cluster_colors"] = L3ann_df["leiden_1.8"].map(cluster_color_dict)

## Create L3_ann_dict
clusters = L3ann_df["L3_ann"]
colors = L3ann_df["cluster_colors"]
L3_colors = dict(zip(clusters, colors))

#plot distribution changes
count_df = count_df.reset_index()
count_df["source"] = pd.Categorical(count_df["source"], ordered=True, categories = ["Patient 20 CD8 T cell",  "CMV-specific clones", "FLU-A-specific clones", "T-B clones", "EBV-specific clones", "EBV-specific T-B clones"])

# Sort and keep source as index
count_df = count_df.sort_values("source").set_index("source")

fig, ax = plt.subplots(figsize=(3,3))

count_df.plot(
    kind="bar",
    stacked=True,
    color=L3_colors,
    edgecolor="white",
    linewidth=0.2,
    width=0.80,
    ax=ax
)

ax.set_ylim(0,100)
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
# ax.set_xticklabels([])

## export the legend once, and remove it again
# plt.legend(title="L3_ann", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.get_legend().remove()

ax.grid(False)

plt.title("Of CSF CD8 T cell (Pt20)")
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/97_Phenotype_of_all_clones_outside_T-B_CSF.pdf")
plt.show()

# Virus-specific clonotypes with sister cells in PB & CSF - Pt17

In [ ]:
## Annotate where all T-B clones are in patient 20
cd8data["gex"].obs["TB_highlight"] = pd.Categorical(np.where((cd8data["gex"].obs["donor"] == "Pt17") & (cd8data["gex"].obs["clone_handle"].isin(TB_total_list)), "Pt17 T-B clones", 
                                                    np.where(cd8data["gex"].obs["donor"] == "Pt17", "Pt17 others", "Other donors")), 
                                                   ordered=True, 
                                                   categories = ["Pt17 T-B clones", 
                                                                "Pt17 others", 
                                                                "Other donors"])

In [ ]:
obs = cd8data["gex"].obs

# Get UMAP coordinates
umap = cd8data["gex"].obsm["X_umap"]

# Define masks
mask_other = obs["TB_highlight"] == "Other donors"
mask_Pt17 = obs["TB_highlight"] == "Pt17 others"
mask_tb = obs["TB_highlight"] == "Pt17 T-B clones"

palette_highlight = {
    "Pt17 others" : "#A1A1A1", 
    "Pt17 T-B clones" : "#000000", 
    "Other donors" : "#D3D3D3", 
}


fig, ax = plt.subplots(figsize=(6,6))

# Layer 1: Other donors (bottom)
ax.scatter(
    umap[mask_other, 0],
    umap[mask_other, 1],
    s=1,
    c=palette_highlight["Other donors"],
    linewidths=0,
    rasterized=True
)

# Layer 2: Pt17 others
ax.scatter(
    umap[mask_Pt17, 0],
    umap[mask_Pt17, 1],
    s=5,
    c=palette_highlight["Pt17 others"],
    linewidths=0,
    rasterized=True
)

# Layer 3: Pt17 T-B clones (top)
ax.scatter(
    umap[mask_tb, 0],
    umap[mask_tb, 1],
    s=20,
    c=palette_highlight["Pt17 T-B clones"],
    linewidths=0,
    rasterized=True
)

ax.set_axis_off()

plt.tight_layout()

fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/XX_Highlight_CD20_TBclones.pdf",
    bbox_inches="tight",
    dpi=300
)

plt.show()

In [ ]:
## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["donor"].isin(["Pt17"])]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
Donor_all = obs_df[["L3_ann"]].value_counts().reset_index()
Donor_all["source"] = "Patient 20 CD8 T cell"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "EBV"]
obs_df = obs_df[obs_df["donor"].isin(["Pt17"])]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
EBV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
EBV_agspf["source"] = "EBV-specific clones"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "CMV"]
obs_df = obs_df[obs_df["donor"].isin(["Pt17"])]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
CMV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
CMV_agspf["source"] = "CMV-specific clones"

## Get the FLU-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "FLU-A"]
obs_df = obs_df[obs_df["donor"].isin(["Pt17"])]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
FLU_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
FLU_agspf["source"] = "FLU-A-specific clones"

# ## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(TB_total_list)]
obs_df = obs_df[obs_df["donor"].isin(["Pt17"])]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_all = obs_df[["L3_ann"]].value_counts().reset_index()
TB_all["source"] = "T-B clones"

## Get the clonotypes of interest (T_B_clone_list_agsp)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(T_B_clone_list_agsp)]
obs_df = obs_df[obs_df["donor"].isin(["Pt17"])]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
TB_agspf["source"] = "EBV-specific T-B clones"

## Concat, pivot and transform
count_df = pd.concat([Donor_all, CMV_agspf, FLU_agspf, EBV_agspf, TB_all, TB_agspf])
count_df = count_df.pivot(index = "source", columns = "L3_ann", values="count")
print("Number of cells for each category")
print(count_df.sum(axis=1))

#Transform into percentages
count_df = count_df.div(count_df.sum(axis=1), axis=0)*100


cluster_color_dict = {
    # --- Pastel colors / TN ---
    "0":  "#dbe9f6",  # very light blue
    "1":  "#c7e9c0",  # pastel green
    "2":  "#e2d9f3",  # light lavender
    "11": "#fdd0a2",  # pastel peach

    # --- TCMs ---
    "7":  "#74c476",  # green
    "13": "#31a354",  # dark green
    "21": "#3BFD16",  # neon

    # --- TEM ---
    "4":  "#fdae6b",  # light orange
    "8":  "#fd8d3c",  # orange

    # --- TEMRA ---
    "9":  "#fb6a4a",  # soft red
    "10": "#8B0000",  # dark red

    # --- TE ---
    "6":  "#C00000",  # red
    "12": "#843c39",  # brown red

    # --- Activated ---
    "17": "#f768a1",  # pink
    "19": "#c51b8a",  # 
    "20": "#FF1B76",  # 
    "22": "#d9d9d9",  # 

    # --- Gamma delta ---
    "15": "#737373",  # Dark grey

    # --- CSF blue / purple-blue gradient ---
    "3":  "#6baed6",  # light blue
    "5":  "#3182bd",  # medium blue
    "14": "#9e9ac8",  # light purple
    "16": "#756bb1",  # purple
    "18": "#1f4fbf",  # dark blue

}

# Annotate a dataframe that contains L3_ann too
L3ann_df = cd8data["gex"].obs[["leiden_1.8", "L3_ann"]].drop_duplicates().copy()
L3ann_df["cluster_colors"] = L3ann_df["leiden_1.8"].map(cluster_color_dict)

## Create L3_ann_dict
clusters = L3ann_df["L3_ann"]
colors = L3ann_df["cluster_colors"]
L3_colors = dict(zip(clusters, colors))

#plot distribution changes
count_df = count_df.reset_index()
count_df["source"] = pd.Categorical(count_df["source"], ordered=True, categories = ["Patient 20 CD8 T cell",  "CMV-specific clones", "FLU-A-specific clones", "T-B clones", "EBV-specific clones", "EBV-specific T-B clones"])

# Sort and keep source as index
count_df = count_df.sort_values("source").set_index("source")

fig, ax = plt.subplots(figsize=(3,3))

count_df.plot(
    kind="bar",
    stacked=True,
    color=L3_colors,
    edgecolor="white",
    linewidth=0.2,
    width=0.80,
    ax=ax
)

ax.set_ylim(0,100)
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
# ax.set_xticklabels([])

## export the legend once, and remove it again
# plt.legend(title="L3_ann", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.get_legend().remove()

ax.grid(False)

plt.title("Of PB CD8 T cell (Pt17)")
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/97_Phenotype_of_all_clones_outside_T-B.pdf")
plt.show()

In [ ]:
## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["donor"].isin(["Pt17"])]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
Donor_all = obs_df[["L3_ann"]].value_counts().reset_index()
Donor_all["source"] = "Patient 20 CD8 T cell"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "EBV"]
obs_df = obs_df[obs_df["donor"].isin(["Pt17"])]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
EBV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
EBV_agspf["source"] = "EBV-specific clones"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "CMV"]
obs_df = obs_df[obs_df["donor"].isin(["Pt17"])]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
CMV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
CMV_agspf["source"] = "CMV-specific clones"

## Get the FLU-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "FLU-A"]
obs_df = obs_df[obs_df["donor"].isin(["Pt17"])]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
FLU_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
FLU_agspf["source"] = "FLU-A-specific clones"

# ## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(TB_total_list)]
obs_df = obs_df[obs_df["donor"].isin(["Pt17"])]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_all = obs_df[["L3_ann"]].value_counts().reset_index()
TB_all["source"] = "T-B clones"

## Get the clonotypes of interest (T_B_clone_list_agsp)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(T_B_clone_list_agsp)]
obs_df = obs_df[obs_df["donor"].isin(["Pt17"])]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
TB_agspf["source"] = "EBV-specific T-B clones"

## Concat, pivot and transform
count_df = pd.concat([Donor_all, CMV_agspf, FLU_agspf, EBV_agspf, TB_all, TB_agspf])
count_df = count_df.pivot(index = "source", columns = "L3_ann", values="count")
print("Number of cells for each category")
print(count_df.sum(axis=1))

#Transform into percentages
count_df = count_df.div(count_df.sum(axis=1), axis=0)*100

cluster_color_dict = {
    # --- Pastel colors / TN ---
    "0":  "#dbe9f6",  # very light blue
    "1":  "#c7e9c0",  # pastel green
    "2":  "#e2d9f3",  # light lavender
    "11": "#fdd0a2",  # pastel peach

    # --- TCMs ---
    "7":  "#74c476",  # green
    "13": "#31a354",  # dark green
    "21": "#3BFD16",  # neon

    # --- TEM ---
    "4":  "#fdae6b",  # light orange
    "8":  "#fd8d3c",  # orange

    # --- TEMRA ---
    "9":  "#fb6a4a",  # soft red
    "10": "#8B0000",  # dark red

    # --- TE ---
    "6":  "#C00000",  # red
    "12": "#843c39",  # brown red

    # --- Activated ---
    "17": "#f768a1",  # pink
    "19": "#c51b8a",  # 
    "20": "#FF1B76",  # 
    "22": "#d9d9d9",  # 

    # --- Gamma delta ---
    "15": "#737373",  # Dark grey

    # --- CSF blue / purple-blue gradient ---
    "3":  "#6baed6",  # light blue
    "5":  "#3182bd",  # medium blue
    "14": "#9e9ac8",  # light purple
    "16": "#756bb1",  # purple
    "18": "#1f4fbf",  # dark blue

}

# Annotate a dataframe that contains L3_ann too
L3ann_df = cd8data["gex"].obs[["leiden_1.8", "L3_ann"]].drop_duplicates().copy()
L3ann_df["cluster_colors"] = L3ann_df["leiden_1.8"].map(cluster_color_dict)

## Create L3_ann_dict
clusters = L3ann_df["L3_ann"]
colors = L3ann_df["cluster_colors"]
L3_colors = dict(zip(clusters, colors))

#plot distribution changes
count_df = count_df.reset_index()
count_df["source"] = pd.Categorical(count_df["source"], ordered=True, categories = ["Patient 20 CD8 T cell",  "CMV-specific clones", "FLU-A-specific clones", "T-B clones", "EBV-specific clones", "EBV-specific T-B clones"])

# Sort and keep source as index
count_df = count_df.sort_values("source").set_index("source")

fig, ax = plt.subplots(figsize=(3,3))

count_df.plot(
    kind="bar",
    stacked=True,
    color=L3_colors,
    edgecolor="white",
    linewidth=0.2,
    width=0.80,
    ax=ax
)

ax.set_ylim(0,100)
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
# ax.set_xticklabels([])

## export the legend once, and remove it again
# plt.legend(title="L3_ann", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.get_legend().remove()

ax.grid(False)

plt.title("Of CSF CD8 T cell (Pt17)")
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/97_Phenotype_of_all_clones_outside_T-B_CSF.pdf")
plt.show()

# Virus-specific clonotypes with sister cells in PB & CSF - Pt12

In [ ]:
## Annotate where all T-B clones are in patient 20
cd8data["gex"].obs["TB_highlight"] = pd.Categorical(np.where((cd8data["gex"].obs["donor"] == "Pt12") & (cd8data["gex"].obs["clone_handle"].isin(TB_total_list)), "Pt12 T-B clones", 
                                                    np.where(cd8data["gex"].obs["donor"] == "Pt12", "Pt12 others", "Other donors")), 
                                                   ordered=True, 
                                                   categories = ["Pt12 T-B clones", 
                                                                "Pt12 others", 
                                                                "Other donors"])

In [ ]:
obs = cd8data["gex"].obs

# Get UMAP coordinates
umap = cd8data["gex"].obsm["X_umap"]

# Define masks
mask_other = obs["TB_highlight"] == "Other donors"
mask_Pt12 = obs["TB_highlight"] == "Pt12 others"
mask_tb = obs["TB_highlight"] == "Pt12 T-B clones"

palette_highlight = {
    "Pt12 others" : "#A1A1A1", 
    "Pt12 T-B clones" : "#000000", 
    "Other donors" : "#D3D3D3", 
}


fig, ax = plt.subplots(figsize=(6,6))

# Layer 1: Other donors (bottom)
ax.scatter(
    umap[mask_other, 0],
    umap[mask_other, 1],
    s=1,
    c=palette_highlight["Other donors"],
    linewidths=0,
    rasterized=True
)

# Layer 2: Pt12 others
ax.scatter(
    umap[mask_Pt12, 0],
    umap[mask_Pt12, 1],
    s=5,
    c=palette_highlight["Pt12 others"],
    linewidths=0,
    rasterized=True
)

# Layer 3: Pt12 T-B clones (top)
ax.scatter(
    umap[mask_tb, 0],
    umap[mask_tb, 1],
    s=20,
    c=palette_highlight["Pt12 T-B clones"],
    linewidths=0,
    rasterized=True
)

ax.set_axis_off()

plt.tight_layout()

fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/XX_Highlight_CD20_TBclones.pdf",
    bbox_inches="tight",
    dpi=300
)

plt.show()

In [ ]:
## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["donor"].isin(["Pt12"])]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
Donor_all = obs_df[["L3_ann"]].value_counts().reset_index()
Donor_all["source"] = "Patient 12 CD8 T cell"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "EBV"]
obs_df = obs_df[obs_df["donor"].isin(["Pt12"])]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
EBV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
EBV_agspf["source"] = "EBV-specific clones"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "CMV"]
obs_df = obs_df[obs_df["donor"].isin(["Pt12"])]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
CMV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
CMV_agspf["source"] = "CMV-specific clones"

## Get the FLU-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "FLU-A"]
obs_df = obs_df[obs_df["donor"].isin(["Pt12"])]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
FLU_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
FLU_agspf["source"] = "FLU-A-specific clones"

# ## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(TB_total_list)]
obs_df = obs_df[obs_df["donor"].isin(["Pt12"])]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_all = obs_df[["L3_ann"]].value_counts().reset_index()
TB_all["source"] = "T-B clones"

## Get the clonotypes of interest (T_B_clone_list_agsp)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(T_B_clone_list_agsp)]
obs_df = obs_df[obs_df["donor"].isin(["Pt12"])]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
TB_agspf["source"] = "EBV-specific T-B clones"

## Concat, pivot and transform
count_df = pd.concat([Donor_all, CMV_agspf, FLU_agspf, EBV_agspf, TB_all, TB_agspf])
count_df = count_df.pivot(index = "source", columns = "L3_ann", values="count")
print("Number of cells for each category")
print(count_df.sum(axis=1))

#Transform into percentages
count_df = count_df.div(count_df.sum(axis=1), axis=0)*100


cluster_color_dict = {
    # --- Pastel colors / TN ---
    "0":  "#dbe9f6",  # very light blue
    "1":  "#c7e9c0",  # pastel green
    "2":  "#e2d9f3",  # light lavender
    "11": "#fdd0a2",  # pastel peach

    # --- TCMs ---
    "7":  "#74c476",  # green
    "13": "#31a354",  # dark green
    "21": "#3BFD16",  # neon

    # --- TEM ---
    "4":  "#fdae6b",  # light orange
    "8":  "#fd8d3c",  # orange

    # --- TEMRA ---
    "9":  "#fb6a4a",  # soft red
    "10": "#8B0000",  # dark red

    # --- TE ---
    "6":  "#C00000",  # red
    "12": "#843c39",  # brown red

    # --- Activated ---
    "17": "#f768a1",  # pink
    "19": "#c51b8a",  # 
    "20": "#FF1B76",  # 
    "22": "#d9d9d9",  # 

    # --- Gamma delta ---
    "15": "#737373",  # Dark grey

    # --- CSF blue / purple-blue gradient ---
    "3":  "#6baed6",  # light blue
    "5":  "#3182bd",  # medium blue
    "14": "#9e9ac8",  # light purple
    "16": "#756bb1",  # purple
    "18": "#1f4fbf",  # dark blue

}

# Annotate a dataframe that contains L3_ann too
L3ann_df = cd8data["gex"].obs[["leiden_1.8", "L3_ann"]].drop_duplicates().copy()
L3ann_df["cluster_colors"] = L3ann_df["leiden_1.8"].map(cluster_color_dict)

## Create L3_ann_dict
clusters = L3ann_df["L3_ann"]
colors = L3ann_df["cluster_colors"]
L3_colors = dict(zip(clusters, colors))

#plot distribution changes
count_df = count_df.reset_index()
count_df["source"] = pd.Categorical(count_df["source"], ordered=True, categories = ["Patient 12 CD8 T cell",  "CMV-specific clones", "FLU-A-specific clones", "T-B clones", "EBV-specific clones", "EBV-specific T-B clones"])

# Sort and keep source as index
count_df = count_df.sort_values("source").set_index("source")

fig, ax = plt.subplots(figsize=(3,3))

count_df.plot(
    kind="bar",
    stacked=True,
    color=L3_colors,
    edgecolor="white",
    linewidth=0.2,
    width=0.80,
    ax=ax
)

ax.set_ylim(0,100)
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
# ax.set_xticklabels([])

## export the legend once, and remove it again
# plt.legend(title="L3_ann", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.get_legend().remove()

ax.grid(False)

plt.title("Of PB CD8 T cell (Pt12)")
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/97_Phenotype_of_all_clones_outside_T-B.pdf")
plt.show()

In [ ]:
## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["donor"].isin(["Pt12"])]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
Donor_all = obs_df[["L3_ann"]].value_counts().reset_index()
Donor_all["source"] = "Patient 12 CD8 T cell"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "EBV"]
obs_df = obs_df[obs_df["donor"].isin(["Pt12"])]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
EBV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
EBV_agspf["source"] = "EBV-specific clones"

## Get the EBV-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "CMV"]
obs_df = obs_df[obs_df["donor"].isin(["Pt12"])]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
CMV_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
CMV_agspf["source"] = "CMV-specific clones"

## Get the FLU-specific clones
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] == "FLU-A"]
obs_df = obs_df[obs_df["donor"].isin(["Pt12"])]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
FLU_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
FLU_agspf["source"] = "FLU-A-specific clones"

# ## Get the clonotypes of interest (T_B_clone_list)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(TB_total_list)]
obs_df = obs_df[obs_df["donor"].isin(["Pt12"])]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
print(len(obs_df))
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_all = obs_df[["L3_ann"]].value_counts().reset_index()
TB_all["source"] = "T-B clones"

## Get the clonotypes of interest (T_B_clone_list_agsp)
obs_df = cd8data["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].isin(T_B_clone_list_agsp)]
obs_df = obs_df[obs_df["donor"].isin(["Pt12"])]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
## How many CD8 T cell and and how CD4 T cell were found with cells associated with T-B cell? 
TB_agspf = obs_df[["L3_ann"]].value_counts().reset_index()
TB_agspf["source"] = "EBV-specific T-B clones"

## Concat, pivot and transform
count_df = pd.concat([Donor_all, CMV_agspf, FLU_agspf, EBV_agspf, TB_all, TB_agspf])
count_df = count_df.pivot(index = "source", columns = "L3_ann", values="count")
print("Number of cells for each category")
print(count_df.sum(axis=1))

#Transform into percentages
count_df = count_df.div(count_df.sum(axis=1), axis=0)*100

cluster_color_dict = {
    # --- Pastel colors / TN ---
    "0":  "#dbe9f6",  # very light blue
    "1":  "#c7e9c0",  # pastel green
    "2":  "#e2d9f3",  # light lavender
    "11": "#fdd0a2",  # pastel peach

    # --- TCMs ---
    "7":  "#74c476",  # green
    "13": "#31a354",  # dark green
    "21": "#3BFD16",  # neon

    # --- TEM ---
    "4":  "#fdae6b",  # light orange
    "8":  "#fd8d3c",  # orange

    # --- TEMRA ---
    "9":  "#fb6a4a",  # soft red
    "10": "#8B0000",  # dark red

    # --- TE ---
    "6":  "#C00000",  # red
    "12": "#843c39",  # brown red

    # --- Activated ---
    "17": "#f768a1",  # pink
    "19": "#c51b8a",  # 
    "20": "#FF1B76",  # 
    "22": "#d9d9d9",  # 

    # --- Gamma delta ---
    "15": "#737373",  # Dark grey

    # --- CSF blue / purple-blue gradient ---
    "3":  "#6baed6",  # light blue
    "5":  "#3182bd",  # medium blue
    "14": "#9e9ac8",  # light purple
    "16": "#756bb1",  # purple
    "18": "#1f4fbf",  # dark blue

}

# Annotate a dataframe that contains L3_ann too
L3ann_df = cd8data["gex"].obs[["leiden_1.8", "L3_ann"]].drop_duplicates().copy()
L3ann_df["cluster_colors"] = L3ann_df["leiden_1.8"].map(cluster_color_dict)

## Create L3_ann_dict
clusters = L3ann_df["L3_ann"]
colors = L3ann_df["cluster_colors"]
L3_colors = dict(zip(clusters, colors))

#plot distribution changes
count_df = count_df.reset_index()
count_df["source"] = pd.Categorical(count_df["source"], ordered=True, categories = ["Patient 12 CD8 T cell",  "CMV-specific clones", "FLU-A-specific clones", "T-B clones", "EBV-specific clones", "EBV-specific T-B clones"])

# Sort and keep source as index
count_df = count_df.sort_values("source").set_index("source")

fig, ax = plt.subplots(figsize=(3,3))

count_df.plot(
    kind="bar",
    stacked=True,
    color=L3_colors,
    edgecolor="white",
    linewidth=0.2,
    width=0.80,
    ax=ax
)

ax.set_ylim(0,100)
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
# ax.set_xticklabels([])

## export the legend once, and remove it again
# plt.legend(title="L3_ann", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.get_legend().remove()

ax.grid(False)

plt.title("Of CSF CD8 T cell (Pt12)")
plt.tight_layout()
plt.savefig("../../05_plots/02_Overview_and_doublet_analysis/97_Phenotype_of_all_clones_outside_T-B_CSF.pdf")
plt.show()

In [ ]:
# msigdb = dc.op.resource("MSigDB")

In [ ]:
# msigdb["collection"].value_counts()

In [ ]:
### Study T-cell derived genes first
deg = heatmap_CD20B[(heatmap_CD20B["TvB_lfc"]>0) & (heatmap_CD20B["logfoldchanges"]>0) & (heatmap_CD20B["stat_cat"] == "p.adj < 0.05 in both")]
deg = deg.dropna(subset=["names", "TvB_lfc"])
deg = deg.drop_duplicates("names")
deg = deg["names"].tolist()

In [ ]:
## List of available libraries
names = gp.get_library_name()
names

In [ ]:
## Run enrichment (online)
gene_list = deg                            # or a Python list / Series
# gene_sets = "KEGG_2016"                  # one library
# gene_sets = "KEGG_2016,KEGG_2013"        # several, comma-separated
gene_sets = [
    # "KEGG_2021_Human", 
    # "GO_Biological_Process_2026",
    "ClinVar_2025", 
    "Reactome_Pathways_2024", 
]   # several, as a list

## Enrich online without a custom background
enr = gp.enrichr(
    gene_list=gene_list,                              # or "./tests/data/gene_list.txt"
    gene_sets=gene_sets,
    organism='human',                                 # set to your organism, e.g. 'Yeast'
    outdir=None,                                       # keep results in memory only
)

In [ ]:
# All enrichment results across the requested libraries.
enr.results.head(5)

In [ ]:
## Plot results
from gseapy import barplot, dotplot

# Single-library views use .res2d (the most recent query).
# Pass ofname='figure.pdf' to save the figure to disk.
ax = dotplot(enr.res2d, title='test', cmap='viridis', size=40, figsize=(3, 5), edgecolor="black")

plt.show()